In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
import datetime as dt
import sqlite3
import matplotlib.pyplot as plt

In [2]:
from nba_api.stats.endpoints import leaguegamefinder


# Fetch all games for the 2023-24 season
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable='2023-24',league_id_nullable='00',season_type_nullable='Regular Season')
games = gamefinder.get_data_frames()[0]

# Display the first few rows
games_1 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
games_2 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
game_sched1 = games_1.merge(games_2, how='left',on=['GAME_ID','GAME_DATE','SEASON_ID'])
game_sched_23 = pd.DataFrame(game_sched1.loc[game_sched1['TEAM_ID_x']!=game_sched1['TEAM_ID_y']]).reset_index(drop=True)
game_sched_23.rename(columns={'TEAM_ID_x':'TEAM_ID','TEAM_ABBREVIATION_x':'TEAM_ABBREVIATION','TEAM_NAME_x':'TEAM_NAME','TEAM_ID_y':'OPPONENT_ID','TEAM_ABBREVIATION_y':'OPPONENT_ABBREVIATION','TEAM_NAME_y':'OPPONENT_NAME'},inplace=True)
game_sched_23['GAME_DATE'] = pd.to_datetime(game_sched_23['GAME_DATE'])

In [3]:
from nba_api.stats.endpoints import leaguegamefinder


# Fetch all games for the 2023-24 season
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable='2024-25',league_id_nullable='00',season_type_nullable='Regular Season')
games = gamefinder.get_data_frames()[0]

# Display the first few rows
games_1 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
games_2 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
game_sched1 = games_1.merge(games_2, how='left',on=['GAME_ID','GAME_DATE','SEASON_ID'])
game_sched_24 = pd.DataFrame(game_sched1.loc[game_sched1['TEAM_ID_x']!=game_sched1['TEAM_ID_y']]).reset_index(drop=True)
game_sched_24.rename(columns={'TEAM_ID_x':'TEAM_ID','TEAM_ABBREVIATION_x':'TEAM_ABBREVIATION','TEAM_NAME_x':'TEAM_NAME','TEAM_ID_y':'OPPONENT_ID','TEAM_ABBREVIATION_y':'OPPONENT_ABBREVIATION','TEAM_NAME_y':'OPPONENT_NAME'},inplace=True)
game_sched_24['GAME_DATE'] = pd.to_datetime(game_sched_24['GAME_DATE'])

In [4]:
game_sched = pd.concat([game_sched_23, game_sched_24])

In [5]:
today = pd.Timestamp.today()

In [6]:
# Connect to the database
conn = sqlite3.connect('nba_data.db')

# Create a cursor object to execute SQL queries
cursor = conn.cursor()

In [7]:
# List all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()



In [809]:
tables

[('passing_df_data',),
 ('rebounding_df_data',),
 ('drives_df_data',),
 ('catchshoot_df_data',),
 ('pullup_df_data',),
 ('speeddistance_data',),
 ('posttouch_data',),
 ('player_type_defensive',),
 ('catchshoot_player_data',),
 ('pullup_player_data',),
 ('passing_data',),
 ('rebounding_player_data',),
 ('drives_player_data',),
 ('speeddistance_player_data',),
 ('posttouch_player_data',),
 ('painttouch_player_data',),
 ('shotclock_data',),
 ('closestdefender_data',),
 ('dribbles_shot_data',),
 ('touchtime_shot_data',),
 ('gamelogs',),
 ('player_points_scores',),
 ('player_3s_cluster',),
 ('player_3s_scores',),
 ('opponent_def_pts_rankings',),
 ('opponent_def_3pt_rankings',),
 ('opponent_def_type_scores',),
 ('closest_def_total',),
 ('playtype_off_reformat',),
 ('shotdetail_player_rolling',),
 ('drives_player_rolling',),
 ('catchshoot_player_rolling',),
 ('pullup_player_rolling',),
 ('rebounding_player_rolling',),
 ('speeddistance_player_rolling',),
 ('drives_team_def_rolling',),
 ('catch

In [8]:

def calculate_weighted_rolling_stats(
    gamelogs: pd.DataFrame,
    rolling_columns: list,
    window_size: int = 60,
    method: str = 'average'
) -> pd.DataFrame:
    """
    Calculate rolling statistics for basketball statistics.
    
    Parameters:
    -----------
    gamelogs : pd.DataFrame
        DataFrame containing game logs with required columns:
        - OPPONENT_ID
        - GAME_DATE
        - Statistical columns specified in rolling_columns
    rolling_columns : list
        List of statistical columns to calculate rolling stats for
    window_size : int
        Number of games to include in rolling window
    method : str
        'average' for rolling mean, 'sum' for rolling sum
    """
    if method not in ['average', 'sum']:
        raise ValueError("method must be either 'average' or 'sum'")
    
    # Validate required columns
    required_columns = {'PLAYER_ID', 'GAME_DATE'}
    missing_columns = required_columns - set(gamelogs.columns)
    if missing_columns:
        raise ValueError(f"DataFrame is missing required columns: {missing_columns}")
    
    # Validate statistical columns
    missing_stat_columns = set(rolling_columns) - set(gamelogs.columns)
    if missing_stat_columns:
        raise ValueError(f"DataFrame is missing specified statistical columns: {missing_stat_columns}")
    
    # Ensure gamelogs are sorted by date
    gamelogs['GAME_DATE'] = pd.to_datetime(gamelogs['GAME_DATE'])
    gamelogs_sorted = gamelogs.sort_values(by=['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)
    
    # Initialize list to store rolling calculations
    rolling_dfs = []
    
    # Calculate games count for each window
    games_in_window = (
        gamelogs_sorted
        .groupby('PLAYER_ID')['GAME_DATE']
        .rolling(window=window_size, min_periods=1)
        .count()
        .reset_index(level=0, drop=True)
        .rename(f'GAMES_IN_WINDOW_{window_size}G')
    )
    rolling_dfs.append(games_in_window)
    
    # Calculate rolling statistics for each statistical column
    for col in rolling_columns:
        try:
            if method == 'average':
                rolling_col = (
                    gamelogs_sorted
                    .groupby('PLAYER_ID')[col]
                    .rolling(window=window_size, min_periods=1)
                    .mean()
                    .reset_index(level=0, drop=True)
                )
                suffix = f'{window_size}G_Mavg'
            else:  # method == 'sum'
                rolling_col = (
                    gamelogs_sorted
                    .groupby('PLAYER_ID')[col]
                    .rolling(window=window_size, min_periods=1)
                    .sum()
                    .reset_index(level=0, drop=True)
                )
                suffix = f'{window_size}G_Sum'
            
            rolling_dfs.append(rolling_col.rename(f'{col}_{suffix}'))
            
        except Exception as e:
            print(f"Error processing column '{col}': {str(e)}")
            continue
    
    if len(rolling_dfs) <= 1:  # Only games count column
        raise ValueError("No statistical columns were successfully processed")
    
    # Combine all rolling statistics with original data
    rolling_df = pd.concat(rolling_dfs, axis=1)
    result_df = pd.concat(
        [gamelogs_sorted.reset_index(drop=True), rolling_df],
        axis=1
    )
    
    return result_df

In [9]:
def calculate_team_rolling_stats(
    gamelogs: pd.DataFrame,
    rolling_columns: list,
    window_size: int = 60,
    method: str = 'average'
) -> pd.DataFrame:
    """
    Calculate rolling statistics for basketball statistics.
    
    Parameters:
    -----------
    gamelogs : pd.DataFrame
        DataFrame containing game logs with required columns:
        - OPPONENT_ID
        - GAME_DATE
        - Statistical columns specified in rolling_columns
    rolling_columns : list
        List of statistical columns to calculate rolling stats for
    window_size : int
        Number of games to include in rolling window
    method : str
        'average' for rolling mean, 'sum' for rolling sum
    """
    if method not in ['average', 'sum']:
        raise ValueError("method must be either 'average' or 'sum'")
    
    # Validate required columns
    required_columns = {'OPPONENT_ID', 'GAME_DATE'}
    missing_columns = required_columns - set(gamelogs.columns)
    if missing_columns:
        raise ValueError(f"DataFrame is missing required columns: {missing_columns}")
    
    # Validate statistical columns
    missing_stat_columns = set(rolling_columns) - set(gamelogs.columns)
    if missing_stat_columns:
        raise ValueError(f"DataFrame is missing specified statistical columns: {missing_stat_columns}")
    
    # Ensure gamelogs are sorted by date
    gamelogs['GAME_DATE'] = pd.to_datetime(gamelogs['GAME_DATE'])
    gamelogs_sorted = gamelogs.sort_values(by=['OPPONENT_ID', 'GAME_DATE']).reset_index(drop=True)
    
    # Initialize list to store rolling calculations
    rolling_dfs = []
    
    # Calculate games count for each window
    games_in_window = (
        gamelogs_sorted
        .groupby('OPPONENT_ID')['GAME_DATE']
        .rolling(window=window_size, min_periods=1)
        .count()
        .reset_index(level=0, drop=True)
        .rename(f'GAMES_IN_WINDOW_{window_size}G')
    )
    rolling_dfs.append(games_in_window)
    
    # Calculate rolling statistics for each statistical column
    for col in rolling_columns:
        try:
            if method == 'average':
                rolling_col = (
                    gamelogs_sorted
                    .groupby('OPPONENT_ID')[col]
                    .rolling(window=window_size, min_periods=1)
                    .mean()
                    .reset_index(level=0, drop=True)
                )
                suffix = f'{window_size}G_Mavg'
            else:  # method == 'sum'
                rolling_col = (
                    gamelogs_sorted
                    .groupby('OPPONENT_ID')[col]
                    .rolling(window=window_size, min_periods=1)
                    .sum()
                    .reset_index(level=0, drop=True)
                )
                suffix = f'{window_size}G_Sum'
            
            rolling_dfs.append(rolling_col.rename(f'{col}_{suffix}'))
            
        except Exception as e:
            print(f"Error processing column '{col}': {str(e)}")
            continue
    
    if len(rolling_dfs) <= 1:  # Only games count column
        raise ValueError("No statistical columns were successfully processed")
    
    # Combine all rolling statistics with original data
    rolling_df = pd.concat(rolling_dfs, axis=1)
    result_df = pd.concat(
        [gamelogs_sorted.reset_index(drop=True), rolling_df],
        axis=1
    )
    
    return result_df

In [10]:
table_name = "shotclock_data"

In [11]:
shotclock_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [12]:
# Define a function to categorize shot clock range
def categorize_shot_clock_range(shot_clock_range):
    if shot_clock_range in ['24-22', '22-18 Very Early', '18-15 Early']:
        return 'Early'
    elif shot_clock_range == '15-7 Average':
        return 'Average'
    elif shot_clock_range in ['4-0 Very Late', '7-4 Late']:
        return 'Late'
    else:
        return 'Unknown'

# Apply the function to create a new column 'SHOT_CLOCK_CATEGORY'
shotclock_data['SHOT_CLOCK_CATEGORY'] = shotclock_data['SHOT_CLOCK_RANGE'].apply(categorize_shot_clock_range)


In [13]:
shotclock_data.loc[shotclock_data['SHOT_CLOCK_CATEGORY']=='Late']['PLAYER_NAME'].nunique()

654

In [14]:

# Group by PLAYER_NAME, PLAYER_LAST_TEAM_ABBREVIATION, and SHOT_CLOCK_RANGE
# Then aggregate sum for FGA, FG2A, FG3A
shotclock_df = shotclock_data.groupby(['PLAYER_ID','PLAYER_NAME', 'PLAYER_LAST_TEAM_ID','PLAYER_LAST_TEAM_ABBREVIATION', 'SHOT_CLOCK_CATEGORY']).agg({
    'FGA': 'sum',
    'FG2A': 'sum',
    'FG3A': 'sum'
}).reset_index()

In [817]:
# Group by PLAYER_NAME, PLAYER_LAST_TEAM_ABBREVIATION, and SHOT_CLOCK_RANGE
# Then aggregate sum for FGA, FG2A, FG3A
shotclock_made_df = shotclock_data.groupby(['PLAYER_ID','PLAYER_NAME', 'PLAYER_LAST_TEAM_ID','PLAYER_LAST_TEAM_ABBREVIATION', 'SHOT_CLOCK_CATEGORY']).agg({
    'FGM': 'sum',
    'FG2M': 'sum',
    'FG3M': 'sum'
}).reset_index()

In [818]:
shotclock_summary= shotclock_df.merge(shotclock_made_df, how='outer')

In [15]:
shotclock_data

,PLAYER_ID,PLAYER_NAME,PLAYER_LAST_TEAM_ID,PLAYER_LAST_TEAM_ABBREVIATION,AGE,GP,G,FGA_FREQUENCY,FGM,FGA,FG_PCT,EFG_PCT,FG2A_FREQUENCY,FG2M,FG2A,FG2_PCT,FG3A_FREQUENCY,FG3M,FG3A,FG3_PCT,GAME_DATE,SHOT_CLOCK_RANGE,id,SEASON_YEAR,SHOT_CLOCK_CATEGORY
0,1641713,GG Jackson,1610612763,MEM,19.0,1,1,0.194,4.0,7.0,0.571,0.643,0.083,3.0,3.0,1.000,0.111,1.0,4.0,0.25,04/14/2024,18-15 Early,1641713_04/14/2024_18-15 Early,2023-24,Early
1,1629875,Xavier Moon,1610612746,LAC,29.0,1,1,0.318,1.0,7.0,0.143,0.143,0.091,1.0,2.0,0.500,0.227,0.0,5.0,0.0,04/14/2024,18-15 Early,1629875_04/14/2024_18-15 Early,2023-24,Early
2,1631099,Keegan Murray,1610612758,SAC,23.0,1,1,0.500,2.0,6.0,0.333,0.333,0.250,2.0,3.0,0.667,0.250,0.0,3.0,0.0,04/14/2024,18-15 Early,1631099_04/14/2024_18-15 Early,2023-24,Early
3,1630702,Jaden Hardy,1610612742,DAL,21.0,1,1,0.429,1.0,6.0,0.167,0.167,0.214,1.0,3.0,0.333,0.214,0.0,3.0,0.0,04/14/2024,18-15 Early,1630702_04/14/2024_18-15 Early,2023-24,Early
4,1628368,De'Aaron Fox,1610612758,SAC,26.0,1,1,0.357,5.0,5.0,1.000,1.100,0.286,4.0,4.0,1.000,0.071,1.0,1.0,1.0,04/14/2024,18-15 Early,1628368_04/14/2024_18-15 Early,2023-24,Early
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150076,1630533,Ziaire Williams,1610612751,BKN,23.0,1,1,0.125,0.0,1.0,0.000,0.000,0.125,0.0,1.0,0.000,0.000,0.0,0.0,None,03/06/2025,4-0 Very Late,1630533_03/06/2025_4-0 Very Late,2024-25,Late
150077,1629656,Quentin Grimes,1610612755,PHI,24.0,1,1,0.091,0.0,1.0,0.000,0.000,0.091,0.0,1.0,0.000,0.000,0.0,0.0,None,03/06/2025,4-0 Very Late,1629656_03/06/2025_4-0 Very Late,2024-25,Late
150078,1629674,Neemias Queta,1610612738,BOS,25.0,1,1,0.100,0.0,1.0,0.000,0.000,0.100,0.0,1.0,0.000,0.000,0.0,0.0,None,03/06/2025,4-0 Very Late,1629674_03/06/2025_4-0 Very Late,2024-25,Late
150079,1630200,Tre Jones,1610612741,CHI,25.0,1,1,0.083,0.0,1.0,0.000,0.000,0.083,0.0,1.0,0.000,0.000,0.0,0.0,None,03/06/2025,4-0 Very Late,1630200_03/06/2025_4-0 Very Late,2024-25,Late


In [820]:
table_name = "gamelogs"
gamelogs_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [821]:
gamelogs_data['GAME_DATE'] = pd.to_datetime(gamelogs_data['GAME_DATE'])

In [822]:
gamelogs_data.drop_duplicates(subset=['PLAYER_ID','GAME_ID'], inplace=True)

In [823]:
#gamelogs_data.sort_values(by=['Player_ID','GAME_DATE'], ascending=[True, False]).groupby('Player_ID').tail(60)
gamelogs_sorted = pd.DataFrame(gamelogs_data.sort_values(by=['PLAYER_ID','GAME_DATE']))

In [824]:
#gamelogs_sorted['PTS_60G_MA'] = gamelogs_sorted.groupby('Player_ID',group_keys=False)['PTS'].apply(lambda x: x.rolling(window=60, min_periods=1).mean())

In [825]:
gamelogs_with_rolling = calculate_weighted_rolling_stats(
    gamelogs=gamelogs_sorted,
    rolling_columns=['MIN', 'PTS', 'REB', 'AST', 'FGA','FG3M','FG3A'])

In [826]:
gamelogs_with_sum = calculate_weighted_rolling_stats(
    gamelogs=gamelogs_sorted,  # Pass in the previous result
    rolling_columns=['MIN', 'PTS', 'REB', 'AST', 'FGA','FG3M','FG3A'],
    method='sum')

In [827]:
gamelogs_sum_add = pd.DataFrame(gamelogs_with_sum[['PLAYER_ID','GAME_DATE','MIN_60G_Sum','PTS_60G_Sum', 'FGA_60G_Sum','FG3M_60G_Sum','FG3A_60G_Sum']])
gamelogs_rolling= gamelogs_with_rolling.merge(gamelogs_sum_add, how='left')
gamelogs_60Day = gamelogs_rolling.merge(game_sched, how='left')

In [828]:

gamelogs_60Day.columns

Index(['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID',
       'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP',
       'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM',
       'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK',
       'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2',
       'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK',
       'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK',
       'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK',
       'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK',
       'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK',
       'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK',
       'WNBA_FANTASY_PTS_RANK', 'AVAILABLE_FLAG', 'MIN_SEC', 'id',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'PTS_60G_Mavg', 'REB_60G_Mavg',
       'AST_60G_Mavg', 'FGA_60G

In [829]:
#gamelogs_60Day.drop_duplicates()

In [830]:
gl_60Day = pd.DataFrame(gamelogs_60Day[['SEASON_YEAR','PLAYER_ID','TEAM_ID','GAME_ID', 'GAME_DATE','GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'PTS_60G_Mavg', 'REB_60G_Mavg',
       'AST_60G_Mavg', 'FGA_60G_Mavg', 'FG3M_60G_Mavg', 'FG3A_60G_Mavg', 'FGA_60G_Sum','FG3M_60G_Sum','FG3A_60G_Sum',
       'MIN_60G_Sum', 'PTS_60G_Sum']].sort_values(by=['PLAYER_ID','GAME_DATE'], ascending=False))

In [831]:
# Most Recent Data
most_recent_mask = gl_60Day['GAME_DATE'] == gl_60Day.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
gl_60Day_new = gl_60Day[most_recent_mask]

In [832]:
#gl_60Day_new.drop_duplicates(subset='PLAYER_ID')

# Dribbles

In [833]:
table_name = "dribbles_shot_data"
dribbles_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

def categorize_assist(dribbles):
    if dribbles in ['0 Dribbles', '1 Dribble','2 Dribbles']:
        return 'Assisted'
    else:
        return 'Created'

dribbles_data['dribble_type'] = dribbles_data['DRIBBLES'].apply(categorize_assist)

dribbles_columns = ['FGA','FG2A','FG3A','FGM','FG2M','FG3M']
dribbles_data['GAME_DATE'] = pd.to_datetime(dribbles_data['GAME_DATE'])
dribbles_sorted = pd.DataFrame(dribbles_data.sort_values(by=['PLAYER_ID','GAME_DATE']))

dribbles_rolling = dribbles_sorted.groupby(['PLAYER_ID','PLAYER_NAME', 'DRIBBLES'], group_keys=False)[dribbles_columns].apply(lambda x: x.rolling(window=60, min_periods=1).sum()).round(1)
dribbles_rolling_avg = dribbles_sorted.groupby(['PLAYER_ID','PLAYER_NAME', 'DRIBBLES'], group_keys=False)[dribbles_columns].apply(lambda x: x.rolling(window=60, min_periods=1).mean()).round(1)

dribbles_60Day= pd.concat([dribbles_sorted, dribbles_rolling.add_suffix('_60G_Msum'), dribbles_rolling_avg.add_suffix('_60G_Mavg')], axis=1)

In [834]:
dribbles_60Day_sum = dribbles_60Day[['PLAYER_ID','PLAYER_NAME','GAME_DATE','DRIBBLES','FGA_60G_Msum','FG2A_60G_Msum','FG3A_60G_Msum','FGM_60G_Msum','FG2M_60G_Msum','FG3M_60G_Msum','FGA_60G_Mavg','FG2A_60G_Mavg','FG3A_60G_Mavg','FGM_60G_Mavg','FG2M_60G_Mavg','FG3M_60G_Mavg']]

In [835]:
# Most Recent Data
most_recent_mask = dribbles_60Day_sum['GAME_DATE'] == dribbles_60Day_sum.groupby(['PLAYER_ID','DRIBBLES'])['GAME_DATE'].transform('max')
dribbles_60Day_new = dribbles_60Day_sum[most_recent_mask]

In [836]:
dribbles_60Day_new

,PLAYER_ID,PLAYER_NAME,GAME_DATE,DRIBBLES,FGA_60G_Msum,FG2A_60G_Msum,FG3A_60G_Msum,FGM_60G_Msum,FG2M_60G_Msum,FG3M_60G_Msum,FGA_60G_Mavg,FG2A_60G_Mavg,FG3A_60G_Mavg,FGM_60G_Mavg,FG2M_60G_Mavg,FG3M_60G_Mavg
119313,2544,LeBron James,2025-03-06,0 Dribbles,342.0,147.0,195.0,190.0,108.0,82.0,5.7,2.4,3.2,3.2,1.8,1.4
119402,2544,LeBron James,2025-03-06,1 Dribble,124.0,76.0,48.0,69.0,56.0,13.0,2.6,1.6,1.0,1.4,1.2,0.3
119483,2544,LeBron James,2025-03-06,2 Dribbles,127.0,109.0,18.0,71.0,62.0,9.0,2.1,1.8,0.3,1.2,1.0,0.2
119540,2544,LeBron James,2025-03-06,3-6 Dribbles,309.0,256.0,53.0,151.0,134.0,17.0,5.2,4.3,0.9,2.5,2.2,0.3
119614,2544,LeBron James,2025-03-06,7+ Dribbles,239.0,202.0,37.0,122.0,108.0,14.0,4.0,3.4,0.6,2.0,1.8,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78188,1642530,Yuki Kawamura,2024-11-27,7+ Dribbles,6.0,3.0,3.0,2.0,1.0,1.0,1.2,0.6,0.6,0.4,0.2,0.2
90421,1642530,Yuki Kawamura,2024-12-29,1 Dribble,2.0,1.0,1.0,1.0,1.0,0.0,1.0,0.5,0.5,0.5,0.5,0.0
90485,1642530,Yuki Kawamura,2024-12-29,2 Dribbles,3.0,2.0,1.0,2.0,2.0,0.0,1.0,0.7,0.3,0.7,0.7,0.0
90573,1642530,Yuki Kawamura,2024-12-29,3-6 Dribbles,4.0,0.0,4.0,2.0,0.0,2.0,1.0,0.0,1.0,0.5,0.0,0.5


# TouchTime

In [837]:
table_name = "touchtime_shot_data"
touchtime_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

touchtime_columns = ['FGA','FG2A','FG3A','FGM','FG2M','FG3M']
touchtime_data['GAME_DATE'] = pd.to_datetime(touchtime_data['GAME_DATE'])
touchtime_sorted = pd.DataFrame(touchtime_data.sort_values(by=['PLAYER_ID','GAME_DATE']))

touchtime_rolling = touchtime_sorted.groupby(['PLAYER_ID','PLAYER_NAME', 'TOUCH_TIME'], group_keys=False)[touchtime_columns].apply(lambda x: x.rolling(window=60, min_periods=1).sum()).round(1)
touchtime_rolling_avg = touchtime_sorted.groupby(['PLAYER_ID','PLAYER_NAME', 'TOUCH_TIME'], group_keys=False)[touchtime_columns].apply(lambda x: x.rolling(window=60, min_periods=1).mean()).round(1)

touchtime_60Day= pd.concat([touchtime_sorted, touchtime_rolling.add_suffix('_60G_Msum'), touchtime_rolling_avg.add_suffix('_60G_Mavg')], axis=1)

In [838]:
touchtime_data['TOUCH_TIME'].unique()

array(['Touch < 2 Seconds', 'Touch 2-6 Seconds', 'Touch 6+ Seconds'],
      dtype=object)

In [839]:
touchtime_60Day_sum = touchtime_60Day[['PLAYER_ID','PLAYER_NAME','GAME_DATE','TOUCH_TIME','FGA_60G_Msum','FG2A_60G_Msum','FG3A_60G_Msum','FGM_60G_Msum','FG2M_60G_Msum','FG3M_60G_Msum',
                                       'FGA_60G_Mavg','FG2A_60G_Mavg','FG3A_60G_Mavg','FGM_60G_Mavg','FG2M_60G_Mavg','FG3M_60G_Mavg']]
# Most Recent Data
most_recent_mask = touchtime_60Day_sum['GAME_DATE'] == touchtime_60Day_sum.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
touchtime_60Day_new = touchtime_60Day_sum[most_recent_mask]

# PlayType Offense

In [840]:
def analyze_player_playtypes(data):
    """
    Analyze player performance across different play types.
    Returns a DataFrame with all offensive metrics using per-game stats.
    """
    if isinstance(data, list):
        df = pd.DataFrame(data)
    else:
        df = data.copy()
    
    def normalize_series(s):
        """Normalize a series to 0-100 scale"""
        min_val = s.values.min()
        max_val = s.values.max()
        if max_val == min_val:
            return pd.Series(50, index=s.index)
        return ((s - min_val) / (max_val - min_val) * 100).round(1)
    
    results = []
    for play_type, play_group in df.groupby('PLAY_TYPE'):
        # Calculate per game metrics first
        play_group['POSS_PER_GAME'] = play_group['POSS'] / play_group['GP']
        play_group['FGA_PER_GAME'] = play_group['FGA'] / play_group['GP']
        
        # Calculate z-scores for key metrics
        metrics = ['PPP', 'FG_PCT', 'SCORE_POSS_PCT', 'FGA_PER_GAME', 'POSS_PER_GAME','FT_POSS_PCT']
        z_scores = {}
        
        for metric in metrics:
            z_score = (play_group[metric] - play_group[metric].mean()) / play_group[metric].std()
            z_scores[metric] = z_score
        
        
        # Calculate component scores
        scoring_efficiency = (
            z_scores['PPP'] * 0.4 +
            z_scores['FG_PCT'] * 0.4 +
            z_scores['SCORE_POSS_PCT'] * 0.2
        )
        
        volume = (
            z_scores['FGA_PER_GAME'] * 0.4 +
            z_scores['POSS_PER_GAME'] * 0.4+
            z_scores['FT_POSS_PCT'] * 0.2
        )
        
        overall = (scoring_efficiency * 0.4 + volume * 0.6)
        
        for player in play_group['PLAYER_NAME'].unique():
            player_mask = play_group['PLAYER_NAME'] == player
            results.append({
                'PLAYER_ID': play_group.loc[player_mask, 'PLAYER_ID'].iloc[0],
                'PLAYER_NAME': player,
                'TEAM_ID': play_group.loc[player_mask, 'TEAM_ID'].iloc[0],
                'PLAY_TYPE': play_type,
                'SCORING_EFFICIENCY': scoring_efficiency[player_mask].iloc[0],
                'VOLUME_SCORE': volume[player_mask].iloc[0],
                'OVERALL_SCORE': overall[player_mask].iloc[0],
                'POSS_PCT': play_group.loc[player_mask, 'POSS_PCT'].iloc[0],
                'PPP': play_group.loc[player_mask, 'PPP'].iloc[0],
                'FG_PCT': play_group.loc[player_mask, 'FG_PCT'].iloc[0],
                'POSS_PER_GAME': play_group.loc[player_mask, 'POSS_PER_GAME'].iloc[0],
                'FGA_PER_GAME': play_group.loc[player_mask, 'FGA_PER_GAME'].iloc[0],
                'GP': play_group.loc[player_mask, 'GP'].iloc[0]
            })
    
    results_df = pd.DataFrame(results)
    
    # Normalize scores to 0-100 scale
    score_columns = ['SCORING_EFFICIENCY', 'VOLUME_SCORE', 'OVERALL_SCORE']
    for col in score_columns:
        results_df[col] = normalize_series(results_df[col])
    
    # Format percentages and rates
    results_df['POSS_PCT'] = (results_df['POSS_PCT'] * 100).round(1)
    results_df['FG_PCT'] = (results_df['FG_PCT'] * 100).round(1)
    results_df['PPP'] = results_df['PPP'].round(3)
    results_df['POSS_PER_GAME'] = results_df['POSS_PER_GAME'].round(1)
    results_df['FGA_PER_GAME'] = results_df['FGA_PER_GAME'].round(1)
    
    # Add percentile rankings
    for col in score_columns:
        results_df[f'{col}_PERCENTILE'] = results_df.groupby('PLAY_TYPE')[col].rank(pct=True) * 100
    
    # Sort and reorder columns
    results_df = results_df.sort_values(['PLAY_TYPE', 'OVERALL_SCORE'], ascending=[True, False])
    
    column_order = [
        'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'PLAY_TYPE',
        'OVERALL_SCORE', 'OVERALL_SCORE_PERCENTILE',
        'SCORING_EFFICIENCY', 'SCORING_EFFICIENCY_PERCENTILE',
        'VOLUME_SCORE', 'VOLUME_SCORE_PERCENTILE',
        'PPP', 'FG_PCT', 'POSS_PCT', 
        'POSS_PER_GAME', 'FGA_PER_GAME', 'GP'
    ]
    
    return results_df[column_order]

def get_player_archetype_profile(df, player_name):
    """
    Get a player's archetype profile showing their strengths across play types
    """
    results = analyze_player_playtypes(df)
    player_profile = results[results['PLAYER_NAME'] == player_name].copy()
    
    def get_archetype_label(row):
        if row['OVERALL_SCORE_PERCENTILE'] >= 85:
            return 'Elite'
        elif row['OVERALL_SCORE_PERCENTILE'] >= 70:
            return 'Strong'
        elif row['OVERALL_SCORE_PERCENTILE'] <= 30:
            return 'Limited'
        return 'Average'
        
    player_profile['ARCHETYPE_STRENGTH'] = player_profile.apply(get_archetype_label, axis=1)
    
    return player_profile.sort_values('OVERALL_SCORE', ascending=False)

def get_player_type_insights(df, player_name):
    """
    Generate detailed insights about a player's performance across play types
    """
    play_type_details = analyze_player_playtypes(df)
    
    player_data = play_type_details[play_type_details['PLAYER_NAME'] == player_name].copy()
    
    player_metrics = player_data.groupby('PLAY_TYPE').agg({
        'POSS_PER_GAME': 'first',
        'FGA_PER_GAME': 'first',
        'PPP': 'first',
        'FG_PCT': 'first',
        'POSS_PCT': 'first',
        'SCORING_EFFICIENCY': 'first',
        'VOLUME_SCORE': 'first',
        'OVERALL_SCORE': 'first',
        'GP': 'first'
    }).round(2)
    
    insights = {
        'player': player_name,
        'metrics': player_metrics,
        'strengths': {
            'overall': player_metrics.nlargest(3, 'OVERALL_SCORE'),
            'efficiency': player_metrics.nlargest(3, 'SCORING_EFFICIENCY'),
            'volume': player_metrics.nlargest(3, 'VOLUME_SCORE')
        },
        'development_areas': {
            'overall': player_metrics.nsmallest(3, 'OVERALL_SCORE'),
            'efficiency': player_metrics.nsmallest(3, 'SCORING_EFFICIENCY'),
            'volume': player_metrics.nsmallest(3, 'VOLUME_SCORE')
        },
        'most_frequent': player_metrics.nlargest(3, 'POSS_PCT')
    }
    
    return insights

def pivot_player_analysis(df):
    """
    Pivot player analysis to have play types as column suffixes.
    """
    # First get base analysis
    results = analyze_player_playtypes(df)
    
    # Select columns to pivot
    cols_to_pivot = [
        'OVERALL_SCORE','OVERALL_SCORE_PERCENTILE', 'SCORING_EFFICIENCY','SCORING_EFFICIENCY_PERCENTILE', 'VOLUME_SCORE','VOLUME_SCORE_PERCENTILE',"POSS_PCT"
    ]
    
    # Create pivot tables for each metric and suffix with play type
    pivot_dfs = []
    
    # Keep player info for joining
    player_info = results[['PLAYER_ID', 'PLAYER_NAME',]].drop_duplicates()
    
    for col in cols_to_pivot:
        pivot = pd.pivot_table(
            results,
            values=col,
            index=['PLAYER_ID', 'PLAYER_NAME',],
            columns='PLAY_TYPE',
            aggfunc='first'
        )
        
        # Rename columns to add metric as prefix
        pivot.columns = [f'{col}_{playtype}' for playtype in pivot.columns]
        pivot_dfs.append(pivot)
    
    # Combine all pivoted dataframes
    final_df = pd.concat(pivot_dfs, axis=1)
    final_df = final_df.reset_index()
    
    # Round numeric columns
    numeric_cols = final_df.select_dtypes(include=['float64']).columns
    final_df[numeric_cols] = final_df[numeric_cols].round(1)
    
    return final_df

In [841]:
table_name = "player_type_offensive"
playtype_off_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [842]:
playtype_off_data['as_of'] = pd.to_datetime(playtype_off_data['as_of'])

In [843]:
playtype_off_data = pd.DataFrame(playtype_off_data.sort_values(by=['PLAYER_ID','PLAY_TYPE','as_of']).drop_duplicates(subset=['PLAYER_ID','PLAY_TYPE'], keep='last'))

In [844]:
results_df_player = pivot_player_analysis(playtype_off_data).fillna(0)

In [845]:
results_df_player

,PLAYER_ID,PLAYER_NAME,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,SCORING_EFFICIENCY_Cut,SCORING_EFFICIENCY_Handoff,SCORING_EFFICIENCY_Isolation,SCORING_EFFICIENCY_Misc,SCORING_EFFICIENCY_OffRebound,SCORING_EFFICIENCY_OffScreen,SCORING_EFFICIENCY_PRBallHandler,SCORING_EFFICIENCY_PRRollMan,SCORING_EFFICIENCY_Postup,SCORING_EFFICIENCY_Spotup,SCORING_EFFICIENCY_Transition,SCORING_EFFICIENCY_PERCENTILE_Cut,SCORING_EFFICIENCY_PERCENTILE_Handoff,SCORING_EFFICIENCY_PERCENTILE_Isolation,SCORING_EFFICIENCY_PERCENTILE_Misc,SCORING_EFFICIENCY_PERCENTILE_OffRebound,SCORING_EFFICIENCY_PERCENTILE_OffScreen,SCORING_EFFICIENCY_PERCENTILE_PRBallHandler,SCORING_EFFICIENCY_PERCENTILE_PRRollMan,SCORING_EFFICIENCY_PERCENTILE_Postup,SCORING_EFFICIENCY_PERCENTILE_Spotup,SCORING_EFFICIENCY_PERCENTILE_Transition,VOLUME_SCORE_Cut,VOLUME_SCORE_Handoff,VOLUME_SCORE_Isolation,VOLUME_SCORE_Misc,VOLUME_SCORE_OffRebound,VOLUME_SCORE_OffScreen,VOLUME_SCORE_PRBallHandler,VOLUME_SCORE_PRRollMan,VOLUME_SCORE_Postup,VOLUME_SCORE_Spotup,VOLUME_SCORE_Transition,VOLUME_SCORE_PERCENTILE_Cut,VOLUME_SCORE_PERCENTILE_Handoff,VOLUME_SCORE_PERCENTILE_Isolation,VOLUME_SCORE_PERCENTILE_Misc,VOLUME_SCORE_PERCENTILE_OffRebound,VOLUME_SCORE_PERCENTILE_OffScreen,VOLUME_SCORE_PERCENTILE_PRBallHandler,VOLUME_SCORE_PERCENTILE_PRRollMan,VOLUME_SCORE_PERCENTILE_Postup,VOLUME_SCORE_PERCENTILE_Spotup,VOLUME_SCORE_PERCENTILE_Transition,POSS_PCT_Cut,POSS_PCT_Handoff,POSS_PCT_Isolation,POSS_PCT_Misc,POSS_PCT_OffRebound,POSS_PCT_OffScreen,POSS_PCT_PRBallHandler,POSS_PCT_PRRollMan,POSS_PCT_Postup,POSS_PCT_Spotup,POSS_PCT_Transition
0,2544,LeBron James,49.0,43.7,64.9,56.8,40.3,43.6,52.3,39.0,58.1,41.8,70.2,78.1,61.6,95.7,89.9,51.0,61.9,83.1,48.2,91.4,51.4,99.4,61.8,48.5,54.7,54.7,48.8,56.8,54.1,48.7,50.1,52.2,57.5,88.1,45.2,72.7,74.7,47.0,79.4,70.9,47.7,58.3,64.3,82.2,26.0,30.1,53.5,42.5,25.3,23.0,37.0,23.6,48.2,24.4,58.1,62.3,72.7,95.4,91.2,58.8,47.6,82.3,56.2,92.6,44.9,97.4,3.2,4.0,16.4,5.7,2.6,2.4,18.6,3.2,12.1,9.8,21.2
1,101108,Chris Paul,0.0,44.0,36.1,43.8,0.0,0.0,46.5,0.0,0.0,37.3,24.6,0.0,63.2,31.7,64.9,0.0,0.0,70.3,0.0,0.0,36.2,8.1,0.0,64.2,45.8,49.8,0.0,0.0,51.0,0.0,0.0,50.8,35.5,0.0,92.4,34.9,55.9,0.0,0.0,57.9,0.0,0.0,57.4,10.5,0.0,17.2,22.2,29.2,0.0,0.0,31.9,0.0,0.0,19.6,15.5,0.0,23.2,49.3,66.7,0.0,0.0,76.0,0.0,0.0,32.8,18.6,0.0,3.8,10.8,8.4,0.0,0.0,40.1,0.0,0.0,20.6,13.5
2,200768,Kyle Lowry,38.2,48.6,18.4,36.9,0.0,48.3,29.8,0.0,0.0,32.9,11.7,44.4,75.9,2.3,38.2,0.0,79.2,17.4,0.0,0.0,24.1,0.4,58.5,66.8,21.4,46.2,0.0,68.5,34.8,0.0,0.0,45.8,19.4,82.3,94.9,1.8,39.7,0.0,95.7,9.8,0.0,0.0,36.1,0.6,14.2,21.1,19.3,22.9,0.0,19.2,23.1,0.0,0.0,17.8,11.9,9.4,42.1,26.6,44.1,0.0,29.7,53.7,0.0,0.0,25.3,6.2,3.5,6.1,5.5,8.8,0.0,3.2,17.7,0.0,0.0,35.4,18.4
3,200782,P.J. Tucker,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,26.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,61.5,0.0
4,201142,Kevin Durant,41.1,45.3,74.7,57.6,26.7,66.9,52.0,47.1,48.4,65.3,58.8,53.4,68.1,98.2,90.9,12.5,97.8,82.2,70.8,78.2,98.5,93.1,52.6,62.2,63.9,59.9,39.4,55.3,55.7,50.9,46.8,60.0,58.8,64.0,90.2,93.3,86.6,18.3,74.0,78.0,54.4,44.5,87.9,85.0,23.1,20.6,58.8,39.2,15.0,55.6,35.2,32.6,38.0,49.5,41.7,48.8,38.7,96.8,85.9,12.4,97.0,79.8,78.9,85.3,96.7,87.0,2.

In [846]:
#playtype_off_df1 = pd.DataFrame(playtype_off_data[['SEASON_ID','PLAYER_ID','PLAYER_NAME','TEAM_ID','GP','PLAY_TYPE','PTS','POSS_PCT','POSS','PPP','FGA','FGM']])

In [847]:
#playtype_off_df = playtype_off_df1.groupby(['SEASON_ID','PLAYER_ID','PLAYER_NAME','PLAY_TYPE']).agg({"GP":'sum',"PTS":'sum',"POSS":'sum',"FGA":'sum',"FGM":'sum',"POSS_PCT":'mean',}).reset_index()

In [848]:
#playtype_off_df['SHOT_FREQ'] = (playtype_off_df['FGA']/playtype_off_df['POSS']).round(2)
#playtype_off_df['POSS/G'] = (playtype_off_df['POSS']/playtype_off_df['GP']).round(1)
#playtype_off_df['FGA/G'] = (playtype_off_df['FGA']/playtype_off_df['GP']).round(1)
#playtype_off_df['PTS/MAKE'] = (playtype_off_df['PTS']/playtype_off_df['FGM']).round(2)
#playtype_off_df['FG_PCT'] = (playtype_off_df['FGM']/playtype_off_df['FGA']).round(2)
#playtype_off_df['PPP'] = (playtype_off_df['PTS']/playtype_off_df['POSS']).round(2)

In [849]:
#playtype_off_df[playtype_off_df['PLAYER_NAME']=='LeBron James']

In [850]:
#playtype_off_pivot = playtype_off_df.pivot_table(index=['PLAYER_ID', 'PLAYER_NAME'], 
#                          columns='PLAY_TYPE', 
#                          values=['POSS_PCT', 'POSS/G', 'PPP', 'FGA/G','SHOT_FREQ','FG_PCT','PTS/MAKE'])

# Flatten the multi-index columns and rename them with the format `metric_PLAYTYPE`
#playtype_off_pivot.columns = [f'{metric}_{play_type}' for metric, play_type in playtype_off_pivot.columns]

# Reset index to turn multi-index back into regular columns
#playtype_off_pivot.reset_index(inplace=True)

In [851]:
#playtype_off_pivot.fillna(0, inplace=True)

In [852]:
#playtype_off_pivot

In [853]:
#playtype_off_pivot['SEASON_YEAR'] = "2024-25"

In [854]:
#playtype_off_pivot['as_of'] = pd.to_datetime(today).strftime('%Y-%m-%d')


In [855]:
'''playtype_off_pivot['as_of'] = pd.to_datetime(today)
playtype_off_pivot['id'] = playtype_off_pivot['as_of'].astype(str)+"_"+playtype_off_pivot['PLAYER_ID'].astype(str)+"_"+playtype_off_pivot['SEASON_YEAR'].astype(str)
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "playtype_off_reformat"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    playtype_off_pivot.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = playtype_off_pivot[~playtype_off_pivot['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")
'''


'playtype_off_pivot[\'as_of\'] = pd.to_datetime(today)\nplaytype_off_pivot[\'id\'] = playtype_off_pivot[\'as_of\'].astype(str)+"_"+playtype_off_pivot[\'PLAYER_ID\'].astype(str)+"_"+playtype_off_pivot[\'SEASON_YEAR\'].astype(str)\n# Connect to SQLite database\nconn = sqlite3.connect(\'nba_data.db\')\ncursor = conn.cursor()\n\n# Step 1: Check if the table exists\ntable_name = "playtype_off_reformat"\ncursor.execute(f"SELECT name FROM sqlite_master WHERE type=\'table\' AND name=\'{table_name}\';")\ntable_exists = cursor.fetchone()\n\n# Step 2: Create the table if it doesn\'t exist\nif not table_exists:\n    playtype_off_pivot.to_sql(table_name, conn, if_exists=\'replace\', index=False)\n    print(f"Table \'{table_name}\' created and data inserted.")\nelse:\n    print(f"Table \'{table_name}\' already exists. Checking for new records...")\n\n    # Step 3: Pull existing IDs from the table\n    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)\n    new_data = playtype_off_pivot

# Shot Locations

In [16]:
table_name = "shot_detail_data"
shotdetail_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)


In [17]:
shotdetail_data

,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,EVENT_TYPE,ACTION_TYPE,SHOT_TYPE,SHOT_ZONE_BASIC,SHOT_ZONE_AREA,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,id,SEASON_YEAR
0,Shot Chart Detail,0022300031,130,1630173,Precious Achiuwa,1610612761,Toronto Raptors,1,1,49,Made Shot,Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,1,-6,16,1,1,20231117,TOR,BOS,1630173_20231117_Center(C),2023-24
1,Shot Chart Detail,0022300031,148,1630173,Precious Achiuwa,1610612761,Toronto Raptors,1,0,11,Missed Shot,Jump Shot,3PT Field Goal,Right Corner 3,Right Side(R),24+ ft.,23,232,4,1,0,20231117,TOR,BOS,1630173_20231117_Right Side(R),2023-24
2,Shot Chart Detail,0022300031,195,1630173,Precious Achiuwa,1610612761,Toronto Raptors,2,8,49,Missed Shot,Jump Shot,3PT Field Goal,Left Corner 3,Left Side(L),24+ ft.,22,-229,1,1,0,20231117,TOR,BOS,1630173_20231117_Left Side(L),2023-24
3,Shot Chart Detail,0022300031,484,1630173,Precious Achiuwa,1610612761,Toronto Raptors,4,9,30,Made Shot,Driving Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,0,-7,-2,1,1,20231117,TOR,BOS,1630173_20231117_Center(C),2023-24
4,Shot Chart Detail,0022300038,141,1630173,Precious Achiuwa,1610612761,Toronto Raptors,1,0,32,Missed Shot,Jump Shot,3PT Field Goal,Above the Break 3,Left Side Center(LC),24+ ft.,26,-200,177,1,0,20231121,ORL,TOR,1630173_20231121_Left Side Center(LC),2023-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
379200,Shot Chart Detail,0022401141,126,1641783,Tristan da Silva,1610612753,Orlando Magic,1,0,58,Missed Shot,Jump Shot,3PT Field Goal,Right Corner 3,Right Side(R),24+ ft.,23,230,31,1,0,20250313,NOP,ORL,16417830022401141Right Side(R)126,None
379201,Shot Chart Detail,0022401141,254,1641783,Tristan da Silva,1610612753,Orlando Magic,2,3,28,Missed Shot,Running Alley Oop Layup Shot,2PT Field Goal,Restricted Area,Center(C),Less Than 8 ft.,2,-21,6,1,0,20250313,NOP,ORL,16417830022401141Center(C)254,None
379202,Shot Chart Detail,0022401141,455,1641783,Tristan da Silva,1610612753,Orlando Magic,3,0,57,Missed Shot,Jump Shot,3PT Field Goal,Above the Break 3,Right Side Center(RC),24+ ft.,24,189,159,1,0,20250313,NOP,ORL,16417830022401141Right Side Center(RC)455,None
379203,Shot Chart Detail,0022401141,591,1641783,Tristan da Silva,1610612753,Orlando Magic,4,3,8,Made Shot,Jump Shot,3PT Field Goal,Left Corner 3,Left Side(L),24+ ft.,22,-225,42,1,1,20250313,NOP,ORL,16417830022401141Left Side(L)591,None


In [858]:
#wrong = shotdetail_data.loc[(shotdetail_data['SEASON_YEAR']=="2023-24")]

In [859]:
#wrong.to_sql(table_name, conn, if_exists='replace', index=False)

In [860]:
shot_zone_abbr = {
    'Above the Break 3': '3_AB',
    'In The Paint (Non-RA)': 'Paint',
    'Mid-Range': 'Mid',
    'Restricted Area': 'RA',
    'Left Corner 3': '3_LC',
    'Right Corner 3': '3_RC',
    'Backcourt': 'BC'
}

shotdetail_data['SHOT_ZONE_BASIC'] = shotdetail_data['SHOT_ZONE_BASIC'].map(shot_zone_abbr)

In [861]:
shotdetail_data.rename(columns={'SHOT_ATTEMPTED_FLAG':'FGA','SHOT_MADE_FLAG':'FGM'},inplace=True)

In [862]:
shotdetail_sum= pd.DataFrame(shotdetail_data.groupby(["PLAYER_ID",'GAME_DATE','SHOT_ZONE_BASIC'])[['FGA','FGM']].sum()).reset_index()

In [863]:
#shotdetail_sum.drop_duplicates()

In [864]:
#shotdetail_data.rename(columns = {'SHOT_ATTEMPTED_FLAG_60G_Msum':'FGA_60G_Msum','SHOT_MADE_FLAG_60G_Msum':'FGM_60G_Msum','SHOT_ATTEMPTED_FLAG_60G_Msavg':'FGA_60G_Mavg','SHOT_MADE_FLAG_60G_Msavg':'FGM_60G_Mavg'}, inplace=True)
shotdetail_pivot = shotdetail_sum.pivot(index=['PLAYER_ID','GAME_DATE'], 
                    columns='SHOT_ZONE_BASIC', 
                    values=['FGA','FGM', ])

shotdetail_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in shotdetail_pivot.columns]
shotdetail_pivot.reset_index(inplace=True)

In [865]:
shotdetail_pivot.fillna(0, inplace=True)

In [866]:
#shotdetail_sum= pd.DataFrame(shotdetail_pivot.groupby(["PLAYER_ID",'GAME_DATE'])[['SHOT_ATTEMPTED_FLAG','SHOT_MADE_FLAG']].sum()).reset_index()

In [867]:
#shotdetail_data['SHOT_ATTEMPTED_FLAG'].sum()
#shotdetail_data['SHOT_MADE_FLAG'].sum()

In [868]:
#shotdetail_sum.loc[(shotdetail_sum['PLAYER_NAME'].str.contains('LeBron James', na=False))&(shotdetail_sum['SHOT_ZONE_BASIC'].str.contains('Back', na=False))]

In [869]:
shotdetail_pivot.columns

Index(['PLAYER_ID', 'GAME_DATE', 'FGA_3_AB', 'FGA_3_LC', 'FGA_3_RC', 'FGA_BC',
       'FGA_Mid', 'FGA_Paint', 'FGA_RA', 'FGM_3_AB', 'FGM_3_LC', 'FGM_3_RC',
       'FGM_BC', 'FGM_Mid', 'FGM_Paint', 'FGM_RA'],
      dtype='object')

In [870]:
gamelogs_shotdetail_sum = calculate_weighted_rolling_stats(
    gamelogs=shotdetail_pivot,
    rolling_columns=['FGA_3_AB', 'FGA_3_LC',
       'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA', 'FGM_3_AB',
       'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint', 'FGM_RA'],
    method = 'sum'
)
gamelogs_shotdetail_avg = calculate_weighted_rolling_stats(
    gamelogs=shotdetail_pivot,
    rolling_columns=['FGA_3_AB', 'FGA_3_LC',
       'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA', 'FGM_3_AB',
       'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint', 'FGM_RA'],
    method = 'average'
)

In [871]:
gamelogs_shotdetail_sum['GAME_DATE'] = pd.to_datetime(gamelogs_shotdetail_sum['GAME_DATE'])
gamelogs_shotdetail_avg['GAME_DATE'] = pd.to_datetime(gamelogs_shotdetail_avg['GAME_DATE'])

In [872]:
gamelogs_shotdetail = gamelogs_shotdetail_sum.merge(gamelogs_shotdetail_avg, how='left')

In [873]:
#gamelogs_shotdetail.drop_duplicates()

In [874]:
gamelogs_shotdetail_60Day = pd.DataFrame(gamelogs_shotdetail[['PLAYER_ID', 'GAME_DATE', 'FGA_3_AB_60G_Mavg',
       'FGA_3_LC_60G_Mavg', 'FGA_3_RC_60G_Mavg', 'FGA_BC_60G_Mavg',
       'FGA_Mid_60G_Mavg', 'FGA_Paint_60G_Mavg', 'FGA_RA_60G_Mavg',
       'FGM_3_AB_60G_Mavg', 'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg',
       'FGM_BC_60G_Mavg', 'FGM_Mid_60G_Mavg', 'FGM_Paint_60G_Mavg',
       'FGM_RA_60G_Mavg']])

In [875]:


#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
gamelogs_shotdetail_60Day['id'] = gamelogs_shotdetail_60Day['GAME_DATE'].astype(str)+"_"+gamelogs_shotdetail_60Day['PLAYER_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "shotdetail_player_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gamelogs_shotdetail_60Day.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gamelogs_shotdetail_60Day[~gamelogs_shotdetail_60Day['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'shotdetail_player_rolling' already exists. Checking for new records...
Inserted 1076 new records into 'shotdetail_player_rolling'.


In [876]:
most_recent_mask = gamelogs_shotdetail_60Day['GAME_DATE'] == gamelogs_shotdetail_60Day.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
shotdetail_60Day_new = gamelogs_shotdetail_60Day[most_recent_mask]

In [877]:
#shotdetail_60Day_new.columns

In [878]:
#shotdetail_60Day_new.rename(columns = {'SHOT_ATTEMPTED_FLAG_60G_Msum':'FGA_60G_Msum','SHOT_MADE_FLAG_60G_Msum':'FGM_60G_Msum','SHOT_ATTEMPTED_FLAG_60G_Msavg':'FGA_60G_Mavg','SHOT_MADE_FLAG_60G_Msavg':'FGM_60G_Mavg'}, inplace=True)

# Tracking Data: Driveses


In [879]:
##### Drives per Min,  FGA per Drives, Pass per Drive,  Ast per Pass 

In [880]:
table_name = "drives_player_data"
drives_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [881]:
drives_data.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W',
       'L', 'MIN', 'DRIVES', 'DRIVE_FGM', 'DRIVE_FGA', 'DRIVE_FG_PCT',
       'DRIVE_FTM', 'DRIVE_FTA', 'DRIVE_FT_PCT', 'DRIVE_PTS', 'DRIVE_PTS_PCT',
       'DRIVE_PASSES', 'DRIVE_PASSES_PCT', 'DRIVE_AST', 'DRIVE_AST_PCT',
       'DRIVE_TOV', 'DRIVE_TOV_PCT', 'DRIVE_PF', 'DRIVE_PF_PCT', 'GAME_DATE',
       'Tracking', 'id', 'SEASON_YEAR'],
      dtype='object')

In [882]:
drives_60Day= calculate_weighted_rolling_stats(
    gamelogs=drives_data,
    rolling_columns=['MIN','DRIVES','DRIVE_FGA','DRIVE_PASSES','DRIVE_AST','DRIVE_PTS'],
    method = 'sum'
)

In [883]:
drives_60Day.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W',
       'L', 'MIN', 'DRIVES', 'DRIVE_FGM', 'DRIVE_FGA', 'DRIVE_FG_PCT',
       'DRIVE_FTM', 'DRIVE_FTA', 'DRIVE_FT_PCT', 'DRIVE_PTS', 'DRIVE_PTS_PCT',
       'DRIVE_PASSES', 'DRIVE_PASSES_PCT', 'DRIVE_AST', 'DRIVE_AST_PCT',
       'DRIVE_TOV', 'DRIVE_TOV_PCT', 'DRIVE_PF', 'DRIVE_PF_PCT', 'GAME_DATE',
       'Tracking', 'id', 'SEASON_YEAR', 'GAMES_IN_WINDOW_60G', 'MIN_60G_Sum',
       'DRIVES_60G_Sum', 'DRIVE_FGA_60G_Sum', 'DRIVE_PASSES_60G_Sum',
       'DRIVE_AST_60G_Sum', 'DRIVE_PTS_60G_Sum'],
      dtype='object')

In [884]:
drives_60Day['DRIVES/MIN_60Day'] = (drives_60Day['DRIVES_60G_Sum']/drives_60Day['MIN_60G_Sum'] ).round(2)
drives_60Day['DRIVES_AST_PASS_RATE'] = (drives_60Day['DRIVE_AST_60G_Sum']/drives_60Day['DRIVE_PASSES_60G_Sum'] ).round(2)
drives_60Day['DRIVES_PASS_RATE'] = (drives_60Day['DRIVE_PASSES_60G_Sum']/drives_60Day['DRIVES_60G_Sum'] ).round(2)
drives_60Day['SHOTS_DRIVE_RATE'] =( drives_60Day['DRIVE_FGA_60G_Sum']/drives_60Day['DRIVES_60G_Sum'] ).round(2)
drives_60Day['PTS_PER_DRIVE'] =( drives_60Day['DRIVE_PTS_60G_Sum']/drives_60Day['DRIVES_60G_Sum'] ).round(2)

In [885]:
drives_60Day_sum = pd.DataFrame(drives_60Day[['PLAYER_ID','PLAYER_NAME','GAME_DATE','Tracking','DRIVES_60G_Sum','DRIVE_FGA_60G_Sum','MIN_60G_Sum','DRIVE_PASSES_60G_Sum','DRIVE_AST_60G_Sum',
                                 'DRIVES/MIN_60Day','PTS_PER_DRIVE','DRIVES_AST_PASS_RATE','DRIVES_PASS_RATE','SHOTS_DRIVE_RATE']])

In [886]:

#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
drives_60Day_sum['id'] = drives_60Day_sum['GAME_DATE'].astype(str)+"_"+drives_60Day_sum['PLAYER_ID'].astype(str)+"_"+drives_60Day_sum['Tracking'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "drives_player_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    drives_60Day_sum.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = drives_60Day_sum[~drives_60Day_sum['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'drives_player_rolling' already exists. Checking for new records...
Inserted 1092 new records into 'drives_player_rolling'.


In [887]:
most_recent_mask = drives_60Day_sum['GAME_DATE'] == drives_60Day_sum.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
drives_60Day_new = drives_60Day_sum[most_recent_mask]

In [888]:
drives_60Day_new.fillna(0, inplace=True)

/var/folders/sj/rsx4zyld6flczf6q6r20b1c80000gn/T/ipykernel_25156/1036217490.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drives_60Day_new.fillna(0, inplace=True)


# Tracking Data: CatchShoot

In [889]:
table_name = "catchshoot_player_data"
CS_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [890]:
CS_60Day = calculate_weighted_rolling_stats(
    gamelogs=CS_data,
    rolling_columns=['MIN','CATCH_SHOOT_FGM','CATCH_SHOOT_FGA','CATCH_SHOOT_FG3M','CATCH_SHOOT_FG3A'],
    method = 'sum'
)

In [891]:
CS_60Day['CS_FG2_PCT_60Day'] =((CS_60Day['CATCH_SHOOT_FGM_60G_Sum']-CS_60Day['CATCH_SHOOT_FG3M_60G_Sum'])/ (CS_60Day['CATCH_SHOOT_FGA_60G_Sum']-CS_60Day['CATCH_SHOOT_FG3A_60G_Sum'])).round(3)
CS_60Day['CS_FG3_PCT_60Day'] = ((CS_60Day['CATCH_SHOOT_FG3M_60G_Sum'])/ (CS_60Day['CATCH_SHOOT_FG3A_60G_Sum'])).round(3)
CS_60Day['CS_FGA_rate_60Day'] = ((CS_60Day['CATCH_SHOOT_FGA_60G_Sum'])/ (CS_60Day['MIN_60G_Sum'])).round(2)
CS_60Day['CS_FG3A_rate_60Day'] = ((CS_60Day['CATCH_SHOOT_FG3A_60G_Sum'])/ (CS_60Day['MIN_60G_Sum'])).round(2)

In [892]:
CS_60Day_sum = pd.DataFrame(CS_60Day[['PLAYER_ID','PLAYER_NAME','GAME_DATE','MIN_60G_Sum','CATCH_SHOOT_FGM_60G_Sum','CATCH_SHOOT_FGA_60G_Sum','CATCH_SHOOT_FG3M_60G_Sum','CATCH_SHOOT_FG3A_60G_Sum',
          'CS_FG2_PCT_60Day','CS_FG3_PCT_60Day','CS_FGA_rate_60Day','CS_FG3A_rate_60Day']])

In [893]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
CS_60Day_sum['id'] = CS_60Day_sum['GAME_DATE'].astype(str)+"_"+CS_60Day_sum['PLAYER_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "catchshoot_player_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    CS_60Day_sum.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = CS_60Day_sum[~CS_60Day_sum['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'catchshoot_player_rolling' already exists. Checking for new records...
Inserted 1252 new records into 'catchshoot_player_rolling'.


In [894]:
most_recent_mask = CS_60Day_sum['GAME_DATE'] == CS_60Day_sum.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
CS_60Day_new = CS_60Day_sum[most_recent_mask]

In [895]:
CS_60Day_new.fillna(0, inplace=True)

/var/folders/sj/rsx4zyld6flczf6q6r20b1c80000gn/T/ipykernel_25156/3545371360.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CS_60Day_new.fillna(0, inplace=True)


# Tracking Data: Pullup

In [896]:
table_name = "pullup_player_data"
PU_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [897]:
PU_data

,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,MIN,PULL_UP_FGM,PULL_UP_FGA,PULL_UP_FG_PCT,PULL_UP_PTS,PULL_UP_FG3M,PULL_UP_FG3A,PULL_UP_FG3_PCT,PULL_UP_EFG_PCT,GAME_DATE,Tracking,id,SEASON_YEAR
0,1630639,A.J. Lawson,1610612742,DAL,1,0,1,23.5,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,04/14/2024,PullUpShot,1630639_04/14/2024_PullUpShot,2023-24
1,1631100,AJ Griffin,1610612737,ATL,1,0,1,19.9,2.0,7.0,0.286,4.0,0.0,4.0,0.0,0.286,04/14/2024,PullUpShot,1631100_04/14/2024_PullUpShot,2023-24
2,203932,Aaron Gordon,1610612743,DEN,1,1,0,26.1,0.0,0.0,None,0.0,0.0,0.0,None,None,04/14/2024,PullUpShot,203932_04/14/2024_PullUpShot,2023-24
3,1628988,Aaron Holiday,1610612745,HOU,1,1,0,16.7,1.0,2.0,0.5,2.0,0.0,1.0,0.0,0.5,04/14/2024,PullUpShot,1628988_04/14/2024_PullUpShot,2023-24
4,1630174,Aaron Nesmith,1610612754,IND,1,1,0,19.8,0.0,0.0,None,0.0,0.0,0.0,None,None,04/14/2024,PullUpShot,1630174_04/14/2024_PullUpShot,2023-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46976,1642274,Yves Missi,1610612740,NOP,1,0,1,27.6,1.0,1.0,1.0,2.0,None,None,None,1.0,03/06/2025,PullUpShot,1642274_03/06/2025_PullUpShot,2024-25
46977,1642258,Zaccharie Risacher,1610612737,ATL,1,1,0,24.9,0.0,0.0,None,0.0,0.0,0.0,None,None,03/06/2025,PullUpShot,1642258_03/06/2025_PullUpShot,2024-25
46978,1628380,Zach Collins,1610612741,CHI,1,1,0,12.0,0.0,0.0,None,0.0,0.0,0.0,None,None,03/06/2025,PullUpShot,1628380_03/06/2025_PullUpShot,2024-25
46979,1630533,Ziaire Williams,1610612751,BKN,1,0,1,24.6,0.0,1.0,0.0,0.0,0.0,0.0,None,0.0,03/06/2025,PullUpShot,1630533_03/06/2025_PullUpShot,2024-25


In [898]:
PU_60Day = calculate_weighted_rolling_stats(
    gamelogs=PU_data,
    rolling_columns=['MIN','PULL_UP_FGM','PULL_UP_FGA','PULL_UP_FG3M','PULL_UP_FG3A'],
    method = 'sum'
)

In [899]:
PU_60Day.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W',
       'L', 'MIN', 'PULL_UP_FGM', 'PULL_UP_FGA', 'PULL_UP_FG_PCT',
       'PULL_UP_PTS', 'PULL_UP_FG3M', 'PULL_UP_FG3A', 'PULL_UP_FG3_PCT',
       'PULL_UP_EFG_PCT', 'GAME_DATE', 'Tracking', 'id', 'SEASON_YEAR',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Sum', 'PULL_UP_FGM_60G_Sum',
       'PULL_UP_FGA_60G_Sum', 'PULL_UP_FG3M_60G_Sum', 'PULL_UP_FG3A_60G_Sum'],
      dtype='object')

In [900]:
PU_60Day['PU_FG2_PCT_60Day'] =((PU_60Day['PULL_UP_FGM_60G_Sum']-PU_60Day['PULL_UP_FG3M_60G_Sum'])/ (PU_60Day['PULL_UP_FGA_60G_Sum']-PU_60Day['PULL_UP_FG3A_60G_Sum'])).round(3)
PU_60Day['PU_FG3_PCT_60Day'] = ((PU_60Day['PULL_UP_FG3M_60G_Sum'])/ (PU_60Day['PULL_UP_FG3A_60G_Sum'])).round(3)
PU_60Day['PU_FGA_PCT_60Day'] = ((PU_60Day['PULL_UP_FGM_60G_Sum'])/ (PU_60Day['PULL_UP_FGA_60G_Sum'])).round(3)
PU_60Day['PU_FGA_rate_60Day'] = ((PU_60Day['PULL_UP_FGA_60G_Sum'])/ (PU_60Day['MIN_60G_Sum'])).round(2)
PU_60Day['PU_FG3A_rate_60Day'] = ((PU_60Day['PULL_UP_FG3A_60G_Sum'])/ (PU_60Day['MIN_60G_Sum'])).round(2)

In [901]:
PU_60Day_sum = pd.DataFrame(PU_60Day[['PLAYER_ID','PLAYER_NAME','GAME_DATE','MIN_60G_Sum','PULL_UP_FGM_60G_Sum','PULL_UP_FGA_60G_Sum','PULL_UP_FG3M_60G_Sum','PULL_UP_FG3A_60G_Sum',
          'PU_FG2_PCT_60Day','PU_FG3_PCT_60Day','PU_FGA_rate_60Day','PU_FG3A_rate_60Day','PU_FGA_PCT_60Day']])

In [902]:
PU_60Day_sum.fillna(0, inplace=True)

In [903]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
PU_60Day_sum['id'] = PU_60Day_sum['GAME_DATE'].astype(str)+"_"+PU_60Day_sum['PLAYER_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "pullup_player_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    PU_60Day_sum.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = PU_60Day_sum[~PU_60Day_sum['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'pullup_player_rolling' already exists. Checking for new records...
Inserted 1252 new records into 'pullup_player_rolling'.


In [904]:
most_recent_mask = PU_60Day_sum['GAME_DATE'] == PU_60Day_sum.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
PU_60Day_new = PU_60Day_sum[most_recent_mask]

# Tracking Data: Passing

In [905]:
table_name = "passing_data"
passing_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [906]:


passing_60Day = calculate_weighted_rolling_stats(
    gamelogs=passing_data,
    rolling_columns= ['MIN','PASSES_MADE','PASSES_RECEIVED','AST','POTENTIAL_AST','SECONDARY_AST'],
    method = 'sum'
)

In [907]:
passing_60Day['PASSES_MIN_60G_Sum'] = passing_60Day['PASSES_MADE_60G_Sum']/passing_60Day['MIN_60G_Sum']
passing_60Day['AST_PASS_60G_Sum'] = passing_60Day['AST_60G_Sum']/passing_60Day['PASSES_MADE_60G_Sum']

In [908]:
passing_60Day.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W',
       'L', 'MIN', 'PASSES_MADE', 'PASSES_RECEIVED', 'AST', 'FT_AST',
       'SECONDARY_AST', 'POTENTIAL_AST', 'AST_POINTS_CREATED', 'AST_ADJ',
       'AST_TO_PASS_PCT', 'AST_TO_PASS_PCT_ADJ', 'GAME_DATE', 'Tracking', 'id',
       'SEASON_YEAR', 'GAMES_IN_WINDOW_60G', 'MIN_60G_Sum',
       'PASSES_MADE_60G_Sum', 'PASSES_RECEIVED_60G_Sum', 'AST_60G_Sum',
       'POTENTIAL_AST_60G_Sum', 'SECONDARY_AST_60G_Sum', 'PASSES_MIN_60G_Sum',
       'AST_PASS_60G_Sum'],
      dtype='object')

In [909]:
passing_60Day_sum = pd.DataFrame(passing_60Day[['PLAYER_ID','PLAYER_NAME','GAME_DATE', 'PASSES_MADE', 'PASSES_RECEIVED', 'AST', 'FT_AST',
       'SECONDARY_AST', 'POTENTIAL_AST', 'AST_POINTS_CREATED', 'AST_ADJ',
       'AST_TO_PASS_PCT', 'AST_TO_PASS_PCT_ADJ','PASSES_MADE_60G_Sum', 'PASSES_RECEIVED_60G_Sum', 'AST_60G_Sum',
       'POTENTIAL_AST_60G_Sum', 'SECONDARY_AST_60G_Sum', 'PASSES_MIN_60G_Sum','MIN_60G_Sum',
       'AST_PASS_60G_Sum']])

In [910]:
# Calculate efficiency metrics with safe division
def safe_div(a, b):
    return np.where(b != 0, a / b, 0)


passing_60Day_sum['AST_PER_MIN'] = safe_div(passing_60Day_sum['AST_60G_Sum'], passing_60Day_sum['MIN_60G_Sum'])
passing_60Day_sum['AST_TO_POTENTIAL'] = safe_div(passing_60Day_sum['AST_60G_Sum'], passing_60Day_sum['POTENTIAL_AST_60G_Sum'])
passing_60Day_sum['SECONDARY_AST_RATE'] = safe_div(passing_60Day_sum['SECONDARY_AST'], passing_60Day_sum['AST_60G_Sum'])
passing_60Day_sum['POINTS_PER_AST'] = safe_div(passing_60Day_sum['AST_POINTS_CREATED'], passing_60Day_sum['AST_60G_Sum'])

In [911]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
passing_60Day_sum['id'] = passing_60Day_sum['GAME_DATE'].astype(str)+"_"+passing_60Day_sum['PLAYER_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "passing_player_rolling_new"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    passing_60Day_sum.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = passing_60Day_sum[~passing_60Day_sum['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'passing_player_rolling_new' already exists. Checking for new records...
Inserted 980 new records into 'passing_player_rolling_new'.


In [912]:
most_recent_mask = passing_60Day_sum['GAME_DATE'] == passing_60Day_sum.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
passing_60Day_new = passing_60Day_sum[most_recent_mask]

In [913]:
passing_60Day_new

,PLAYER_ID,PLAYER_NAME,GAME_DATE,PASSES_MADE,PASSES_RECEIVED,AST,FT_AST,SECONDARY_AST,POTENTIAL_AST,AST_POINTS_CREATED,AST_ADJ,AST_TO_PASS_PCT,AST_TO_PASS_PCT_ADJ,PASSES_MADE_60G_Sum,PASSES_RECEIVED_60G_Sum,AST_60G_Sum,POTENTIAL_AST_60G_Sum,SECONDARY_AST_60G_Sum,PASSES_MIN_60G_Sum,MIN_60G_Sum,AST_PASS_60G_Sum,AST_PER_MIN,AST_TO_POTENTIAL,SECONDARY_AST_RATE,POINTS_PER_AST,id
125,2544,LeBron James,2025-03-06,58.0,67.0,8.0,1.0,3.0,15.0,22.0,12.0,0.138,0.207,3522.0,4068.0,520.0,943.0,74.0,1.674193,2103.7,0.147643,0.247184,0.551432,0.005769,0.042308,2025-03-06_2544
245,101108,Chris Paul,2025-03-12,42.0,48.0,9.0,0.0,0.0,8.0,19.0,9.0,0.214,0.214,3114.0,3071.0,478.0,800.0,41.0,1.816909,1713.9,0.153500,0.278896,0.597500,0.000000,0.039749,2025-03-12_101108
337,200768,Kyle Lowry,2025-02-09,8.0,4.0,1.0,0.0,1.0,1.0,2.0,2.0,0.125,0.250,2313.0,2078.0,209.0,390.0,14.0,1.674752,1381.1,0.090359,0.151329,0.535897,0.004785,0.009569,2025-02-09_200768
368,200782,P.J. Tucker,2024-04-14,5.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,346.0,167.0,16.0,30.0,2.0,0.711495,486.3,0.046243,0.032902,0.533333,0.000000,0.000000,2024-04-14_200782
495,201142,Kevin Durant,2025-03-12,30.0,32.0,0.0,0.0,1.0,4.0,0.0,1.0,0.000,0.033,2297.0,2903.0,254.0,448.0,49.0,1.029030,2232.2,0.110579,0.113789,0.566964,0.003937,0.000000,2025-03-12_201142
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46904,1642461,Spencer Jones,2025-03-12,4.0,3.0,1.0,0.0,0.0,1.0,3.0,1.0,0.250,0.250,30.0,29.0,1.0,3.0,0.0,0.751880,39.9,0.033333,0.025063,0.333333,0.000000,3.000000,2025-03-12_1642461
46912,1642484,RayJ Dennis,2025-03-10,12.0,13.0,3.0,0.0,1.0,3.0,7.0,4.0,0.250,0.333,43.0,46.0,5.0,7.0,1.0,1.877729,22.9,0.116279,0.218341,0.714286,0.200000,1.400000,2025-03-10_1642484
46918,1642502,Malevy Leons,2024-11-15,5.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.000,0.000,19.0,19.0,1.0,1.0,0.0,0.904762,21.0,0.052632,0.047619,1.000000,0.000000,0.000000,2024-11-15_1642502
46935,1642505,Alex Ducas,2025-03-02,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.000,35.0,44.0,2.0,7.0,0.0,0.461133,75.9,0.057143,0.026350,0.285714,0.000000,0.000000,2025-03-02_1642505


# Tracking Data: Rebounding

In [914]:
table_name = "rebounding_player_data"
rebounding_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [915]:
#rebounding_columns = ['MIN','OREB_CHANCES','OREB','OREB_CONTEST','DREB','DREB_CHANCES','DREB_CONTEST']
rebounding_60Day = calculate_weighted_rolling_stats(
    gamelogs=rebounding_data,
    rolling_columns= ['MIN','OREB_CHANCES','OREB','OREB_CONTEST','DREB','DREB_CHANCES','DREB_CONTEST'],
    method = 'sum'
)

In [916]:
rebounding_60Day

,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,MIN,OREB,OREB_CONTEST,OREB_UNCONTEST,OREB_CONTEST_PCT,OREB_CHANCES,OREB_CHANCE_PCT,OREB_CHANCE_DEFER,OREB_CHANCE_PCT_ADJ,AVG_OREB_DIST,DREB,DREB_CONTEST,DREB_UNCONTEST,DREB_CONTEST_PCT,DREB_CHANCES,DREB_CHANCE_PCT,DREB_CHANCE_DEFER,DREB_CHANCE_PCT_ADJ,AVG_DREB_DIST,REB,REB_CONTEST,REB_UNCONTEST,REB_CONTEST_PCT,REB_CHANCES,REB_CHANCE_PCT,REB_CHANCE_DEFER,REB_CHANCE_PCT_ADJ,AVG_REB_DIST,GAME_DATE,Tracking,id,SEASON_YEAR,GAMES_IN_WINDOW_60G,MIN_60G_Sum,OREB_CHANCES_60G_Sum,OREB_60G_Sum,OREB_CONTEST_60G_Sum,DREB_60G_Sum,DREB_CHANCES_60G_Sum,DREB_CONTEST_60G_Sum
0,2544,LeBron James,1610612747,LAL,1,0,1,29.0,1.0,1.0,0.0,1.0,2.0,0.500,0.0,0.500,1.6,7.0,2.0,5.0,0.286,9.0,0.778,0.0,0.778,6.0,8.0,3.0,5.0,0.375,11.0,0.727,0.0,0.727,5.5,2023-10-24,Rebounding,2544_10/24/2023_Rebounding,2023-24,1.0,29.0,2.0,1.0,1.0,7.0,9.0,2.0
1,2544,LeBron James,1610612747,LAL,1,1,0,35.0,1.0,0.0,1.0,0.0,1.0,1.000,0.0,1.000,27.0,7.0,4.0,3.0,0.571,8.0,0.875,1.0,1.000,5.1,8.0,4.0,4.0,0.500,9.0,0.889,1.0,1.000,7.9,2023-10-26,Rebounding,2544_10/26/2023_Rebounding,2023-24,2.0,64.0,3.0,2.0,1.0,14.0,17.0,6.0
2,2544,LeBron James,1610612747,LAL,1,0,1,39.1,0.0,0.0,0.0,0.0,2.0,0.000,0.0,0.000,0.0,15.0,1.0,12.0,0.067,15.0,1.000,1.0,1.071,10.0,15.0,1.0,12.0,0.067,16.0,0.938,1.0,1.000,10.0,2023-10-29,Rebounding,2544_10/29/2023_Rebounding,2023-24,3.0,103.1,5.0,2.0,1.0,29.0,32.0,7.0
3,2544,LeBron James,1610612747,LAL,1,1,0,32.8,0.0,0.0,0.0,0.0,1.0,0.000,0.0,0.000,0.0,3.0,1.0,2.0,0.333,6.0,0.500,0.0,0.500,5.6,3.0,1.0,2.0,0.333,6.0,0.500,0.0,0.500,5.6,2023-10-30,Rebounding,2544_10/30/2023_Rebounding,2023-24,4.0,135.9,6.0,2.0,1.0,32.0,38.0,8.0
4,2544,LeBron James,1610612747,LAL,1,1,0,42.5,0.0,0.0,0.0,0.0,1.0,0.000,0.0,0.000,0.0,12.0,3.0,9.0,0.250,14.0,0.857,1.0,0.923,7.9,12.0,3.0,9.0,0.250,15.0,0.800,1.0,0.857,7.9,2023-11-01,Rebounding,2544_11/01/2023_Rebounding,2023-24,5.0,178.4,7.0,2.0,1.0,44.0,52.0,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41228,1642505,Alex Ducas,1610612760,OKC,1,1,0,6.2,1.0,1.0,0.0,1.0,3.0,0.333,0.0,0.333,5.3,1.0,1.0,0.0,1.000,1.0,1.000,0.0,1.000,10.3,2.0,2.0,0.0,1.000,4.0,0.500,0.0,0.500,7.8,2025-02-10,Rebounding,1642505_02/10/2025_Rebounding,2024-25,7.0,44.0,13.0,5.0,3.0,10.0,14.0,5.0
41229,1642530,Yuki Kawamura,1610612763,MEM,1,1,0,4.7,0.0,0.0,0.0,0.0,1.0,0.000,0.0,0.000,0.0,2.0,0.0,2.0,0.000,2.0,1.000,0.0,1.000,11.4,2.0,0.0,2.0,0.000,2.0,1.000,0.0,1.000,11.4,2024-11-08,Rebounding,1642530_11/08/2024_Rebounding,2024-25,1.0,4.7,1.0,0.0,0.0,2.0,2.0,0.0
41230,1642530,Yuki Kawamura,1610612763,MEM,1,0,1,11.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,3.0,0.0,3.0,0.000,3.0,1.000,0.0,1.000,11.5,3.0,0.0,3.0,0.000,3.0,1.000,0.0,1.000,11.5,2024-12-29,Rebounding,1642530_12/29/2024_Rebounding,2024-25,2.0,15.7,1.0,0.0,0.0,5.0,5.0,0.0
41231,1642530,Yuki Kawamura,1610612763,MEM,1,1,0,2.3,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.000,0.0,1.0,0.0,1.0,0.000,1.0,1.000,0.0,1.000,4.4,1.0,0.0,1.0,0.000,1.0,1.000,0.0,1.000,4.4,2025-01-06,Rebounding,1642530_01/06/2025_Rebounding,2024-25,3.0,18.0,1.0,0.0,0.0,6.0,6.0,0.0


In [917]:
rebounding_60Day['OREB_PER_CHANCE'] = rebounding_60Day['OREB'] / rebounding_60Day['OREB_CHANCES']
rebounding_60Day['CONTEST_OREB_PER_CHANCE'] = rebounding_60Day['OREB_CONTEST'] / rebounding_60Day['OREB_CHANCES']

rebounding_60Day['DREB_PER_CHANCE'] = rebounding_60Day['DREB'] / rebounding_60Day['DREB_CHANCES']
rebounding_60Day['CONTEST_DREB_PER_CHANCE'] = rebounding_60Day['DREB_CONTEST'] / rebounding_60Day['DREB_CHANCES']

rebounding_60Day['OREB_PER_CHANCE_60G_Sum'] = rebounding_60Day['OREB_60G_Sum'] / rebounding_60Day['OREB_CHANCES_60G_Sum']
rebounding_60Day['CONTEST_OREB_PER_CHANCE_60G_Sum'] = rebounding_60Day['OREB_CONTEST_60G_Sum'] / rebounding_60Day['OREB_CHANCES_60G_Sum']

rebounding_60Day['DREB_PER_CHANCE_60G_Sum'] = rebounding_60Day['DREB_60G_Sum'] / rebounding_60Day['DREB_CHANCES_60G_Sum']
rebounding_60Day['CONTEST_DREB_PER_CHANCE_60G_Sum'] = rebounding_60Day['DREB_CONTEST_60G_Sum'] / rebounding_60Day['DREB_CHANCES_60G_Sum']

rebounding_60Day['DREB_PER_CHANCE_60G_Sum'] = rebounding_60Day['DREB_60G_Sum'] / rebounding_60Day['OREB_CHANCES_60G_Sum']
rebounding_60Day['CONTEST_DREB_PER_CHANCE_60G_Sum'] = rebounding_60Day['DREB_CONTEST_60G_Sum'] / rebounding_60Day['DREB_CHANCES_60G_Sum']

In [918]:
rebounding_60Day_sum = pd.DataFrame(rebounding_60Day[['PLAYER_ID','PLAYER_NAME','GAME_DATE','MIN_60G_Sum','OREB_CHANCES_60G_Sum','OREB_60G_Sum','OREB_CONTEST_60G_Sum','DREB_60G_Sum','DREB_CHANCES_60G_Sum','DREB_CONTEST_60G_Sum',
                  'OREB_PER_CHANCE','CONTEST_OREB_PER_CHANCE','DREB_PER_CHANCE','CONTEST_DREB_PER_CHANCE','OREB_PER_CHANCE_60G_Sum','CONTEST_OREB_PER_CHANCE_60G_Sum','DREB_PER_CHANCE_60G_Sum','CONTEST_DREB_PER_CHANCE_60G_Sum',]])

In [919]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
rebounding_60Day_sum['id'] = rebounding_60Day_sum['GAME_DATE'].astype(str)+"_"+rebounding_60Day_sum['PLAYER_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "rebounding_player_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    rebounding_60Day_sum.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = rebounding_60Day_sum[~rebounding_60Day_sum['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'rebounding_player_rolling' already exists. Checking for new records...
Inserted 882 new records into 'rebounding_player_rolling'.


In [920]:
most_recent_mask = rebounding_60Day_sum['GAME_DATE'] == rebounding_60Day_sum.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
rebounding_60Day_new = rebounding_60Day_sum[most_recent_mask]

In [921]:
rebounding_60Day_new

,PLAYER_ID,PLAYER_NAME,GAME_DATE,MIN_60G_Sum,OREB_CHANCES_60G_Sum,OREB_60G_Sum,OREB_CONTEST_60G_Sum,DREB_60G_Sum,DREB_CHANCES_60G_Sum,DREB_CONTEST_60G_Sum,OREB_PER_CHANCE,CONTEST_OREB_PER_CHANCE,DREB_PER_CHANCE,CONTEST_DREB_PER_CHANCE,OREB_PER_CHANCE_60G_Sum,CONTEST_OREB_PER_CHANCE_60G_Sum,DREB_PER_CHANCE_60G_Sum,CONTEST_DREB_PER_CHANCE_60G_Sum,id
124,2544,LeBron James,2025-03-06,2103.7,165.0,60.0,31.0,423.0,595.0,97.0,0.333333,0.000000,0.647059,0.235294,0.363636,0.187879,2.563636,0.163025,2025-03-06_2544
238,101108,Chris Paul,2025-03-12,1701.1,47.0,27.0,9.0,214.0,363.0,30.0,NaN,NaN,1.000000,1.000000,0.574468,0.191489,4.553191,0.082645,2025-03-12_101108
322,200768,Kyle Lowry,2025-02-09,1418.1,85.0,25.0,7.0,130.0,290.0,22.0,NaN,NaN,1.000000,0.000000,0.294118,0.082353,1.529412,0.075862,2025-02-09_200768
347,200782,P.J. Tucker,2024-04-12,434.8,60.0,28.0,11.0,57.0,99.0,12.0,0.666667,0.333333,0.600000,0.200000,0.466667,0.183333,0.950000,0.121212,2024-04-12_200782
474,201142,Kevin Durant,2025-03-12,2232.2,79.0,24.0,7.0,338.0,476.0,87.0,0.000000,0.000000,0.875000,0.375000,0.303797,0.088608,4.278481,0.182773,2025-03-12_201142
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41214,1642461,Spencer Jones,2025-03-12,23.0,2.0,2.0,0.0,5.0,5.0,2.0,NaN,NaN,1.000000,0.000000,1.000000,0.000000,2.500000,0.400000,2025-03-12_1642461
41218,1642484,RayJ Dennis,2025-03-10,16.6,2.0,2.0,0.0,4.0,5.0,1.0,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,2.000000,0.200000,2025-03-10_1642484
41221,1642502,Malevy Leons,2024-11-15,10.4,2.0,1.0,1.0,2.0,2.0,0.0,NaN,NaN,1.000000,0.000000,0.500000,0.500000,1.000000,0.000000,2024-11-15_1642502
41228,1642505,Alex Ducas,2025-02-10,44.0,13.0,5.0,3.0,10.0,14.0,5.0,0.333333,0.333333,1.000000,1.000000,0.384615,0.230769,0.769231,0.357143,2025-02-10_1642505


# Tracking Data: Touches

In [922]:
table_name = "speeddistance_player_data"
speeddistance_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [923]:
#speeddistance_columns = ['MIN','DIST_MILES_OFF','DIST_MILES_DEF','AVG_SPEED_OFF','AVG_SPEED_DEF']

speeddistance_60Day = calculate_weighted_rolling_stats(
    gamelogs=speeddistance_data,
    rolling_columns= ['MIN','DIST_MILES_OFF','DIST_MILES_DEF','AVG_SPEED_OFF','AVG_SPEED_DEF'],
    method = 'sum'
)

In [924]:
speeddistance_60Day['OFFmiles_PER_MIN'] = speeddistance_60Day['DIST_MILES_OFF_60G_Sum']/speeddistance_60Day['MIN_60G_Sum']
speeddistance_60Day['DEFmiles_PER_MIN'] = speeddistance_60Day['DIST_MILES_DEF_60G_Sum']/speeddistance_60Day['MIN_60G_Sum']

In [925]:
speeddistance_60Day_sum = pd.DataFrame(speeddistance_60Day[['PLAYER_ID','PLAYER_NAME','GAME_DATE','MIN_60G_Sum','DIST_MILES_OFF_60G_Sum','DIST_MILES_DEF_60G_Sum','AVG_SPEED_OFF_60G_Sum','AVG_SPEED_DEF_60G_Sum','OFFmiles_PER_MIN','DEFmiles_PER_MIN']])

In [926]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
speeddistance_60Day_sum['id'] = speeddistance_60Day_sum['GAME_DATE'].astype(str)+"_"+speeddistance_60Day_sum['PLAYER_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "speeddistance_player_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    speeddistance_60Day_sum.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = speeddistance_60Day_sum[~speeddistance_60Day_sum['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'speeddistance_player_rolling' already exists. Checking for new records...
Inserted 1092 new records into 'speeddistance_player_rolling'.


In [927]:
most_recent_mask = speeddistance_60Day_sum['GAME_DATE'] == speeddistance_60Day_sum.groupby('PLAYER_ID')['GAME_DATE'].transform('max')
speeddistance_60Day_new = speeddistance_60Day_sum[most_recent_mask]

In [928]:
speeddistance_60Day_new

,PLAYER_ID,PLAYER_NAME,GAME_DATE,MIN_60G_Sum,DIST_MILES_OFF_60G_Sum,DIST_MILES_DEF_60G_Sum,AVG_SPEED_OFF_60G_Sum,AVG_SPEED_DEF_60G_Sum,OFFmiles_PER_MIN,DEFmiles_PER_MIN,id
125,2544,LeBron James,2025-03-06,2103.18,74.47,62.43,235.69,202.37,0.035408,0.029684,2025-03-06_2544
245,101108,Chris Paul,2025-03-12,1713.46,59.98,51.07,238.01,201.43,0.035005,0.029805,2025-03-12_101108
337,200768,Kyle Lowry,2025-02-09,1380.92,53.18,43.18,255.69,215.11,0.038511,0.031269,2025-02-09_200768
368,200782,P.J. Tucker,2024-04-14,486.34,18.69,17.35,133.26,127.07,0.038430,0.035675,2024-04-14_200782
495,201142,Kevin Durant,2025-03-12,2231.37,80.25,68.56,235.27,215.43,0.035964,0.030726,2025-03-12_201142
...,...,...,...,...,...,...,...,...,...,...,...
46995,1642461,Spencer Jones,2025-03-12,39.80,1.76,1.45,70.89,48.82,0.044221,0.036432,2025-03-12_1642461
47003,1642484,RayJ Dennis,2025-03-10,22.85,0.99,0.85,34.80,36.10,0.043326,0.037199,2025-03-10_1642484
47009,1642502,Malevy Leons,2024-11-15,20.96,0.99,0.77,23.89,24.70,0.047233,0.036737,2024-11-15_1642502
47026,1642505,Alex Ducas,2025-03-02,75.66,3.36,2.58,72.33,69.52,0.044409,0.034100,2025-03-02_1642505


In [929]:
from nba_api.stats.endpoints import CommonAllPlayers
import pandas as pd

# Get all players for the 2023-24 season
players_data = CommonAllPlayers(is_only_current_season=1, league_id='00')
players_df = players_data.get_data_frames()[0]

# Extract relevant player IDs and names
player_ids = pd.DataFrame(players_df['PERSON_ID'])

In [930]:
player_ids.rename(columns={'PERSON_ID':'PLAYER_ID'},inplace=True)

In [931]:
#gl_60Day_new.rename(columns={"Player_ID":'PLAYER_ID',"Game_ID":'GAME_ID'}, inplace=True)

In [932]:
table_name = "player_info"
player_info = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [933]:
player_info.rename(columns={'PERSON_ID':'PLAYER_ID'}, inplace=True)

In [934]:
merged_df= gl_60Day_new.merge(player_info.drop(columns=['FIRST_NAME',"LAST_NAME"]), how='left')

In [935]:
merged_df.columns

Index(['SEASON_YEAR', 'PLAYER_ID', 'TEAM_ID', 'GAME_ID', 'GAME_DATE',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'PTS_60G_Mavg', 'REB_60G_Mavg',
       'AST_60G_Mavg', 'FGA_60G_Mavg', 'FG3M_60G_Mavg', 'FG3A_60G_Mavg',
       'FGA_60G_Sum', 'FG3M_60G_Sum', 'FG3A_60G_Sum', 'MIN_60G_Sum',
       'PTS_60G_Sum', 'HEIGHT_INCHES', 'WEIGHT', 'SEASON_EXP'],
      dtype='object')

In [936]:
#playtype_off_pivot_ = playtype_off_pivot[['PLAYER_ID', 'PLAYER_NAME', 'FGA/G_Cut', 'FGA/G_Handoff',
#       'FGA/G_Isolation', 'FGA/G_Misc', 'FGA/G_OffRebound', 'FGA/G_OffScreen',
#       'FGA/G_PRBallHandler', 'FGA/G_PRRollMan', 'FGA/G_Postup',
#       'FGA/G_Spotup', 'FGA/G_Transition', 'FG_PCT_Cut', 'FG_PCT_Handoff',
#       'FG_PCT_Isolation', 'FG_PCT_Misc', 'FG_PCT_OffRebound',
#       'FG_PCT_OffScreen', 'FG_PCT_PRBallHandler', 'FG_PCT_PRRollMan',
#       'FG_PCT_Postup', 'FG_PCT_Spotup', 'FG_PCT_Transition', 'POSS/G_Cut',
#       'POSS/G_Handoff', 'POSS/G_Isolation', 'POSS/G_Misc',
#       'POSS/G_OffRebound', 'POSS/G_OffScreen', 'POSS/G_PRBallHandler',
#       'POSS/G_PRRollMan', 'POSS/G_Postup', 'POSS/G_Spotup',
#       'POSS/G_Transition', 'POSS_PCT_Cut', 'POSS_PCT_Handoff',
#       'POSS_PCT_Isolation', 'POSS_PCT_Misc', 'POSS_PCT_OffRebound',
#       'POSS_PCT_OffScreen', 'POSS_PCT_PRBallHandler', 'POSS_PCT_PRRollMan',
#       'POSS_PCT_Postup', 'POSS_PCT_Spotup', 'POSS_PCT_Transition', 'PPP_Cut',
#       'PPP_Handoff', 'PPP_Isolation', 'PPP_Misc', 'PPP_OffRebound',
#       'PPP_OffScreen', 'PPP_PRBallHandler', 'PPP_PRRollMan', 'PPP_Postup',
#       'PPP_Spotup', 'PPP_Transition', 'PTS/MAKE_Cut', 'PTS/MAKE_Handoff',
#       'PTS/MAKE_Isolation', 'PTS/MAKE_Misc', 'PTS/MAKE_OffRebound',
#       'PTS/MAKE_OffScreen', 'PTS/MAKE_PRBallHandler', 'PTS/MAKE_PRRollMan',
#       'PTS/MAKE_Postup', 'PTS/MAKE_Spotup', 'PTS/MAKE_Transition',
#       'SHOT_FREQ_Cut', 'SHOT_FREQ_Handoff', 'SHOT_FREQ_Isolation',
#       'SHOT_FREQ_Misc', 'SHOT_FREQ_OffRebound', 'SHOT_FREQ_OffScreen',
#       'SHOT_FREQ_PRBallHandler', 'SHOT_FREQ_PRRollMan', 'SHOT_FREQ_Postup',
#       'SHOT_FREQ_Spotup', 'SHOT_FREQ_Transition']]

In [937]:
#playtype_off_pivot_.loc[playtype_off_pivot_['PLAYER_ID']==1626157]

In [938]:
results_df_player.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'OVERALL_SCORE_Cut',
       'OVERALL_SCORE_Handoff', 'OVERALL_SCORE_Isolation',
       'OVERALL_SCORE_Misc', 'OVERALL_SCORE_OffRebound',
       'OVERALL_SCORE_OffScreen', 'OVERALL_SCORE_PRBallHandler',
       'OVERALL_SCORE_PRRollMan', 'OVERALL_SCORE_Postup',
       'OVERALL_SCORE_Spotup', 'OVERALL_SCORE_Transition',
       'OVERALL_SCORE_PERCENTILE_Cut', 'OVERALL_SCORE_PERCENTILE_Handoff',
       'OVERALL_SCORE_PERCENTILE_Isolation', 'OVERALL_SCORE_PERCENTILE_Misc',
       'OVERALL_SCORE_PERCENTILE_OffRebound',
       'OVERALL_SCORE_PERCENTILE_OffScreen',
       'OVERALL_SCORE_PERCENTILE_PRBallHandler',
       'OVERALL_SCORE_PERCENTILE_PRRollMan', 'OVERALL_SCORE_PERCENTILE_Postup',
       'OVERALL_SCORE_PERCENTILE_Spotup',
       'OVERALL_SCORE_PERCENTILE_Transition', 'SCORING_EFFICIENCY_Cut',
       'SCORING_EFFICIENCY_Handoff', 'SCORING_EFFICIENCY_Isolation',
       'SCORING_EFFICIENCY_Misc', 'SCORING_EFFICIENCY_OffRebound',
       'SCORING_EFFICIE

In [939]:
merged_df_1 = merged_df.merge(results_df_player, how='left')

In [940]:
merged_df_1.loc[merged_df_1['PLAYER_NAME'].notna()]['PLAYER_NAME'].nunique()

509

In [941]:
merged_df_1.loc[merged_df_1['PLAYER_ID']==1626157]

,SEASON_YEAR,PLAYER_ID,TEAM_ID,GAME_ID,GAME_DATE,GAMES_IN_WINDOW_60G,MIN_60G_Mavg,PTS_60G_Mavg,REB_60G_Mavg,AST_60G_Mavg,FGA_60G_Mavg,FG3M_60G_Mavg,FG3A_60G_Mavg,FGA_60G_Sum,FG3M_60G_Sum,FG3A_60G_Sum,MIN_60G_Sum,PTS_60G_Sum,HEIGHT_INCHES,WEIGHT,SEASON_EXP,PLAYER_NAME,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,SCORING_EFFICIENCY_Cut,SCORING_EFFICIENCY_Handoff,SCORING_EFFICIENCY_Isolation,SCORING_EFFICIENCY_Misc,SCORING_EFFICIENCY_OffRebound,SCORING_EFFICIENCY_OffScreen,SCORING_EFFICIENCY_PRBallHandler,SCORING_EFFICIENCY_PRRollMan,SCORING_EFFICIENCY_Postup,SCORING_EFFICIENCY_Spotup,SCORING_EFFICIENCY_Transition,SCORING_EFFICIENCY_PERCENTILE_Cut,SCORING_EFFICIENCY_PERCENTILE_Handoff,SCORING_EFFICIENCY_PERCENTILE_Isolation,SCORING_EFFICIENCY_PERCENTILE_Misc,SCORING_EFFICIENCY_PERCENTILE_OffRebound,SCORING_EFFICIENCY_PERCENTILE_OffScreen,SCORING_EFFICIENCY_PERCENTILE_PRBallHandler,SCORING_EFFICIENCY_PERCENTILE_PRRollMan,SCORING_EFFICIENCY_PERCENTILE_Postup,SCORING_EFFICIENCY_PERCENTILE_Spotup,SCORING_EFFICIENCY_PERCENTILE_Transition,VOLUME_SCORE_Cut,VOLUME_SCORE_Handoff,VOLUME_SCORE_Isolation,VOLUME_SCORE_Misc,VOLUME_SCORE_OffRebound,VOLUME_SCORE_OffScreen,VOLUME_SCORE_PRBallHandler,VOLUME_SCORE_PRRollMan,VOLUME_SCORE_Postup,VOLUME_SCORE_Spotup,VOLUME_SCORE_Transition,VOLUME_SCORE_PERCENTILE_Cut,VOLUME_SCORE_PERCENTILE_Handoff,VOLUME_SCORE_PERCENTILE_Isolation,VOLUME_SCORE_PERCENTILE_Misc,VOLUME_SCORE_PERCENTILE_OffRebound,VOLUME_SCORE_PERCENTILE_OffScreen,VOLUME_SCORE_PERCENTILE_PRBallHandler,VOLUME_SCORE_PERCENTILE_PRRollMan,VOLUME_SCORE_PERCENTILE_Postup,VOLUME_SCORE_PERCENTILE_Spotup,VOLUME_SCORE_PERCENTILE_Transition,POSS_PCT_Cut,POSS_PCT_Handoff,POSS_PCT_Isolation,POSS_PCT_Misc,POSS_PCT_OffRebound,POSS_PCT_OffScreen,POSS_PCT_PRBallHandler,POSS_PCT_PRRollMan,POSS_PCT_Postup,POSS_PCT_Spotup,POSS_PCT_Transition
578,2024-25,1626157,1610612752,0022400953,2025-03-12,60.0,34.487944,23.333333,12.616667,3.216667,16.166667,1.983333,4.816667,970.0,119.0,289.0,2069.276667,1400.0,84.0,248,9.0,Karl-Anthony Towns,50.9,38.0,50.3,60.6,76.5,48.2,31.4,67.4,57.5,63.0,51.6,82.7,40.8,83.3,93.9,99.3,78.1,22.4,96.1,90.8,97.2,82.2,54.9,51.0,48.8,58.3,60.4,54.0,40.1,55.1,49.9,65.7,50.6,70.6,56.8,47.3,83.0,87.1,69.0,20.5,70.5,56.4,94.8,60.8,34.5,20.4,38.8,44.5,64.2,31.5,20.8,56.5,47.6,41.5,39.0,79.9,37.9,87.2,92.4,98.6,77.9,41.8,96.1,92.0,87.9,83.2,5.6,2.1,10.4,6.0,11.3,3.4,2.7,14.4,13.6,16.9,13.8


In [942]:
passing_60Day_new.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'GAME_DATE', 'PASSES_MADE',
       'PASSES_RECEIVED', 'AST', 'FT_AST', 'SECONDARY_AST', 'POTENTIAL_AST',
       'AST_POINTS_CREATED', 'AST_ADJ', 'AST_TO_PASS_PCT',
       'AST_TO_PASS_PCT_ADJ', 'PASSES_MADE_60G_Sum', 'PASSES_RECEIVED_60G_Sum',
       'AST_60G_Sum', 'POTENTIAL_AST_60G_Sum', 'SECONDARY_AST_60G_Sum',
       'PASSES_MIN_60G_Sum', 'MIN_60G_Sum', 'AST_PASS_60G_Sum', 'AST_PER_MIN',
       'AST_TO_POTENTIAL', 'SECONDARY_AST_RATE', 'POINTS_PER_AST', 'id'],
      dtype='object')

In [943]:
drives_add = drives_60Day_new[['PLAYER_ID', 'DRIVES/MIN_60Day', 'DRIVES_AST_PASS_RATE',
       'DRIVES_PASS_RATE', 'SHOTS_DRIVE_RATE','PTS_PER_DRIVE']]
CS_add = CS_60Day_new[['PLAYER_ID','CS_FG2_PCT_60Day', 'CS_FG3_PCT_60Day', 'CS_FGA_rate_60Day',
       'CS_FG3A_rate_60Day']]
PU_add = PU_60Day_new[['PLAYER_ID','PU_FGA_PCT_60Day','PU_FG2_PCT_60Day', 'PU_FG3_PCT_60Day',
       'PU_FGA_rate_60Day', 'PU_FG3A_rate_60Day']]
passing_add = passing_60Day_new[['PLAYER_ID', 'PLAYER_NAME',  'PASSES_MADE_60G_Sum', 'PASSES_RECEIVED_60G_Sum',
       'AST_60G_Sum', 'POTENTIAL_AST_60G_Sum', 'SECONDARY_AST_60G_Sum',
       'PASSES_MIN_60G_Sum', 'AST_PASS_60G_Sum', 'AST_PER_MIN',
       'AST_TO_POTENTIAL', 'SECONDARY_AST_RATE', 'POINTS_PER_AST']]

rebounding_add = rebounding_60Day_new[['PLAYER_ID','OREB_PER_CHANCE', 'CONTEST_OREB_PER_CHANCE', 'DREB_PER_CHANCE',
       'CONTEST_DREB_PER_CHANCE','OREB_CHANCES_60G_Sum','DREB_CHANCES_60G_Sum']]
speeddistance_add = speeddistance_60Day_new[['PLAYER_ID','AVG_SPEED_OFF_60G_Sum', 'AVG_SPEED_DEF_60G_Sum', 'OFFmiles_PER_MIN',
       'DEFmiles_PER_MIN']]

In [944]:
rebounding_add

,PLAYER_ID,OREB_PER_CHANCE,CONTEST_OREB_PER_CHANCE,DREB_PER_CHANCE,CONTEST_DREB_PER_CHANCE,OREB_CHANCES_60G_Sum,DREB_CHANCES_60G_Sum
124,2544,0.333333,0.000000,0.647059,0.235294,165.0,595.0
238,101108,NaN,NaN,1.000000,1.000000,47.0,363.0
322,200768,NaN,NaN,1.000000,0.000000,85.0,290.0
347,200782,0.666667,0.333333,0.600000,0.200000,60.0,99.0
474,201142,0.000000,0.000000,0.875000,0.375000,79.0,476.0
...,...,...,...,...,...,...,...
41214,1642461,NaN,NaN,1.000000,0.000000,2.0,5.0
41218,1642484,1.000000,0.000000,1.000000,0.000000,2.0,5.0
41221,1642502,NaN,NaN,1.000000,0.000000,2.0,2.0
41228,1642505,0.333333,0.333333,1.000000,1.000000,13.0,14.0


In [945]:

dribbles_60_piv = dribbles_60Day_new[['PLAYER_ID','DRIBBLES','FGA_60G_Mavg','FG2M_60G_Mavg', 'FG3M_60G_Mavg']]
touchtime_60_piv = touchtime_60Day_new[['PLAYER_ID',"TOUCH_TIME",'FGA_60G_Mavg','FG2M_60G_Mavg', 'FG3M_60G_Mavg']]

In [946]:
#dribbles_60Day_new

In [947]:
dribbles_pivot = dribbles_60Day_new.pivot(index=['PLAYER_ID'], 
                    columns='DRIBBLES', 
                    values=['FGA_60G_Mavg','FG2M_60G_Mavg', 'FG3M_60G_Mavg'])

dribbles_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in dribbles_pivot.columns]
dribbles_pivot.reset_index(inplace=True)

In [948]:
dribbles_pivot.fillna(0,inplace=True)

In [949]:
touchtime_pivot = touchtime_60Day_new.pivot(index=['PLAYER_ID'], 
                    columns='TOUCH_TIME', 
                    values=['FGA_60G_Mavg','FG2M_60G_Mavg', 'FG3M_60G_Mavg'])

touchtime_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in touchtime_pivot.columns]
touchtime_pivot.reset_index(inplace=True)

In [950]:
touchtime_pivot.fillna(0, inplace=True)

In [951]:
# List of DataFrames to merge
dfs_to_merge = [
    speeddistance_add, rebounding_add, passing_add,
    PU_add, CS_add, drives_add, dribbles_pivot, touchtime_pivot
]

# Loop through each DataFrame in the list and merge, dropping 'PLAYER_NAME' and 'GAME_DATE' columns
for df in dfs_to_merge:
    merged_df_1 = merged_df_1.merge(df, how='left')

# The final merged DataFrame
df_playtypes = merged_df_1

In [952]:
df_playtypes_cat = df_playtypes.loc[df_playtypes['PLAYER_NAME'].notna()]

In [953]:
df_playtypes_cat.fillna(0,inplace=True)

/var/folders/sj/rsx4zyld6flczf6q6r20b1c80000gn/T/ipykernel_25156/2051849817.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_playtypes_cat.fillna(0,inplace=True)


In [954]:
df_playtypes_cat.reset_index(drop=True,inplace=True)

In [955]:
#df_playtypes_cat.loc[df_playtypes_cat['PLAYER_NAME'].str.contains('Karl-')]

df_playtypes_cat.drop_duplicates(subset='PLAYER_ID',inplace=True)

/var/folders/sj/rsx4zyld6flczf6q6r20b1c80000gn/T/ipykernel_25156/1593420775.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_playtypes_cat.drop_duplicates(subset='PLAYER_ID',inplace=True)


In [956]:
threshold_20 = df_playtypes_cat['PTS_60G_Mavg'].max()*.20

In [957]:
threshold_20 = df_playtypes_cat['PTS_60G_Mavg'].max()*.25
df_playtypes_cat_pts = pd.DataFrame(df_playtypes_cat.loc[df_playtypes_cat['PTS_60G_Mavg']>threshold_20]).reset_index(drop=True)

In [958]:
shotdetail_60Day_new.columns

Index(['PLAYER_ID', 'GAME_DATE', 'FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGA_BC_60G_Mavg', 'FGA_Mid_60G_Mavg',
       'FGA_Paint_60G_Mavg', 'FGA_RA_60G_Mavg', 'FGM_3_AB_60G_Mavg',
       'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg', 'FGM_BC_60G_Mavg',
       'FGM_Mid_60G_Mavg', 'FGM_Paint_60G_Mavg', 'FGM_RA_60G_Mavg', 'id'],
      dtype='object')

In [959]:
shotdetail_60Day_ptsadd = shotdetail_60Day_new[['PLAYER_ID', 'FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGA_Mid_60G_Mavg',
       'FGA_Paint_60G_Mavg', 'FGA_RA_60G_Mavg', ]].fillna(0)

In [960]:
df_playtypes_cat_pts = df_playtypes_cat_pts.merge(shotdetail_60Day_ptsadd, how='left')

In [961]:
df_playtypes_scoring = pd.DataFrame(df_playtypes_cat_pts[[ 'MIN_60G_Mavg', 'PTS_60G_Mavg', 'FGA_60G_Mavg', 'FG3A_60G_Mavg', 'MIN_60G_Sum',
                  'HEIGHT_INCHES', 'WEIGHT', 'OVERALL_SCORE_Cut',
       'OVERALL_SCORE_Handoff', 'OVERALL_SCORE_Isolation',
       'OVERALL_SCORE_Misc', 'OVERALL_SCORE_OffRebound',
       'OVERALL_SCORE_OffScreen', 'OVERALL_SCORE_PRBallHandler',
       'OVERALL_SCORE_PRRollMan', 'OVERALL_SCORE_Postup',
       'OVERALL_SCORE_Spotup', 'OVERALL_SCORE_Transition',
       'OVERALL_SCORE_PERCENTILE_Cut', 'OVERALL_SCORE_PERCENTILE_Handoff',
       'OVERALL_SCORE_PERCENTILE_Isolation', 'OVERALL_SCORE_PERCENTILE_Misc',
       'OVERALL_SCORE_PERCENTILE_OffRebound',
       'OVERALL_SCORE_PERCENTILE_OffScreen',
       'OVERALL_SCORE_PERCENTILE_PRBallHandler',
       'OVERALL_SCORE_PERCENTILE_PRRollMan', 'OVERALL_SCORE_PERCENTILE_Postup',
       'OVERALL_SCORE_PERCENTILE_Spotup',
       'OVERALL_SCORE_PERCENTILE_Transition', 'AVG_SPEED_OFF_60G_Sum', 'OFFmiles_PER_MIN',
       'PU_FG2_PCT_60Day', 'PU_FG3_PCT_60Day', 'PU_FGA_rate_60Day',
       'PU_FG3A_rate_60Day', 'CS_FG2_PCT_60Day', 'CS_FG3_PCT_60Day',
       'CS_FGA_rate_60Day', 'CS_FG3A_rate_60Day', 'DRIVES/MIN_60Day','SHOTS_DRIVE_RATE',
       'FG2M_60G_Mavg_Touch 2-6 Seconds', 'FG2M_60G_Mavg_Touch 6+ Seconds',
       'FG2M_60G_Mavg_Touch < 2 Seconds', 'FG3M_60G_Mavg_Touch 2-6 Seconds',
       'FG3M_60G_Mavg_Touch 6+ Seconds', 'FG3M_60G_Mavg_Touch < 2 Seconds','FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGA_Mid_60G_Mavg',
       'FGA_Paint_60G_Mavg', 'FGA_RA_60G_Mavg', ]])

In [962]:
df_scorer_cluster = pd.DataFrame(df_playtypes_cat_pts[[ 'PLAYER_ID','PLAYER_NAME','PTS_60G_Mavg','OVERALL_SCORE_Cut',
       'OVERALL_SCORE_Handoff', 'OVERALL_SCORE_Isolation',
       'OVERALL_SCORE_Misc', 'OVERALL_SCORE_OffRebound',
       'OVERALL_SCORE_OffScreen', 'OVERALL_SCORE_PRBallHandler',
       'OVERALL_SCORE_PRRollMan', 'OVERALL_SCORE_Postup',
       'OVERALL_SCORE_Spotup', 'OVERALL_SCORE_Transition',
       'OVERALL_SCORE_PERCENTILE_Cut', 'OVERALL_SCORE_PERCENTILE_Handoff',
       'OVERALL_SCORE_PERCENTILE_Isolation', 'OVERALL_SCORE_PERCENTILE_Misc',
       'OVERALL_SCORE_PERCENTILE_OffRebound',
       'OVERALL_SCORE_PERCENTILE_OffScreen',
       'OVERALL_SCORE_PERCENTILE_PRBallHandler',
       'OVERALL_SCORE_PERCENTILE_PRRollMan', 'OVERALL_SCORE_PERCENTILE_Postup',
       'OVERALL_SCORE_PERCENTILE_Spotup',
       'OVERALL_SCORE_PERCENTILE_Transition']])

In [963]:
df_playtypes_cat['AST_60G_Mavg'].max()*.25

2.875

In [964]:
threshold_40 = df_playtypes_cat['AST_60G_Mavg'].max()*.25
df_playtypes_cat_ast = pd.DataFrame(df_playtypes_cat.loc[df_playtypes_cat['AST_60G_Mavg']>threshold_40]).reset_index(drop=True)

In [965]:
passing_qualified = pd.DataFrame(df_playtypes_cat_ast[['SEASON_YEAR','PLAYER_NAME', 'PLAYER_ID', 'TEAM_ID', 'GAME_ID', 'GAME_DATE',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'PTS_60G_Mavg', 'REB_60G_Mavg',
       'AST_60G_Mavg','PASSES_MADE_60G_Sum', 'PASSES_RECEIVED_60G_Sum',
       'AST_60G_Sum', 'POTENTIAL_AST_60G_Sum', 'SECONDARY_AST_60G_Sum',
       'PASSES_MIN_60G_Sum', 'AST_PASS_60G_Sum', 'AST_PER_MIN',
       'AST_TO_POTENTIAL', 'SECONDARY_AST_RATE', 'POINTS_PER_AST','POSS_PCT_Cut', 'POSS_PCT_Handoff', 'POSS_PCT_Isolation',
       'POSS_PCT_Misc', 'POSS_PCT_OffRebound', 'POSS_PCT_OffScreen',
       'POSS_PCT_PRBallHandler', 'POSS_PCT_PRRollMan', 'POSS_PCT_Postup',
       'POSS_PCT_Spotup', 'POSS_PCT_Transition', 'DRIVES/MIN_60Day','DRIVES_AST_PASS_RATE',
       'DRIVES_PASS_RATE']])

In [966]:
passing_qualified.loc[passing_qualified['PLAYER_ID']==202710]

,SEASON_YEAR,PLAYER_NAME,PLAYER_ID,TEAM_ID,GAME_ID,GAME_DATE,GAMES_IN_WINDOW_60G,MIN_60G_Mavg,PTS_60G_Mavg,REB_60G_Mavg,AST_60G_Mavg,PASSES_MADE_60G_Sum,PASSES_RECEIVED_60G_Sum,AST_60G_Sum,POTENTIAL_AST_60G_Sum,SECONDARY_AST_60G_Sum,PASSES_MIN_60G_Sum,AST_PASS_60G_Sum,AST_PER_MIN,AST_TO_POTENTIAL,SECONDARY_AST_RATE,POINTS_PER_AST,POSS_PCT_Cut,POSS_PCT_Handoff,POSS_PCT_Isolation,POSS_PCT_Misc,POSS_PCT_OffRebound,POSS_PCT_OffScreen,POSS_PCT_PRBallHandler,POSS_PCT_PRRollMan,POSS_PCT_Postup,POSS_PCT_Spotup,POSS_PCT_Transition,DRIVES/MIN_60Day,DRIVES_AST_PASS_RATE,DRIVES_PASS_RATE
126,2024-25,Jimmy Butler,202710,1610612744,0022400957,2025-03-13,60.0,32.224194,17.783333,5.25,5.283333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.6,0.0,0.0,0.0,2.9,0.0,0.0,0.0,0.0,0.0,0.4,0.22,0.52


# Cluster Ast

In [967]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select features for clustering
cluster_features = [
    'PASSES_MADE_60G_Sum', 'PASSES_RECEIVED_60G_Sum','AST_60G_Mavg', 'POTENTIAL_AST_60G_Sum', 'SECONDARY_AST_60G_Sum',
       'PASSES_MIN_60G_Sum', 'AST_PASS_60G_Sum', 'AST_PER_MIN',
       'AST_TO_POTENTIAL', 'SECONDARY_AST_RATE', 'POINTS_PER_AST', 'POSS_PCT_Isolation','POSS_PCT_PRBallHandler',
       'POSS_PCT_PRRollMan', 'POSS_PCT_Postup', 'POSS_PCT_Transition', 'DRIVES/MIN_60Day','SHOTS_DRIVE_RATE',
]

# Clean features before standardization
X = df_playtypes_cat_ast[cluster_features].copy()
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

feature_weights = [3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 1, 2, 1, 1, 1] * (len(X_scaled) // 16 + 1)
feature_weights = feature_weights[:len(X_scaled)]

# Fit the KMeans model with weighted Euclidean distance
kmeans = KMeans(n_clusters=4, random_state=42)
passing_qualified['Passing_Cluster'] = kmeans.fit_predict(X_scaled, )

In [968]:
passing_qualified['PLAYER_NAME'].nunique()

145

In [969]:
passing_qualified['as_of'] = pd.to_datetime(today)
passing_qualified['id'] = passing_qualified['as_of'].astype(str)+"_"+passing_qualified['PLAYER_ID'].astype(str)

In [970]:
#passing_qualified.loc[passing_qualified['Passing_Cluster']==3]

In [971]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_passing_clusters"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    passing_qualified.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = passing_qualified[~passing_qualified['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'player_passing_clusters' already exists. Checking for new records...
Inserted 145 new records into 'player_passing_clusters'.


# Cluster Rebs

In [972]:
#df_playtypes_cat_rebs['REB_60G_Mavg'].max()*.25

In [973]:

shotdetail_60Day_pts_add = shotdetail_60Day_new[['PLAYER_ID', 'FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGA_BC_60G_Mavg', 'FGA_Mid_60G_Mavg',
       'FGA_Paint_60G_Mavg', 'FGA_RA_60G_Mavg', 'FGM_3_AB_60G_Mavg',
       'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg', 'FGM_BC_60G_Mavg',
       'FGM_Mid_60G_Mavg', 'FGM_Paint_60G_Mavg', 'FGM_RA_60G_Mavg']]

df_playtypes_cat_rebs =  df_playtypes_cat.merge(shotdetail_60Day_pts_add)

threshold_reb = df_playtypes_cat_rebs['REB_60G_Mavg'].max()*.25
#threshold_reb
df_playtypes_cat_reb = pd.DataFrame(df_playtypes_cat_rebs.loc[df_playtypes_cat_rebs['REB_60G_Mavg']>threshold_reb]).reset_index(drop=True)

In [974]:
df_playtypes_cat_reb['OREB_CHANCES_GAME'] = df_playtypes_cat_reb['GAMES_IN_WINDOW_60G'] /df_playtypes_cat_reb['OREB_CHANCES_60G_Sum']
df_playtypes_cat_reb['DREB_CHANCES_GAME'] = df_playtypes_cat_reb['GAMES_IN_WINDOW_60G'] /df_playtypes_cat_reb['DREB_CHANCES_60G_Sum']

In [975]:
rebounding_qualified = pd.DataFrame(df_playtypes_cat_reb[['SEASON_YEAR','PLAYER_NAME', 'PLAYER_ID', 'TEAM_ID', 'GAME_ID', 'GAME_DATE',
       'GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg', 'PTS_60G_Mavg', 'REB_60G_Mavg',
       'AST_60G_Mavg',  'HEIGHT_INCHES','WEIGHT','AVG_SPEED_DEF_60G_Sum','DEFmiles_PER_MIN','OREB_PER_CHANCE','CONTEST_OREB_PER_CHANCE','DREB_PER_CHANCE',
    'CONTEST_DREB_PER_CHANCE','OREB_CHANCES_GAME','DREB_CHANCES_GAME','FGA_RA_60G_Mavg','FGA_Paint_60G_Mavg',
       'VOLUME_SCORE_Cut',
       'VOLUME_SCORE_Handoff', 'VOLUME_SCORE_Isolation',
       'VOLUME_SCORE_Misc', 'VOLUME_SCORE_OffRebound',
       'VOLUME_SCORE_OffScreen', 'VOLUME_SCORE_PRBallHandler',
       'VOLUME_SCORE_PRRollMan', 'VOLUME_SCORE_Postup',
       'VOLUME_SCORE_Spotup', 'VOLUME_SCORE_Transition',]])

In [976]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select features for clustering
cluster_features = [
    'REB_60G_Mavg','MIN_60G_Mavg',  'HEIGHT_INCHES','WEIGHT','AVG_SPEED_DEF_60G_Sum','DEFmiles_PER_MIN','OREB_PER_CHANCE','CONTEST_OREB_PER_CHANCE','DREB_PER_CHANCE',
    'CONTEST_DREB_PER_CHANCE','OREB_CHANCES_GAME','DREB_CHANCES_GAME',
       'VOLUME_SCORE_Cut',
       'VOLUME_SCORE_Handoff', 'VOLUME_SCORE_Isolation',
       'VOLUME_SCORE_Misc', 'VOLUME_SCORE_OffRebound',
       'VOLUME_SCORE_OffScreen', 'VOLUME_SCORE_PRBallHandler',
       'VOLUME_SCORE_PRRollMan', 'VOLUME_SCORE_Postup',
       'VOLUME_SCORE_Spotup', 'VOLUME_SCORE_Transition', 
]

# Clean features before standardization
X = df_playtypes_cat_reb[cluster_features].copy()
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.select_dtypes(include=['number']).mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

feature_weights = [6, 2, 3, 3, 1, 1, 2, 2, 2, 2,2, 2, 1,1,1,1,1,1,1,1,1,1,1,  ] * (len(X_scaled) // 23 + 1)
feature_weights = feature_weights[:len(X_scaled)]

# Fit the KMeans model with weighted Euclidean distance
kmeans = KMeans(n_clusters=5, random_state=42)
rebounding_qualified['Rebounding_Cluster'] = kmeans.fit_predict(X_scaled, sample_weight=feature_weights)

In [977]:
rebounding_qualified.loc[rebounding_qualified['Rebounding_Cluster']==1].sort_values(by='REB_60G_Mavg', ascending=False)

,SEASON_YEAR,PLAYER_NAME,PLAYER_ID,TEAM_ID,GAME_ID,GAME_DATE,GAMES_IN_WINDOW_60G,MIN_60G_Mavg,PTS_60G_Mavg,REB_60G_Mavg,AST_60G_Mavg,HEIGHT_INCHES,WEIGHT,AVG_SPEED_DEF_60G_Sum,DEFmiles_PER_MIN,OREB_PER_CHANCE,CONTEST_OREB_PER_CHANCE,DREB_PER_CHANCE,CONTEST_DREB_PER_CHANCE,OREB_CHANCES_GAME,DREB_CHANCES_GAME,FGA_RA_60G_Mavg,FGA_Paint_60G_Mavg,VOLUME_SCORE_Cut,VOLUME_SCORE_Handoff,VOLUME_SCORE_Isolation,VOLUME_SCORE_Misc,VOLUME_SCORE_OffRebound,VOLUME_SCORE_OffScreen,VOLUME_SCORE_PRBallHandler,VOLUME_SCORE_PRRollMan,VOLUME_SCORE_Postup,VOLUME_SCORE_Spotup,VOLUME_SCORE_Transition,Rebounding_Cluster
173,2024-25,Domantas Sabonis,1627734,1610612758,0022400862,2025-03-01,60.0,35.174667,19.416667,14.133333,6.200000,82.0,240,233.20,0.036101,0.000000,0.000000,0.500000,0.166667,0.111317,0.061287,6.900000,2.900000,61.3,0.0,20.6,42.9,69.6,0.0,15.6,67.1,35.3,20.0,29.2,1
188,2024-25,Nikola Jokić,203999,1610612743,0022400952,2025-03-12,60.0,36.422250,28.883333,12.866667,10.283333,83.0,284,203.31,0.031222,0.500000,0.333333,0.454545,0.090909,0.173913,0.065288,6.200000,6.933333,64.6,12.8,27.5,78.0,58.5,38.7,22.3,50.0,83.4,28.5,30.1,1
186,2024-25,Karl-Anthony Towns,1626157,1610612752,0022400953,2025-03-12,60.0,34.487944,23.333333,12.616667,3.216667,84.0,248,227.54,0.032732,0.600000,0.600000,0.363636,0.000000,0.158730,0.069767,6.916667,3.383333,34.5,20.4,38.8,44.5,64.2,31.5,20.8,56.5,47.6,41.5,39.0,1
209,2024-25,Anthony Davis,203076,1610612742,0022400741,2025-02-08,60.0,34.376194,25.183333,12.533333,3.350000,82.0,253,215.77,0.031681,1.000000,0.750000,0.800000,0.266667,0.171429,0.070423,6.866667,5.183333,55.6,23.2,42.7,57.4,64.0,34.0,28.9,74.0,50.8,21.1,33.3,1
197,2024-25,Giannis Antetokounmpo,203507,1610612749,0022400955,2025-03-13,60.0,34.112306,30.066667,12.416667,6.066667,83.0,243,217.84,0.032518,0.333333,0.333333,0.882353,0.117647,0.185759,0.073171,12.166667,3.116667,54.0,17.3,62.5,45.1,51.4,0.0,31.6,51.4,65.3,24.6,72.9,1
162,2024-25,Ivica Zubac,1627826,1610612746,0022400948,2025-03-12,60.0,32.524667,15.816667,12.366667,2.400000,84.0,240,227.45,0.033117,0.750000,0.625000,0.666667,0.250000,0.135440,0.072464,6.150000,4.866667,64.2,0.0,0.0,24.1,69.9,0.0,0.0,47.5,61.2,25.1,20.1,1
36,2024-25,Walker Kessler,1631117,1610612762,0022400950,2025-03-12,60.0,29.011056,10.566667,11.316667,1.583333,84.0,245,250.57,0.035880,0.500000,0.500000,0.588235,0.294118,0.112570,0.083449,5.233333,1.083333,58.5,0.0,0.0,33.7,72.7,0.0,0.0,38.0,14.2,1.5,20.2,1
25,2024-25,Victor Wembanyama,1641705,1610612759,0022400769,2025-02-12,60.0,33.068139,24.016667,11.083333,4.083333,87.0,235,231.26,0.034893,0.333333,0.333333,0.750000,0.125000,0.255319,0.069930,5.066667,2.333333,38.1,23.3,34.4,47.1,33.5,54.3,22.8,61.0,40.9,38.9,41.7,1
122,2024-25,Deandre Ayton,1629028,1610612757,0022400763,2025-02-10,60.0,31.177111,17.066667,10.650000,1.700000,84.0,252,217.20,0.031941,0.454545,0.272727,0.818182,0.363636,0.166205,0.088626,4.833333,4.533333,62.3,0.0,14.4,29.4,54.3,14.2,0.0,65.6,26.2,12.6,20.1,1
199,2024-25,Rudy Gobert,203497,1610612750,0022400952,2025-03-12,60.0,33.118833,11.500000,10.583333,1.833333,85.0,258,239.69,0.035177,0.600000,0.400000,0.875000,0.250000,0.128205,0.084626,5.483333,1.133333,67.1,0.0,0.0,30.6,62.9,0.0,0.0,44.7,16.7,0.0,17.5,1


In [978]:
#rebounding_qualified.loc[rebounding_qualified['PLAYER_ID']==202710]

In [979]:
rebounding_qualified['as_of'] = pd.to_datetime(today)
rebounding_qualified['id'] = rebounding_qualified['as_of'].astype(str)+"_"+rebounding_qualified['PLAYER_ID'].astype(str)

In [980]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_rebounding_clusters"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    rebounding_qualified.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = rebounding_qualified[~rebounding_qualified['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'player_rebounding_clusters' already exists. Checking for new records...
Inserted 231 new records into 'player_rebounding_clusters'.


# Cluster PTS

In [981]:
df_playtypes_scoring.fillna(0, inplace=True)

In [982]:
from sklearn.preprocessing import StandardScaler

# Assume df contains your original data (without non-numeric columns like player names or IDs)
scaler = StandardScaler()
df_scaled_player = scaler.fit_transform(df_playtypes_scoring)

In [983]:
#df_playtypes_scoring

In [984]:
#df_scaled_player

In [985]:
from sklearn.cluster import KMeans

# Define the number of clusters (e.g., 5 categories)
kmeans = KMeans(n_clusters=7, random_state=23)
df_scorer_cluster['Cluster'] = kmeans.fit_predict(df_scaled_player)

In [986]:
df_scorer_cluster.loc[df_scorer_cluster['PLAYER_ID']==202710]

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,Cluster
219,202710,Jimmy Butler,17.783333,0.0,30.6,0.0,0.0,0.0,62.7,0.0,0.0,0.0,0.0,0.0,0.0,21.1,0.0,0.0,0.0,95.7,0.0,0.0,0.0,0.0,0.0,6


In [987]:
df_scorer_cluster.loc[df_scorer_cluster['Cluster']==0]

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,Cluster
18,1641718,Keyonte George,16.616667,45.1,60.7,45.9,44.9,27.3,41.4,51.7,32.1,0.0,51.0,40.6,65.1,92.4,73.2,67.8,14.3,53.0,81.8,25.2,0.0,77.1,50.7,0
33,1631114,Jalen Williams,21.433333,49.4,49.2,47.2,42.2,36.2,44.3,62.7,31.2,0.0,57.7,60.5,78.7,78.9,77.1,58.8,36.4,64.9,94.1,22.6,0.0,91.3,94.4,0
40,1631101,Shaedon Sharpe,17.316667,33.9,44.2,51.0,56.3,46.6,38.3,53.3,0.0,0.0,44.3,58.2,27.2,63.5,85.1,88.7,70.8,39.4,84.4,0.0,0.0,61.0,92.6,0
53,1630595,Cade Cunningham,25.766667,33.8,75.0,55.1,60.1,27.2,42.1,78.2,0.0,42.1,41.4,53.5,26.4,99.4,89.7,92.9,14.0,54.3,99.4,0.0,56.4,50.6,86.6,0
63,1630560,Cam Thomas,24.250000,20.4,100.0,57.2,47.3,50.7,43.4,67.9,41.1,0.0,61.7,40.8,3.1,100.0,91.8,73.1,80.6,60.4,96.4,55.4,0.0,97.0,52.0,0
64,1630559,Austin Reaves,19.000000,36.9,52.5,44.9,60.0,31.5,48.4,58.6,34.3,0.0,58.7,50.7,37.1,84.4,69.3,92.6,20.2,80.3,91.1,34.3,0.0,92.7,79.7,0
80,1630224,Jalen Green,20.816667,32.6,54.2,44.9,51.6,26.7,24.9,71.1,0.0,0.0,46.3,58.2,22.3,86.0,69.3,80.7,12.5,7.8,97.6,0.0,0.0,65.9,92.6,0
81,1630217,Desmond Bane,18.583333,35.4,38.2,46.7,54.7,39.3,47.6,48.3,0.0,0.0,49.0,65.4,32.0,41.3,75.5,86.2,48.5,76.2,75.5,0.0,0.0,72.3,97.2,0
85,1630193,Immanuel Quickley,17.916667,31.6,57.7,37.2,56.0,0.0,46.2,54.6,35.5,0.0,42.0,53.2,20.5,90.0,35.3,88.2,0.0,73.4,86.1,37.7,0.0,51.9,86.1,0
88,1630178,Tyrese Maxey,26.500000,53.7,52.0,62.3,71.9,34.8,57.5,68.9,48.3,0.0,53.0,65.7,86.2,83.5,94.5,98.8,30.0,90.5,96.7,73.1,0.0,82.2,97.4,0


In [988]:
df_scorer_cluster.loc[df_scorer_cluster['Cluster']==1]

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,Cluster
0,1642377,Jaylen Wells,11.250000,43.2,0.0,37.8,52.4,38.2,39.1,47.1,0.0,0.0,59.1,45.9,60.1,0.0,39.2,81.8,42.8,44.6,72.1,0.0,0.0,93.9,66.6,1
1,1642366,Quinten Post,8.444444,37.1,0.0,0.0,16.8,35.1,0.0,0.0,39.2,0.0,44.7,43.2,38.7,0.0,0.0,1.5,31.5,0.0,0.0,48.7,0.0,61.8,60.1,1
4,1642273,Kyshawn George,8.474576,23.0,40.0,33.6,47.9,21.7,20.6,30.1,0.0,0.0,43.2,36.1,5.4,49.0,22.7,74.8,5.2,3.5,18.1,0.0,0.0,56.5,33.4,1
8,1642261,Dalton Knecht,9.200000,46.0,39.1,25.1,25.6,0.0,36.0,33.7,33.4,0.0,48.4,40.1,69.4,45.2,8.5,10.1,0.0,34.8,29.4,29.3,0.0,70.3,48.4,1
10,1642258,Zaccharie Risacher,11.586207,43.9,41.3,0.0,32.2,39.8,27.1,34.7,32.4,0.0,47.2,51.7,61.9,53.2,0.0,22.9,49.7,10.4,32.5,26.1,0.0,67.7,82.4,1
15,1641729,Brice Sensabaugh,10.116667,37.4,41.2,36.7,31.6,39.1,36.9,40.6,0.0,0.0,42.6,37.7,40.0,52.7,33.9,20.9,47.8,36.8,51.9,0.0,0.0,54.4,39.5,1
17,1641722,Jordan Hawkins,9.116667,0.0,44.7,19.2,39.6,44.6,44.7,43.4,36.4,0.0,43.5,43.4,0.0,66.0,2.8,49.9,65.0,65.4,62.3,40.0,0.0,58.1,60.8,1
19,1641715,Cam Whitmore,10.033333,31.4,41.0,23.6,37.5,44.1,30.6,34.4,0.0,0.0,40.7,48.5,20.2,51.7,7.1,41.4,63.1,17.5,30.9,0.0,0.0,47.4,73.7,1
20,1641713,GG Jackson,13.216667,50.2,36.9,38.9,43.4,34.1,39.4,45.4,50.7,0.0,40.8,51.2,81.2,36.3,43.4,64.0,27.8,46.1,67.2,80.2,0.0,48.2,80.9,1
21,1641711,Gradey Dick,14.366667,30.9,45.4,0.0,34.9,34.0,63.5,32.2,48.1,0.0,59.4,47.0,19.7,68.6,0.0,29.9,27.5,96.5,25.2,72.8,0.0,94.9,69.5,1


In [989]:
df_scorer_cluster.loc[df_scorer_cluster['Cluster']==2]

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,Cluster
7,1642264,Stephon Castle,13.466667,36.4,38.6,40.6,44.3,37.4,39.6,44.3,33.2,0.0,44.9,53.5,34.9,43.5,50.2,66.7,40.4,46.8,64.4,28.7,0.0,62.1,86.6,2
9,1642259,Alex Sarr,11.880000,43.7,0.0,26.4,46.8,45.4,0.0,0.0,56.7,0.0,45.8,34.6,61.5,0.0,11.7,71.9,66.9,0.0,0.0,89.8,0.0,64.5,29.8,2
13,1641739,Toumani Camara,10.316667,27.1,38.3,39.5,35.2,47.6,0.0,31.1,43.7,0.0,55.1,42.1,10.6,41.6,46.5,31.9,72.5,0.0,20.9,63.0,0.0,86.8,57.4,2
14,1641731,Bilal Coulibaly,12.366667,38.5,35.2,48.2,31.3,44.0,33.5,42.9,35.9,0.0,40.9,54.2,45.1,32.7,79.4,19.0,62.7,25.8,60.2,38.7,0.0,49.0,87.8,2
23,1641709,Ausar Thompson,9.933333,52.7,41.6,35.8,46.0,43.6,0.0,45.0,34.5,0.0,35.4,50.2,85.3,54.1,30.7,69.0,60.9,0.0,66.3,35.1,0.0,31.3,78.4,2
24,1641708,Amen Thompson,14.000000,61.1,39.7,43.8,37.4,56.0,0.0,44.5,37.4,0.0,28.1,59.1,94.2,48.1,65.6,40.7,90.1,0.0,65.1,42.0,0.0,14.9,93.7,2
30,1631128,Christian Braun,15.566667,58.7,39.6,50.8,40.1,42.1,0.0,43.6,49.7,0.0,44.2,71.1,92.7,47.5,84.8,51.5,55.1,0.0,62.8,77.4,0.0,60.6,99.6,2
34,1631110,Jeremy Sochan,11.616667,56.5,29.6,35.0,53.5,46.6,0.0,48.9,56.9,39.9,34.7,45.7,89.8,18.3,28.0,84.6,70.8,0.0,76.9,90.2,50.9,29.1,66.2,2
38,1631106,Tari Eason,11.216667,41.1,37.6,38.8,42.5,52.2,0.0,36.3,36.6,0.0,43.7,55.9,53.4,38.7,42.6,60.1,84.6,0.0,37.8,40.3,0.0,58.9,90.5,2
43,1631096,Chet Holmgren,15.616667,45.3,27.6,42.6,53.1,63.2,42.6,43.0,52.3,9.9,52.7,54.8,65.7,13.0,60.3,83.5,94.2,57.6,61.0,82.5,0.6,81.6,88.4,2


In [990]:
df_scorer_cluster.loc[df_scorer_cluster['Cluster']==3]

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,Cluster
2,1642276,Kel'el Ware,8.425532,55.1,0.0,0.0,40.5,42.2,0.0,0.0,59.5,18.1,32.1,32.3,88.5,0.0,0.0,53.2,55.6,0.0,0.0,91.1,3.1,22.1,22.9,3
3,1642274,Yves Missi,8.850000,50.0,0.0,0.0,39.3,62.0,0.0,0.0,62.2,32.0,26.0,47.4,80.8,0.0,0.0,48.4,93.7,0.0,0.0,93.4,22.7,11.0,70.7,3
12,1641744,Zach Edey,9.142857,51.7,0.0,0.0,42.7,63.7,0.0,0.0,52.6,50.0,28.2,35.8,84.0,0.0,0.0,61.4,94.8,0.0,0.0,83.0,81.0,15.3,32.3,3
16,1641726,Dereck Lively II,9.000000,67.7,0.0,0.0,30.4,51.4,0.0,0.0,56.2,47.8,0.0,55.3,96.9,0.0,0.0,17.7,82.0,0.0,0.0,88.9,76.1,0.0,89.2,3
32,1631117,Walker Kessler,10.566667,71.1,0.0,0.0,43.0,84.9,0.0,0.0,54.9,30.1,7.9,51.2,97.9,0.0,0.0,62.5,100.0,0.0,0.0,86.7,18.7,1.5,80.9,3
35,1631109,Mark Williams,14.588235,81.0,0.0,0.0,39.0,66.4,0.0,0.0,76.2,40.1,0.0,55.6,100.0,0.0,0.0,47.3,95.9,0.0,0.0,98.7,51.5,0.0,89.8,3
39,1631105,Jalen Duren,11.416667,58.8,0.0,48.9,32.1,68.1,0.0,67.1,60.4,43.4,46.3,55.9,93.0,0.0,80.7,22.4,96.7,0.0,96.1,92.5,61.3,65.9,90.5,3
82,1630208,Nick Richards,9.533333,44.8,0.0,0.0,34.9,63.6,0.0,0.0,61.4,17.9,0.0,31.2,64.4,0.0,0.0,29.9,94.5,0.0,0.0,92.8,2.5,0.0,19.8,3
94,1630168,Onyeka Okongwu,12.633333,63.9,0.0,0.0,43.4,61.8,0.0,0.0,64.5,43.7,30.4,48.8,96.1,0.0,0.0,64.0,93.3,0.0,0.0,94.1,63.2,18.6,74.4,3
105,1629655,Daniel Gafford,11.866667,69.0,0.0,0.0,51.4,69.2,0.0,0.0,54.9,48.2,42.9,57.7,97.2,0.0,0.0,80.1,97.0,0.0,0.0,86.7,77.3,55.4,92.1,3


In [991]:
df_scorer_cluster.loc[df_scorer_cluster['Cluster']==4]

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,Cluster
25,1641706,Brandon Miller,20.150000,0.0,47.6,47.2,60.8,57.0,79.7,50.7,40.2,0.0,53.9,46.1,0.0,73.7,77.1,94.2,91.6,99.6,80.3,53.1,0.0,84.1,67.1,4
41,1631099,Keegan Murray,12.083333,33.2,47.5,43.7,36.5,40.5,38.5,36.7,26.1,23.6,57.1,48.4,23.9,73.3,65.2,37.1,51.5,41.1,40.5,11.5,6.1,90.5,73.3,4
46,1631093,Jaden Ivey,16.500000,53.1,59.5,35.8,48.5,45.5,33.9,51.0,0.0,0.0,58.6,52.2,85.8,91.7,30.7,76.0,67.6,27.3,81.0,0.0,0.0,92.5,83.7,4
55,1630591,Jalen Suggs,14.883333,0.0,46.9,46.0,46.1,38.8,24.0,56.7,39.9,0.0,39.9,49.6,0.0,71.7,73.9,69.7,45.9,6.1,90.2,50.8,0.0,44.6,77.1,4
74,1630530,Trey Murphy III,20.200000,37.2,61.6,37.3,51.6,49.5,48.6,39.3,44.0,0.0,72.0,65.0,39.4,94.0,36.0,80.7,77.7,81.6,47.2,64.1,0.0,99.8,97.0,4
83,1630202,Payton Pritchard,13.883333,43.7,47.7,38.7,51.7,38.6,56.2,48.1,35.0,0.0,58.1,42.3,61.5,74.1,42.0,81.1,44.5,89.6,74.5,36.4,0.0,92.0,58.4,4
87,1630180,Saddiq Bey,13.833333,41.4,37.8,42.5,42.4,0.0,38.6,37.2,45.7,37.8,54.7,46.1,54.6,40.0,59.2,59.5,0.0,42.2,42.4,67.4,41.1,85.8,67.1,4
92,1630170,Devin Vassell,16.816667,50.4,57.6,45.3,36.4,0.0,48.3,45.9,0.0,0.0,58.9,45.3,81.6,89.5,70.9,36.6,0.0,79.2,68.8,0.0,0.0,93.3,65.4,4
96,1630166,Deni Avdija,14.966667,32.3,42.4,55.3,49.3,48.9,49.0,48.1,30.9,43.1,53.1,55.5,22.0,57.3,90.1,77.6,76.0,82.5,74.5,22.1,59.5,82.5,89.5,4
100,1629675,Naz Reid,14.933333,38.0,40.2,37.3,41.6,50.5,54.0,32.7,49.1,38.2,57.7,41.1,43.3,49.5,36.0,55.9,79.5,88.3,26.7,75.1,42.0,91.3,52.9,4


In [992]:
df_scorer_cluster.loc[df_scorer_cluster['Cluster']==5]

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,Cluster
26,1641705,Victor Wembanyama,24.016667,48.5,39.5,41.9,60.3,55.1,62.1,34.6,67.9,54.5,51.4,52.9,77.0,47.0,55.9,93.4,88.2,95.2,31.6,96.4,86.5,78.8,85.3,5
42,1631097,Bennedict Mathurin,16.283333,43.5,41.7,42.5,53.6,40.6,54.4,49.7,49.2,43.6,40.4,62.4,60.5,54.8,59.2,85.0,51.8,88.7,78.5,75.6,62.6,46.4,95.5,5
45,1631094,Paolo Banchero,23.433333,47.1,19.7,66.8,50.1,39.2,42.4,60.8,53.6,43.0,40.8,64.1,73.6,2.2,96.1,78.4,48.2,56.5,92.6,84.4,58.9,48.2,96.1,5
59,1630578,Alperen Sengun,19.166667,44.6,0.0,52.9,61.7,62.9,0.0,52.8,68.4,70.5,33.2,37.6,63.1,0.0,87.6,95.2,93.9,0.0,83.8,96.7,98.2,24.9,39.0,5
62,1630567,Scottie Barnes,19.383333,42.0,39.4,45.7,57.7,43.8,30.3,53.4,46.6,51.5,42.7,54.0,56.2,46.7,72.5,91.2,61.8,16.7,85.0,70.0,83.4,54.7,87.4,5
73,1630532,Franz Wagner,23.200000,58.1,53.4,48.2,58.5,33.0,46.1,62.6,45.0,49.6,47.1,61.1,91.6,85.4,79.4,91.6,25.1,72.3,93.8,66.2,80.1,67.5,94.6,5
116,1629628,RJ Barrett,22.016667,47.6,59.2,45.7,46.8,46.0,45.9,55.9,24.8,42.5,56.8,69.9,75.3,91.4,72.5,71.9,69.6,70.3,88.3,8.9,57.1,90.0,98.9,5
117,1629627,Zion Williamson,24.250000,51.8,59.9,75.4,48.3,65.9,42.7,58.8,51.3,56.3,52.6,64.0,84.3,92.1,98.6,75.4,95.6,58.2,91.4,80.8,89.6,81.1,95.9,5
135,1628991,Jaren Jackson Jr.,22.766667,53.8,32.6,56.5,55.9,52.3,40.5,30.0,41.6,63.4,75.9,59.1,86.6,25.6,91.1,87.7,85.0,50.4,17.8,58.4,95.7,100.0,93.7,5
151,1628389,Bam Adebayo,17.983333,64.8,0.0,49.4,55.2,53.6,38.5,36.1,66.2,46.0,41.1,34.4,96.5,0.0,82.3,86.9,86.0,41.1,36.8,95.4,70.6,49.7,28.6,5


In [993]:
df_scorer_cluster.loc[df_scorer_cluster['Cluster']==6]

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,Cluster
5,1642272,Jared McCain,15.304348,0.0,46.6,51.2,0.0,0.0,34.0,49.7,0.0,0.0,58.4,50.5,0.0,70.6,85.5,0.0,0.0,27.9,78.5,0.0,0.0,92.3,78.9,6
6,1642267,Bub Carrington,9.316667,20.5,47.8,25.5,28.8,0.0,32.4,45.0,0.0,0.0,37.6,35.7,3.4,74.6,9.8,14.7,0.0,23.2,66.3,0.0,0.0,37.6,31.9,6
11,1641764,Brandin Podziemski,9.883333,29.1,37.8,39.4,33.6,32.8,48.6,41.3,24.1,0.0,44.5,39.3,15.4,40.0,46.1,27.1,24.1,81.6,53.7,7.2,0.0,61.5,45.8,6
22,1641710,Anthony Black,9.050000,28.0,43.3,49.1,48.7,10.8,10.4,44.5,0.0,0.0,36.9,42.7,13.0,60.2,81.2,76.7,1.1,0.4,65.1,0.0,0.0,35.5,58.9,6
29,1631170,Jaime Jaquez Jr.,8.566667,32.0,50.8,54.1,36.0,37.7,28.4,40.3,0.0,34.1,41.0,36.5,21.5,82.1,88.7,35.4,40.9,13.0,50.1,0.0,29.4,49.4,34.8,6
47,1630703,Scoot Henderson,13.100000,26.6,46.0,41.5,52.9,34.7,31.5,48.7,0.0,0.0,48.5,41.7,10.0,69.7,54.6,82.4,29.5,20.3,76.1,0.0,0.0,70.5,56.1,6
49,1630631,Jose Alvarado,9.066667,29.1,38.5,52.6,35.1,39.9,0.0,47.5,0.0,0.0,42.2,31.5,15.4,42.9,86.5,31.1,50.1,0.0,72.8,0.0,0.0,53.1,20.6,6
50,1630625,Dalano Banton,9.883333,37.7,40.0,43.2,59.2,38.6,30.9,37.3,0.0,0.0,40.0,36.9,41.7,49.0,62.8,91.9,44.5,19.0,42.7,0.0,0.0,45.2,36.2,6
56,1630590,Scotty Pippen Jr.,9.016667,22.0,42.1,37.5,41.2,19.7,0.0,29.6,29.2,0.0,53.9,46.8,5.0,56.5,37.1,54.8,3.6,0.0,16.6,17.9,0.0,84.1,68.7,6
61,1630570,Trendon Watford,8.466667,21.0,48.8,48.9,36.3,43.5,0.0,48.7,29.3,42.6,32.3,39.0,3.9,76.5,80.7,36.0,60.3,0.0,76.1,18.5,57.7,22.8,45.4,6


In [994]:
today = pd.Timestamp.today()

In [995]:
df_scorer_cluster['as_of'] = pd.to_datetime(today)

In [996]:

df_scorer_cluster['id'] = df_scorer_cluster['as_of'].astype(str)+"_"+df_scorer_cluster['PLAYER_ID'].astype(str)

In [997]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_scoring_clusters_new"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    df_scorer_cluster.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = df_scorer_cluster[~df_scorer_cluster['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'player_scoring_clusters_new' already exists. Checking for new records...
Inserted 239 new records into 'player_scoring_clusters_new'.


# Categorize Scorers

### Penetrators, Rim Runner, Natural, Paceman, Skilled Bigs, Offball

##### [Drives, PNR_BH, Paint Shots] , [PNRMan, Cut, Paint Shots] , [ISO, Pullup, Mid Range, 3 Pts, Drives], ['Transition',''], ['PNRMan, ElbowTouches','Postup','']

In [998]:
shotdetail_60Day_new.columns

Index(['PLAYER_ID', 'GAME_DATE', 'FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGA_BC_60G_Mavg', 'FGA_Mid_60G_Mavg',
       'FGA_Paint_60G_Mavg', 'FGA_RA_60G_Mavg', 'FGM_3_AB_60G_Mavg',
       'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg', 'FGM_BC_60G_Mavg',
       'FGM_Mid_60G_Mavg', 'FGM_Paint_60G_Mavg', 'FGM_RA_60G_Mavg', 'id'],
      dtype='object')

In [999]:
shotdetail_60Day_pts_add = shotdetail_60Day_new[['PLAYER_ID', 'FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGA_BC_60G_Mavg', 'FGA_Mid_60G_Mavg',
       'FGA_Paint_60G_Mavg', 'FGA_RA_60G_Mavg', 'FGM_3_AB_60G_Mavg',
       'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg', 'FGM_BC_60G_Mavg',
       'FGM_Mid_60G_Mavg', 'FGM_Paint_60G_Mavg', 'FGM_RA_60G_Mavg']]

In [1000]:
df_full_pts = df_playtypes_cat_pts.merge(shotdetail_60Day_pts_add, how='left')

In [1001]:
df_scorer_cat = pd.DataFrame(df_full_pts[['PLAYER_ID', 'PLAYER_NAME','MIN_60G_Mavg', 'PTS_60G_Mavg', 'FGA_60G_Mavg', 'FG3A_60G_Mavg', 'MIN_60G_Sum',
                  'HEIGHT_INCHES', 'WEIGHT', 'OVERALL_SCORE_Cut',
       'OVERALL_SCORE_Handoff', 'OVERALL_SCORE_Isolation',
       'OVERALL_SCORE_Misc', 'OVERALL_SCORE_OffRebound',
       'OVERALL_SCORE_OffScreen', 'OVERALL_SCORE_PRBallHandler',
       'OVERALL_SCORE_PRRollMan', 'OVERALL_SCORE_Postup',
       'OVERALL_SCORE_Spotup', 'OVERALL_SCORE_Transition',
       'OVERALL_SCORE_PERCENTILE_Cut', 'OVERALL_SCORE_PERCENTILE_Handoff',
       'OVERALL_SCORE_PERCENTILE_Isolation', 'OVERALL_SCORE_PERCENTILE_Misc',
       'OVERALL_SCORE_PERCENTILE_OffRebound',
       'OVERALL_SCORE_PERCENTILE_OffScreen',
       'OVERALL_SCORE_PERCENTILE_PRBallHandler',
       'OVERALL_SCORE_PERCENTILE_PRRollMan', 'OVERALL_SCORE_PERCENTILE_Postup',
       'OVERALL_SCORE_PERCENTILE_Spotup',
       'OVERALL_SCORE_PERCENTILE_Transition', 'SCORING_EFFICIENCY_Cut',
       'SCORING_EFFICIENCY_Handoff', 'SCORING_EFFICIENCY_Isolation',
       'SCORING_EFFICIENCY_Misc', 'SCORING_EFFICIENCY_OffRebound',
       'SCORING_EFFICIENCY_OffScreen', 'SCORING_EFFICIENCY_PRBallHandler',
       'SCORING_EFFICIENCY_PRRollMan', 'SCORING_EFFICIENCY_Postup',
       'SCORING_EFFICIENCY_Spotup', 'SCORING_EFFICIENCY_Transition',
       'SCORING_EFFICIENCY_PERCENTILE_Cut',
       'SCORING_EFFICIENCY_PERCENTILE_Handoff',
       'SCORING_EFFICIENCY_PERCENTILE_Isolation',
       'SCORING_EFFICIENCY_PERCENTILE_Misc',
       'SCORING_EFFICIENCY_PERCENTILE_OffRebound',
       'SCORING_EFFICIENCY_PERCENTILE_OffScreen',
       'SCORING_EFFICIENCY_PERCENTILE_PRBallHandler',
       'SCORING_EFFICIENCY_PERCENTILE_PRRollMan',
       'SCORING_EFFICIENCY_PERCENTILE_Postup',
       'SCORING_EFFICIENCY_PERCENTILE_Spotup',
       'SCORING_EFFICIENCY_PERCENTILE_Transition', 'VOLUME_SCORE_Cut',
       'VOLUME_SCORE_Handoff', 'VOLUME_SCORE_Isolation', 'VOLUME_SCORE_Misc',
       'VOLUME_SCORE_OffRebound', 'VOLUME_SCORE_OffScreen',
       'VOLUME_SCORE_PRBallHandler', 'VOLUME_SCORE_PRRollMan',
       'VOLUME_SCORE_Postup', 'VOLUME_SCORE_Spotup', 'VOLUME_SCORE_Transition',
       'VOLUME_SCORE_PERCENTILE_Cut', 'VOLUME_SCORE_PERCENTILE_Handoff',
       'VOLUME_SCORE_PERCENTILE_Isolation', 'VOLUME_SCORE_PERCENTILE_Misc',
       'VOLUME_SCORE_PERCENTILE_OffRebound',
       'VOLUME_SCORE_PERCENTILE_OffScreen',
       'VOLUME_SCORE_PERCENTILE_PRBallHandler',
       'VOLUME_SCORE_PERCENTILE_PRRollMan', 'VOLUME_SCORE_PERCENTILE_Postup',
       'VOLUME_SCORE_PERCENTILE_Spotup', 'VOLUME_SCORE_PERCENTILE_Transition', 'AVG_SPEED_OFF_60G_Sum', 'OFFmiles_PER_MIN',
       'PU_FG2_PCT_60Day', 'PU_FG3_PCT_60Day','PU_FGA_PCT_60Day', 'PU_FGA_rate_60Day',
       'PU_FG3A_rate_60Day', 'CS_FG2_PCT_60Day', 'CS_FG3_PCT_60Day',
       'CS_FGA_rate_60Day', 'CS_FG3A_rate_60Day','PTS_PER_DRIVE', 'DRIVES/MIN_60Day','SHOTS_DRIVE_RATE','FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGA_BC_60G_Mavg', 'FGA_Mid_60G_Mavg',
       'FGA_Paint_60G_Mavg', 'FGA_RA_60G_Mavg', 'FGM_3_AB_60G_Mavg',
       'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg', 'FGM_BC_60G_Mavg',
       'FGM_Mid_60G_Mavg', 'FGM_Paint_60G_Mavg', 'FGM_RA_60G_Mavg']])

In [1002]:
#df_scorer_cat.loc[((df_scorer_cat['SHOTS_DRIVE_RATE']>SHOT_DRIVE)&(df_scorer_cat['SHOTS_DRIVE_RATE']>SHOT_DRIVE))&((df_scorer_cat['PPP_PRBallHandler']>PNR_H_PPP)&(df_scorer_cat['POSS_PCT_PRBallHandler']>PNR_H_FREQ))]

In [1003]:
df_scorer_cat.fillna(0,inplace=True)

In [1004]:
df_scorer_cat['RA_PCT'] = df_scorer_cat['FGM_RA_60G_Mavg']/df_scorer_cat['FGA_RA_60G_Mavg']
df_scorer_cat['Paint_PCT'] = df_scorer_cat['FGM_Paint_60G_Mavg']/df_scorer_cat['FGA_Paint_60G_Mavg']
df_scorer_cat['RA_RATE'] = df_scorer_cat['FGA_RA_60G_Mavg']/df_scorer_cat['MIN_60G_Mavg']
df_scorer_cat['Paint_RATE'] = df_scorer_cat['FGA_Paint_60G_Mavg']/df_scorer_cat['MIN_60G_Mavg']

df_scorer_cat['MID_PCT'] = df_scorer_cat['FGM_Mid_60G_Mavg']/df_scorer_cat['FGA_Mid_60G_Mavg']
df_scorer_cat['FG3_PCT'] = (df_scorer_cat['FGM_3_RC_60G_Mavg']+df_scorer_cat['FGM_3_LC_60G_Mavg']+df_scorer_cat['FGM_3_AB_60G_Mavg'])/df_scorer_cat['FG3A_60G_Mavg']
df_scorer_cat['MID_RATE'] = df_scorer_cat['FGA_Mid_60G_Mavg']/df_scorer_cat['MIN_60G_Mavg']
df_scorer_cat['FG3_RATE'] = df_scorer_cat['FG3A_60G_Mavg']/df_scorer_cat['MIN_60G_Mavg']

In [1005]:
df_scorer_cat.fillna(0,inplace=True)

In [1006]:
df_scorer_cat

,PLAYER_ID,PLAYER_NAME,MIN_60G_Mavg,PTS_60G_Mavg,FGA_60G_Mavg,FG3A_60G_Mavg,MIN_60G_Sum,HEIGHT_INCHES,WEIGHT,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,SCORING_EFFICIENCY_Cut,SCORING_EFFICIENCY_Handoff,SCORING_EFFICIENCY_Isolation,SCORING_EFFICIENCY_Misc,SCORING_EFFICIENCY_OffRebound,SCORING_EFFICIENCY_OffScreen,SCORING_EFFICIENCY_PRBallHandler,SCORING_EFFICIENCY_PRRollMan,SCORING_EFFICIENCY_Postup,SCORING_EFFICIENCY_Spotup,SCORING_EFFICIENCY_Transition,SCORING_EFFICIENCY_PERCENTILE_Cut,SCORING_EFFICIENCY_PERCENTILE_Handoff,SCORING_EFFICIENCY_PERCENTILE_Isolation,SCORING_EFFICIENCY_PERCENTILE_Misc,SCORING_EFFICIENCY_PERCENTILE_OffRebound,SCORING_EFFICIENCY_PERCENTILE_OffScreen,SCORING_EFFICIENCY_PERCENTILE_PRBallHandler,SCORING_EFFICIENCY_PERCENTILE_PRRollMan,SCORING_EFFICIENCY_PERCENTILE_Postup,SCORING_EFFICIENCY_PERCENTILE_Spotup,SCORING_EFFICIENCY_PERCENTILE_Transition,VOLUME_SCORE_Cut,VOLUME_SCORE_Handoff,VOLUME_SCORE_Isolation,VOLUME_SCORE_Misc,VOLUME_SCORE_OffRebound,VOLUME_SCORE_OffScreen,VOLUME_SCORE_PRBallHandler,VOLUME_SCORE_PRRollMan,VOLUME_SCORE_Postup,VOLUME_SCORE_Spotup,VOLUME_SCORE_Transition,VOLUME_SCORE_PERCENTILE_Cut,VOLUME_SCORE_PERCENTILE_Handoff,VOLUME_SCORE_PERCENTILE_Isolation,VOLUME_SCORE_PERCENTILE_Misc,VOLUME_SCORE_PERCENTILE_OffRebound,VOLUME_SCORE_PERCENTILE_OffScreen,VOLUME_SCORE_PERCENTILE_PRBallHandler,VOLUME_SCORE_PERCENTILE_PRRollMan,VOLUME_SCORE_PERCENTILE_Postup,VOLUME_SCORE_PERCENTILE_Spotup,VOLUME_SCORE_PERCENTILE_Transition,AVG_SPEED_OFF_60G_Sum,OFFmiles_PER_MIN,PU_FG2_PCT_60Day,PU_FG3_PCT_60Day,PU_FGA_PCT_60Day,PU_FGA_rate_60Day,PU_FG3A_rate_60Day,CS_FG2_PCT_60Day,CS_FG3_PCT_60Day,CS_FGA_rate_60Day,CS_FG3A_rate_60Day,PTS_PER_DRIVE,DRIVES/MIN_60Day,SHOTS_DRIVE_RATE,FGA_3_AB_60G_Mavg,FGA_3_LC_60G_Mavg,FGA_3_RC_60G_Mavg,FGA_BC_60G_Mavg,FGA_Mid_60G_Mavg,FGA_Paint_60G_Mavg,FGA_RA_60G_Mavg,FGM_3_AB_60G_Mavg,FGM_3_LC_60G_Mavg,FGM_3_RC_60G_Mavg,FGM_BC_60G_Mavg,FGM_Mid_60G_Mavg,FGM_Paint_60G_Mavg,FGM_RA_60G_Mavg,RA_PCT,Paint_PCT,RA_RATE,Paint_RATE,MID_PCT,FG3_PCT,MID_RATE,FG3_RATE
0,1642377,Jaylen Wells,26.380472,11.250000,8.833333,5.233333,1582.828333,80.0,206,43.2,0.0,37.8,52.4,38.2,39.1,47.1,0.0,0.0,59.1,45.9,60.1,0.0,39.2,81.8,42.8,44.6,72.1,0.0,0.0,93.9,66.6,56.2,0.0,51.7,61.5,45.4,56.6,69.8,0.0,0.0,49.9,47.1,74.1,0.0,63.3,89.7,36.4,78.8,97.3,0.0,0.0,52.8,45.9,22.9,0.0,19.5,30.8,25.5,16.9,16.6,0.0,0.0,49.7,34.3,47.9,0.0,28.7,71.4,60.3,16.5,18.1,0.0,0.0,97.0,77.1,267.29,0.038706,0.500,0.410,0.458,0.05,0.02,0.500,0.371,0.17,0.16,0.49,0.17,0.35,3.133333,1.366667,0.666667,0.000000,0.783333,1.366667,1.450000,1.100000,0.516667,0.300000,0.0,0.366667,0.650000,0.966667,0.666667,0.475610,0.054965,0.051806,0.468085,0.366242,0.029694,0.198379
1,1642366,Quinten Post,16.130062,8.444444,6.814815,4.111111,435.511667,84.0,238,37.1,0.0,0.0,16.8,35.1,0.0,0.0,39.2,0.0,44.7,43.2,38.7,0.0,0.0,1.5,31.5,0.0,0.0,48.7,0.0,61.8,60.1,45.8,0.0,0.0,27.9,40.3,0.0,0.0,42.6,0.0,56.6,60.9,36.5,0.0,0.0,1.1,21.9,0.0,0.0,30.8,0.0,80.8,88.4,23.5,0.0,0.0,11.6,25.5,0.0,0.0,29.0,0.0,24.6,18.9,51.6,0.0,0.0,3.7,60.3,0.0,0.0,71.8,0.0,45.6,33.3,112.71,0.042121,1.000,0.333,0.500,0.01,0.01,0.667,0.456,0.23,0.22,0.42,0.05,0.58,3.192308,0.269231,0.769231,0.038462,0.269231,0.961538,1.576923,1.307692,0.076923,0.423077,0.0,0.153846,0.538462,0.730769,0.463415,0.560000,0.097763,0.059612,0.571429,0.439709,0.016691,0.254873
2,1642276,Kel'el Ware,20.471596,8.425532,6.617021,1.765

In [1007]:
# Drives
league_avg_pts_per_drive = df_scorer_cat['PTS_PER_DRIVE'].mean()
league_avg_DSR = df_scorer_cat['SHOTS_DRIVE_RATE'].mean()
league_avg_drives_min = df_scorer_cat['DRIVES/MIN_60Day'].mean()

#ISO
league_avg_iso_shot = df_scorer_cat['OVERALL_SCORE_Isolation'].mean()
#league_avg_iso_pct = df_scorer_cat['FG_PCT_Isolation'].mean()

#PNR BallHandler
league_avg_pnrh_shot = df_scorer_cat['OVERALL_SCORE_PRBallHandler'].mean()
#league_avg_pnrh_pct = df_scorer_cat['FG_PCT_PRBallHandler'].mean()

#PNRRollman
league_avg_pnrman_shot = df_scorer_cat['OVERALL_SCORE_PRRollMan'].mean()
#eague_avg_pnrman_pct = df_scorer_cat['FG_PCT_PRRollMan'].mean()

#PNRRollman
league_avg_putback_shot = df_scorer_cat['OVERALL_SCORE_OffRebound'].mean()
#league_avg_putback_pct = df_scorer_cat['FG_PCT_OffRebound'].mean()

#_Cut
league_avg_cut_shot = df_scorer_cat['OVERALL_SCORE_Cut'].mean()
#league_avg_cut_pct = df_scorer_cat['FG_PCT_Cut'].mean()

#_Postup
league_avg_postup_shot = df_scorer_cat['OVERALL_SCORE_Postup'].mean()
#league_avg_cut_pct = df_scorer_cat['FG_PCT_Postup'].mean()

#_Pullup
league_avg_fg_pct_PU = df_scorer_cat['PU_FGA_PCT_60Day'].mean()
league_avg_fga_per_min_PU = df_scorer_cat['PU_FGA_rate_60Day'].mean()

#Paint Shots
league_avg_ra_pct = df_scorer_cat['RA_PCT'].mean()
league_avg_paint_pct = df_scorer_cat['Paint_PCT'].mean()
league_avg_ra_min = df_scorer_cat['RA_RATE'].mean()
league_avg_paint_min = df_scorer_cat['Paint_RATE'].mean()
league_avg_mid_pct = df_scorer_cat['MID_PCT'].mean()
league_avg_fg3_pct = df_scorer_cat['FG3_PCT'].mean()
league_avg_mid_min = df_scorer_cat['MID_RATE'].mean()
league_avg_fg3_min = df_scorer_cat['FG3_RATE'].mean()



       

df_scorer_cat['pu_eff_ratio'] = (df_scorer_cat['PU_FGA_PCT_60Day'] / league_avg_fg_pct_PU) 
df_scorer_cat['drive_eff_ratio'] = df_scorer_cat['PTS_PER_DRIVE'] / league_avg_pts_per_drive
#df_scorer_cat['pnr_eff_ratio'] = df_scorer_cat['FG_PCT_PRBallHandler'] / league_avg_pnrh_pct
#df_scorer_cat['pnrman_eff_ratio'] = df_scorer_cat['FG_PCT_PRRollMan'] / league_avg_pnrman_pct
#df_scorer_cat['iso_eff_ratio'] = df_scorer_cat['FG_PCT_Isolation'] / league_avg_iso_pct
#df_scorer_cat['cut_eff_ratio'] = df_scorer_cat['FG_PCT_Cut'] / league_avg_cut_pct
#df_scorer_cat['putback_eff_ratio'] = df_scorer_cat['FG_PCT_OffRebound'] / league_avg_putback_pct
df_scorer_cat['ra_eff_ratio'] = df_scorer_cat['RA_PCT'] / league_avg_ra_pct
df_scorer_cat['paint_eff_ratio'] = df_scorer_cat['Paint_PCT'] / league_avg_paint_pct
df_scorer_cat['mid_eff_ratio'] = df_scorer_cat['MID_PCT'] / league_avg_mid_pct
df_scorer_cat['fg3_eff_ratio'] = df_scorer_cat['FG3_PCT'] / league_avg_fg3_pct
    
    # Calculate volume vs. league average ratios
df_scorer_cat['pu_vol_ratio'] = (df_scorer_cat['PU_FGA_rate_60Day'] / league_avg_fga_per_min_PU)
df_scorer_cat['drive_vol_ratio'] = df_scorer_cat['DRIVES/MIN_60Day'] / league_avg_drives_min
#df_scorer_cat['pnr_vol_ratio'] = df_scorer_cat['FGA/G_PRBallHandler'] / league_avg_pnrh_shot
#df_scorer_cat['pnrman_vol_ratio'] = df_scorer_cat['FGA/G_PRRollMan'] / league_avg_pnrman_shot
#df_scorer_cat['iso_vol_ratio'] = df_scorer_cat['FGA/G_Isolation'] / league_avg_iso_shot
#df_scorer_cat['cut_vol_ratio'] = df_scorer_cat['FGA/G_Cut'] / league_avg_cut_shot
#df_scorer_cat['putback_vol_ratio'] = df_scorer_cat['FGA/G_OffRebound'] / league_avg_putback_shot
df_scorer_cat['ra_vol_ratio'] = df_scorer_cat['RA_RATE'] / league_avg_ra_min
df_scorer_cat['paint_vol_ratio'] = df_scorer_cat['Paint_RATE'] / league_avg_paint_min
df_scorer_cat['mid_vol_ratio'] = df_scorer_cat['MID_RATE'] / league_avg_mid_min
df_scorer_cat['fg3_vol_ratio'] = df_scorer_cat['FG3_RATE'] / league_avg_fg3_min

#drives
df_scorer_cat['driving_score'] = (
    df_scorer_cat['drive_eff_ratio'] * 0.4 +
    df_scorer_cat['drive_vol_ratio'] * 0.6
)

#iso
df_scorer_cat['iso_score'] = (
        (df_scorer_cat['OVERALL_SCORE_Isolation'] / league_avg_iso_shot) # Scale by frequency to reward volume
    )
#putback
df_scorer_cat['putback_score'] = (
        (df_scorer_cat['OVERALL_SCORE_OffRebound'] / league_avg_putback_shot) # Scale by frequency to reward volume
    )

#cut
df_scorer_cat['cut_score'] = (
        (df_scorer_cat['OVERALL_SCORE_Cut'] / league_avg_cut_shot) # Scale by frequency to reward volume
    )

#postup
df_scorer_cat['postup_score'] = (
        (df_scorer_cat['OVERALL_SCORE_Postup'] / league_avg_postup_shot) # Scale by frequency to reward volume
    )
#pullup
df_scorer_cat['pullup_score'] = (
    df_scorer_cat['pu_eff_ratio'] * 0.5 +
    df_scorer_cat['pu_vol_ratio'] * 0.5
)

#pnrman_score
df_scorer_cat['pnrman_score'] = (
        (df_scorer_cat['OVERALL_SCORE_PRRollMan'] / league_avg_pnrman_shot) # Scale by frequency to reward volume
    )

    # PnR Score (30% weight)
df_scorer_cat['pnr_score'] = (
        (df_scorer_cat['OVERALL_SCORE_PRBallHandler'] / league_avg_pnrh_shot) # Scale by frequency to reward volume
    )

    # mid
df_scorer_cat['mid_score'] = (
    df_scorer_cat['mid_eff_ratio'] * 0.5 +
    df_scorer_cat['mid_vol_ratio'] * 0.5
)

    # fg3
df_scorer_cat['fg3_score'] = (
    df_scorer_cat['fg3_eff_ratio'] * 0.5 +
    df_scorer_cat['fg3_vol_ratio'] * 0.5
)

# Interior Score (30% weight)
df_scorer_cat['interior_score'] = (
    (df_scorer_cat['ra_eff_ratio'] * 0.45 + df_scorer_cat['ra_vol_ratio'] * 0.55) * 0.5 +  # RA component
    (df_scorer_cat['paint_eff_ratio'] * 0.45 + df_scorer_cat['paint_vol_ratio'] * 0.55) * 0.5  # Paint component
)

df_scorer_cat['penetrator_score'] = (
        df_scorer_cat['driving_score'] * 0.40 +    # Driving ability
        df_scorer_cat['pnr_score'] * 0.30 +        # Pick and Roll scoring
        df_scorer_cat['interior_score'] * 0.30  # Interior scoring
    ).round(4)

average_score = df_scorer_cat['penetrator_score'].mean()
df_scorer_cat['penetrator_score'] = (df_scorer_cat['penetrator_score'] / average_score) * 100



df_scorer_cat['rim_runner_score'] = (
        df_scorer_cat['pnrman_score'] * 0.25 +    
        df_scorer_cat['cut_score'] * 0.25 + 
        df_scorer_cat['putback_score'] * 0.20 + 
        df_scorer_cat['interior_score'] * 0.30     
    ).round(4)

average_score = df_scorer_cat['rim_runner_score'].mean()
df_scorer_cat['rim_runner_score'] = (df_scorer_cat['rim_runner_score'] / average_score) * 100

df_scorer_cat['pure_score'] = (
        df_scorer_cat['iso_score'] * 0.30 + 
        df_scorer_cat['pullup_score'] * 0.30 +  
        df_scorer_cat['mid_score'] * 0.15 +  # Driving ability
        df_scorer_cat['fg3_score'] * 0.20 +        # Pick and Roll scoring
        #df_scorer_cat['driving_score'] * 0.20 +
        df_scorer_cat['driving_score'] * 0.25 # Interior scoring
    ).round(4)



average_score = df_scorer_cat['pure_score'].mean()
df_scorer_cat['pure_score'] = (df_scorer_cat['pure_score'] / average_score) * 100

df_scorer_cat.fillna(0,inplace=True)


In [1008]:
df_scorer_cat.sort_values(by='penetrator_score', ascending=False).head(20)

,PLAYER_ID,PLAYER_NAME,MIN_60G_Mavg,PTS_60G_Mavg,FGA_60G_Mavg,FG3A_60G_Mavg,MIN_60G_Sum,HEIGHT_INCHES,WEIGHT,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,SCORING_EFFICIENCY_Cut,SCORING_EFFICIENCY_Handoff,SCORING_EFFICIENCY_Isolation,SCORING_EFFICIENCY_Misc,SCORING_EFFICIENCY_OffRebound,SCORING_EFFICIENCY_OffScreen,SCORING_EFFICIENCY_PRBallHandler,SCORING_EFFICIENCY_PRRollMan,SCORING_EFFICIENCY_Postup,SCORING_EFFICIENCY_Spotup,SCORING_EFFICIENCY_Transition,SCORING_EFFICIENCY_PERCENTILE_Cut,SCORING_EFFICIENCY_PERCENTILE_Handoff,SCORING_EFFICIENCY_PERCENTILE_Isolation,SCORING_EFFICIENCY_PERCENTILE_Misc,SCORING_EFFICIENCY_PERCENTILE_OffRebound,SCORING_EFFICIENCY_PERCENTILE_OffScreen,SCORING_EFFICIENCY_PERCENTILE_PRBallHandler,SCORING_EFFICIENCY_PERCENTILE_PRRollMan,SCORING_EFFICIENCY_PERCENTILE_Postup,SCORING_EFFICIENCY_PERCENTILE_Spotup,SCORING_EFFICIENCY_PERCENTILE_Transition,VOLUME_SCORE_Cut,VOLUME_SCORE_Handoff,VOLUME_SCORE_Isolation,VOLUME_SCORE_Misc,VOLUME_SCORE_OffRebound,VOLUME_SCORE_OffScreen,VOLUME_SCORE_PRBallHandler,VOLUME_SCORE_PRRollMan,VOLUME_SCORE_Postup,VOLUME_SCORE_Spotup,VOLUME_SCORE_Transition,VOLUME_SCORE_PERCENTILE_Cut,VOLUME_SCORE_PERCENTILE_Handoff,VOLUME_SCORE_PERCENTILE_Isolation,VOLUME_SCORE_PERCENTILE_Misc,VOLUME_SCORE_PERCENTILE_OffRebound,VOLUME_SCORE_PERCENTILE_OffScreen,VOLUME_SCORE_PERCENTILE_PRBallHandler,VOLUME_SCORE_PERCENTILE_PRRollMan,VOLUME_SCORE_PERCENTILE_Postup,VOLUME_SCORE_PERCENTILE_Spotup,VOLUME_SCORE_PERCENTILE_Transition,AVG_SPEED_OFF_60G_Sum,OFFmiles_PER_MIN,PU_FG2_PCT_60Day,PU_FG3_PCT_60Day,PU_FGA_PCT_60Day,PU_FGA_rate_60Day,PU_FG3A_rate_60Day,CS_FG2_PCT_60Day,CS_FG3_PCT_60Day,CS_FGA_rate_60Day,CS_FG3A_rate_60Day,PTS_PER_DRIVE,DRIVES/MIN_60Day,SHOTS_DRIVE_RATE,FGA_3_AB_60G_Mavg,FGA_3_LC_60G_Mavg,FGA_3_RC_60G_Mavg,FGA_BC_60G_Mavg,FGA_Mid_60G_Mavg,FGA_Paint_60G_Mavg,FGA_RA_60G_Mavg,FGM_3_AB_60G_Mavg,FGM_3_LC_60G_Mavg,FGM_3_RC_60G_Mavg,FGM_BC_60G_Mavg,FGM_Mid_60G_Mavg,FGM_Paint_60G_Mavg,FGM_RA_60G_Mavg,RA_PCT,Paint_PCT,RA_RATE,Paint_RATE,MID_PCT,FG3_PCT,MID_RATE,FG3_RATE,pu_eff_ratio,drive_eff_ratio,ra_eff_ratio,paint_eff_ratio,mid_eff_ratio,fg3_eff_ratio,pu_vol_ratio,drive_vol_ratio,ra_vol_ratio,paint_vol_ratio,mid_vol_ratio,fg3_vol_ratio,driving_score,iso_score,putback_score,cut_score,postup_score,pullup_score,pnrman_score,pnr_score,mid_score,fg3_score,interior_score,penetrator_score,rim_runner_score,pure_score
136,1628983,Shai Gilgeous-Alexander,34.545639,32.950000,21.433333,5.766667,2072.738333,78.0,195,44.1,54.5,89.8,56.7,49.3,45.4,79.1,0.0,44.4,53.8,69.7,62.5,86.3,99.6,89.6,77.0,67.3,99.7,0.0,66.9,83.4,98.7,59.1,69.6,57.6,62.4,70.6,57.3,62.6,0.0,51.3,70.2,54.1,84.0,96.8,82.6,92.1,98.8,81.6,93.2,0.0,62.6,98.1,73.9,21.6,26.7,84.5,35.8,18.8,24.9,65.9,0.0,28.6,25.3,60.4,44.0,62.1,99.6,81.1,28.7,58.2,98.5,0.0,73.9,48.0,99.1,266.86,0.038971,0.507,0.342,0.440,0.33,0.13,0.684,0.500,0.03,0.03,0.77,0.59,0.49,5.483333,0.133333,0.033333,0.000000,4.650000,5.583333,5.533333,2.100000,0.050000,0.000000,0.000000,2.400000,2.900000,3.950000,0.713855,0.519403,0.160175,0.161622,0.516129,0.372832,0.134605,0.166929,1.213954,1.295165,1.087491,1.204024,1.308729,1.105488,3.159856,2.672669,1.431197,1.966162,3.122638,1.030409,2.121667,2.443245,1.304507,1.140349,2.103556,2.186905,0.000000,2.030449,2.215684,1.067949,1.449864,189.279683,98.090205,205.449857
117,1629627,Zion Williamson,30.959583,24.250000,16.833333,0.350000,1857.575000,78.0,284

In [1009]:
#average_score

In [1010]:
df_scores = pd.DataFrame(df_scorer_cat[['PLAYER_ID','PLAYER_NAME','PTS_60G_Mavg','penetrator_score','rim_runner_score','pure_score','OVERALL_SCORE_Cut',
       'OVERALL_SCORE_Handoff', 'OVERALL_SCORE_Isolation',
       'OVERALL_SCORE_Misc', 'OVERALL_SCORE_OffRebound',
       'OVERALL_SCORE_OffScreen', 'OVERALL_SCORE_PRBallHandler',
       'OVERALL_SCORE_PRRollMan', 'OVERALL_SCORE_Postup',
       'OVERALL_SCORE_Spotup', 'OVERALL_SCORE_Transition',
       'OVERALL_SCORE_PERCENTILE_Cut', 'OVERALL_SCORE_PERCENTILE_Handoff',
       'OVERALL_SCORE_PERCENTILE_Isolation', 'OVERALL_SCORE_PERCENTILE_Misc',
       'OVERALL_SCORE_PERCENTILE_OffRebound',
       'OVERALL_SCORE_PERCENTILE_OffScreen',
       'OVERALL_SCORE_PERCENTILE_PRBallHandler',
       'OVERALL_SCORE_PERCENTILE_PRRollMan', 'OVERALL_SCORE_PERCENTILE_Postup',
       'OVERALL_SCORE_PERCENTILE_Spotup',
       'OVERALL_SCORE_PERCENTILE_Transition', 'SCORING_EFFICIENCY_Cut',
       'SCORING_EFFICIENCY_Handoff', 'SCORING_EFFICIENCY_Isolation',
       'SCORING_EFFICIENCY_Misc', 'SCORING_EFFICIENCY_OffRebound',
       'SCORING_EFFICIENCY_OffScreen', 'SCORING_EFFICIENCY_PRBallHandler',
       'SCORING_EFFICIENCY_PRRollMan', 'SCORING_EFFICIENCY_Postup',
       'SCORING_EFFICIENCY_Spotup', 'SCORING_EFFICIENCY_Transition',
       'SCORING_EFFICIENCY_PERCENTILE_Cut',
       'SCORING_EFFICIENCY_PERCENTILE_Handoff',
       'SCORING_EFFICIENCY_PERCENTILE_Isolation',
       'SCORING_EFFICIENCY_PERCENTILE_Misc',
       'SCORING_EFFICIENCY_PERCENTILE_OffRebound',
       'SCORING_EFFICIENCY_PERCENTILE_OffScreen',
       'SCORING_EFFICIENCY_PERCENTILE_PRBallHandler',
       'SCORING_EFFICIENCY_PERCENTILE_PRRollMan',
       'SCORING_EFFICIENCY_PERCENTILE_Postup',
       'SCORING_EFFICIENCY_PERCENTILE_Spotup',
       'SCORING_EFFICIENCY_PERCENTILE_Transition', 'VOLUME_SCORE_Cut',
       'VOLUME_SCORE_Handoff', 'VOLUME_SCORE_Isolation', 'VOLUME_SCORE_Misc',
       'VOLUME_SCORE_OffRebound', 'VOLUME_SCORE_OffScreen',
       'VOLUME_SCORE_PRBallHandler', 'VOLUME_SCORE_PRRollMan',
       'VOLUME_SCORE_Postup', 'VOLUME_SCORE_Spotup', 'VOLUME_SCORE_Transition',
       'VOLUME_SCORE_PERCENTILE_Cut', 'VOLUME_SCORE_PERCENTILE_Handoff',
       'VOLUME_SCORE_PERCENTILE_Isolation', 'VOLUME_SCORE_PERCENTILE_Misc',
       'VOLUME_SCORE_PERCENTILE_OffRebound',
       'VOLUME_SCORE_PERCENTILE_OffScreen',
       'VOLUME_SCORE_PERCENTILE_PRBallHandler',
       'VOLUME_SCORE_PERCENTILE_PRRollMan', 'VOLUME_SCORE_PERCENTILE_Postup',
       'VOLUME_SCORE_PERCENTILE_Spotup', 'VOLUME_SCORE_PERCENTILE_Transition','interior_score',]])

In [1011]:
df_scores.sort_values(by='pure_score')

,PLAYER_ID,PLAYER_NAME,PTS_60G_Mavg,penetrator_score,rim_runner_score,pure_score,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,SCORING_EFFICIENCY_Cut,SCORING_EFFICIENCY_Handoff,SCORING_EFFICIENCY_Isolation,SCORING_EFFICIENCY_Misc,SCORING_EFFICIENCY_OffRebound,SCORING_EFFICIENCY_OffScreen,SCORING_EFFICIENCY_PRBallHandler,SCORING_EFFICIENCY_PRRollMan,SCORING_EFFICIENCY_Postup,SCORING_EFFICIENCY_Spotup,SCORING_EFFICIENCY_Transition,SCORING_EFFICIENCY_PERCENTILE_Cut,SCORING_EFFICIENCY_PERCENTILE_Handoff,SCORING_EFFICIENCY_PERCENTILE_Isolation,SCORING_EFFICIENCY_PERCENTILE_Misc,SCORING_EFFICIENCY_PERCENTILE_OffRebound,SCORING_EFFICIENCY_PERCENTILE_OffScreen,SCORING_EFFICIENCY_PERCENTILE_PRBallHandler,SCORING_EFFICIENCY_PERCENTILE_PRRollMan,SCORING_EFFICIENCY_PERCENTILE_Postup,SCORING_EFFICIENCY_PERCENTILE_Spotup,SCORING_EFFICIENCY_PERCENTILE_Transition,VOLUME_SCORE_Cut,VOLUME_SCORE_Handoff,VOLUME_SCORE_Isolation,VOLUME_SCORE_Misc,VOLUME_SCORE_OffRebound,VOLUME_SCORE_OffScreen,VOLUME_SCORE_PRBallHandler,VOLUME_SCORE_PRRollMan,VOLUME_SCORE_Postup,VOLUME_SCORE_Spotup,VOLUME_SCORE_Transition,VOLUME_SCORE_PERCENTILE_Cut,VOLUME_SCORE_PERCENTILE_Handoff,VOLUME_SCORE_PERCENTILE_Isolation,VOLUME_SCORE_PERCENTILE_Misc,VOLUME_SCORE_PERCENTILE_OffRebound,VOLUME_SCORE_PERCENTILE_OffScreen,VOLUME_SCORE_PERCENTILE_PRBallHandler,VOLUME_SCORE_PERCENTILE_PRRollMan,VOLUME_SCORE_PERCENTILE_Postup,VOLUME_SCORE_PERCENTILE_Spotup,VOLUME_SCORE_PERCENTILE_Transition,interior_score
207,203497,Rudy Gobert,11.500000,41.149931,159.960335,7.666661,73.5,0.0,0.0,52.8,70.6,0.0,0.0,59.7,26.8,0.0,47.0,98.8,0.0,0.0,82.1,97.7,0.0,0.0,91.5,11.0,0.0,69.5,52.3,0.0,0.0,62.4,52.6,0.0,0.0,56.7,37.5,0.0,68.6,62.6,0.0,0.0,92.1,62.4,0.0,0.0,75.9,15.0,0.0,94.9,67.1,0.0,0.0,30.6,62.9,0.0,0.0,44.7,16.7,0.0,17.5,99.2,0.0,0.0,69.9,97.2,0.0,0.0,90.5,15.3,0.0,26.9,0.904490
32,1631117,Walker Kessler,10.566667,41.889930,169.070354,11.499992,71.1,0.0,0.0,43.0,84.9,0.0,0.0,54.9,30.1,7.9,51.2,97.9,0.0,0.0,62.5,100.0,0.0,0.0,86.7,18.7,1.5,80.9,58.6,0.0,0.0,43.2,63.8,0.0,0.0,57.0,45.6,25.5,72.0,82.8,0.0,0.0,29.5,94.5,0.0,0.0,77.2,36.2,4.5,96.6,58.5,0.0,0.0,33.7,72.7,0.0,0.0,38.0,14.2,1.5,20.2,97.1,0.0,0.0,76.9,99.7,0.0,0.0,85.4,5.8,1.1,39.1,1.136177
192,203994,Jusuf Nurkić,8.450000,47.849920,135.900284,26.674981,64.8,0.0,0.0,43.2,42.0,0.0,0.0,52.3,50.2,7.0,20.5,96.5,0.0,0.0,63.6,54.4,0.0,0.0,82.5,81.6,1.3,4.2,69.5,0.0,0.0,39.4,42.2,0.0,0.0,41.0,49.3,10.2,31.2,98.2,0.0,0.0,20.0,26.6,0.0,0.0,26.6,52.1,0.4,5.4,40.7,0.0,0.0,37.3,33.2,0.0,0.0,48.1,38.3,13.4,13.5,87.0,0.0,0.0,83.5,79.1,0.0,0.0,93.6,85.9,12.4,11.5,0.992725
35,1631109,Mark Williams,14.588235,65.769890,190.780399,27.258314,81.0,0.0,0.0,39.0,66.4,0.0,0.0,76.2,40.1,0.0,55.6,100.0,0.0,0.0,47.3,95.9,0.0,0.0,98.7,51.5,0.0,89.8,50.4,0.0,0.0,42.9,48.1,0.0,0.0,55.0,42.9,0.0,71.5,55.6,0.0,0.0,28.5,44.9,0.0,0.0,70.2,26.4,0.0,96.4,78.9,0.0,0.0,28.6,61.1,0.0,0.0,68.5,30.1,0.0,26.5,100.0,0.0,0.0,65.6,97.0,0.0,0.0,98.7,78.5,0.0,58.6,1.402494
82,1630208,Nick Richards,9.533333,60.989898,144.910303,29.366646,44.8,0.0,0.0,34.9,63.6,0.0,0.0,61.4,17.9,0.0,31.2,64.4,0.0,0.0,29.9,94.5,0.0,0.0,92.8,2.5,0.0,19.8,27.8,0.0,0.0,34.8,66.3,0.0,0.0,58.7,25.6,0.0,46.5,3.8,0.0,0.0,9.8,96.6,0.0,0.0,82.8,2.5,0.0,42.8,49.4,0.0,0.0,30.0,41.8,0.0,0.0,45.3,15.0,0.0,15.0,94.0,0.0,0.0,67.9,88.8,0.0,0.0,90.8,9.5,0.0,17.3,1.099340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [1012]:
df_scores['as_of'] = pd.to_datetime(today)
df_scores['id'] = df_scores['as_of'].astype(str)+"_"+df_scores['PLAYER_ID'].astype(str)

In [1013]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_points_scores_new"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    df_scores.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = df_scores[~df_scores['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'player_points_scores_new' already exists. Checking for new records...
Inserted 239 new records into 'player_points_scores_new'.


# Shooters Categories

In [1014]:
threshold_3 = 3
df_playtypes_cat_3s = pd.DataFrame(df_playtypes_cat.loc[df_playtypes_cat['FG3A_60G_Mavg']>threshold_3]).reset_index(drop=True)

In [1015]:
df_playtypes_cat.loc[df_playtypes_cat['FG3A_60G_Mavg']>threshold_3]['PLAYER_NAME'].nunique()

241

In [1016]:
shotdetail_60Day_3add = shotdetail_60Day_new[['PLAYER_ID', 'FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGM_3_AB_60G_Mavg',
       'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg', ]]

In [1017]:
df_full_pt = df_playtypes_cat_3s.merge(shotdetail_60Day_3add, how='left')

In [1018]:
df_full_pt

,SEASON_YEAR,PLAYER_ID,TEAM_ID,GAME_ID,GAME_DATE,GAMES_IN_WINDOW_60G,MIN_60G_Mavg,PTS_60G_Mavg,REB_60G_Mavg,AST_60G_Mavg,FGA_60G_Mavg,FG3M_60G_Mavg,FG3A_60G_Mavg,FGA_60G_Sum,FG3M_60G_Sum,FG3A_60G_Sum,MIN_60G_Sum,PTS_60G_Sum,HEIGHT_INCHES,WEIGHT,SEASON_EXP,PLAYER_NAME,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,SCORING_EFFICIENCY_Cut,SCORING_EFFICIENCY_Handoff,SCORING_EFFICIENCY_Isolation,SCORING_EFFICIENCY_Misc,SCORING_EFFICIENCY_OffRebound,SCORING_EFFICIENCY_OffScreen,SCORING_EFFICIENCY_PRBallHandler,SCORING_EFFICIENCY_PRRollMan,SCORING_EFFICIENCY_Postup,SCORING_EFFICIENCY_Spotup,SCORING_EFFICIENCY_Transition,SCORING_EFFICIENCY_PERCENTILE_Cut,SCORING_EFFICIENCY_PERCENTILE_Handoff,SCORING_EFFICIENCY_PERCENTILE_Isolation,SCORING_EFFICIENCY_PERCENTILE_Misc,SCORING_EFFICIENCY_PERCENTILE_OffRebound,SCORING_EFFICIENCY_PERCENTILE_OffScreen,SCORING_EFFICIENCY_PERCENTILE_PRBallHandler,SCORING_EFFICIENCY_PERCENTILE_PRRollMan,SCORING_EFFICIENCY_PERCENTILE_Postup,SCORING_EFFICIENCY_PERCENTILE_Spotup,SCORING_EFFICIENCY_PERCENTILE_Transition,VOLUME_SCORE_Cut,VOLUME_SCORE_Handoff,VOLUME_SCORE_Isolation,VOLUME_SCORE_Misc,VOLUME_SCORE_OffRebound,VOLUME_SCORE_OffScreen,VOLUME_SCORE_PRBallHandler,VOLUME_SCORE_PRRollMan,VOLUME_SCORE_Postup,VOLUME_SCORE_Spotup,VOLUME_SCORE_Transition,VOLUME_SCORE_PERCENTILE_Cut,VOLUME_SCORE_PERCENTILE_Handoff,VOLUME_SCORE_PERCENTILE_Isolation,VOLUME_SCORE_PERCENTILE_Misc,VOLUME_SCORE_PERCENTILE_OffRebound,VOLUME_SCORE_PERCENTILE_OffScreen,VOLUME_SCORE_PERCENTILE_PRBallHandler,VOLUME_SCORE_PERCENTILE_PRRollMan,VOLUME_SCORE_PERCENTILE_Postup,VOLUME_SCORE_PERCENTILE_Spotup,VOLUME_SCORE_PERCENTILE_Transition,POSS_PCT_Cut,POSS_PCT_Handoff,POSS_PCT_Isolation,POSS_PCT_Misc,POSS_PCT_OffRebound,POSS_PCT_OffScreen,POSS_PCT_PRBallHandler,POSS_PCT_PRRollMan,POSS_PCT_Postup,POSS_PCT_Spotup,POSS_PCT_Transition,AVG_SPEED_OFF_60G_Sum,AVG_SPEED_DEF_60G_Sum,OFFmiles_PER_MIN,DEFmiles_PER_MIN,OREB_PER_CHANCE,CONTEST_OREB_PER_CHANCE,DREB_PER_CHANCE,CONTEST_DREB_PER_CHANCE,OREB_CHANCES_60G_Sum,DREB_CHANCES_60G_Sum,PASSES_MADE_60G_Sum,PASSES_RECEIVED_60G_Sum,AST_60G_Sum,POTENTIAL_AST_60G_Sum,SECONDARY_AST_60G_Sum,PASSES_MIN_60G_Sum,AST_PASS_60G_Sum,AST_PER_MIN,AST_TO_POTENTIAL,SECONDARY_AST_RATE,POINTS_PER_AST,PU_FGA_PCT_60Day,PU_FG2_PCT_60Day,PU_FG3_PCT_60Day,PU_FGA_rate_60Day,PU_FG3A_rate_60Day,CS_FG2_PCT_60Day,CS_FG3_PCT_60Day,CS_FGA_rate_60Day,CS_FG3A_rate_60Day,DRIVES/MIN_60Day,DRIVES_AST_PASS_RATE,DRIVES_PASS_RATE,SHOTS_DRIVE_RATE,PTS_PER_DRIVE,FGA_60G_Mavg_0 Dribbles,FGA_60G_Mavg_1 Dribble,FGA_60G_Mavg_2 Dribbles,FGA_60G_Mavg_3-6 Dribbles,FGA_60G_Mavg_7+ Dribbles,FG2M_60G_Mavg_0 Dribbles,FG2M_60G_Mavg_1 Dribble,FG2M_60G_Mavg_2 Dribbles,FG2M_60G_Mavg_3-6 Dribbles,FG2M_60G_Mavg_7+ Dribbles,FG3M_60G_Mavg_0 Dribbles,FG3M_60G_Mavg_1 Dribble,FG3M_60G_Mavg_2 Dribbles,FG3M_60G_Mavg_3-6 Dribbles,FG3M_60G_Mavg_7+ Dribbles,FGA_60G_Mavg_Touch 2-6 Seconds,FGA_60G_Mavg_Touch 6+ Seconds,FGA_60G_Mavg_Touch < 2 Seconds,FG2M_60G_Mavg_Touch 2-6 Seconds,FG2M_60G_Mavg_Touch 6+ Seconds,FG2M_60G_Mavg_Touch < 2 Seconds,FG3M_60G_Mavg_Touch 2-6 Seconds,FG3M_60G_Mavg_Touch 6+ Seconds,FG3M_60G_Mavg_Touch < 2 Seconds,FGA_3_AB_60G_Mavg,FGA_3_LC_60G_Mavg,FGA_3_RC_60G_Mavg,FGM_3_AB_60G_Mavg,FGM_3_LC_60G_Mavg,FGM_3_RC_60G_Mavg
0,2024-25,1642419,1610612761,0022400947,2025-03-12,43.0,13.619070,5.860465,2.116279,0.790698,4.581395,1.441860,3.465116,197.0,62.0,149.0,585.620000,252.0,79.0,220,0.0,Ja

In [1019]:
df_full_pt['FG3A_per_MIN_60G']  = df_full_pt['FG3A_60G_Sum']/df_full_pt['MIN_60G_Sum']

In [1020]:
df_3s_category = pd.DataFrame(df_full_pt[['MIN_60G_Mavg','FG3A_per_MIN_60G','FG3A_60G_Mavg','HEIGHT_INCHES',
       'VOLUME_SCORE_Handoff', 'VOLUME_SCORE_Isolation', 'VOLUME_SCORE_OffScreen',
       'VOLUME_SCORE_PRBallHandler', 'VOLUME_SCORE_PRRollMan', 'VOLUME_SCORE_Spotup', 'PU_FG3_PCT_60Day','PU_FG3A_rate_60Day','CS_FG3_PCT_60Day','CS_FG3A_rate_60Day',
           'FG3M_60G_Mavg_0 Dribbles','FG3M_60G_Mavg_2 Dribbles','FG3M_60G_Mavg_3-6 Dribbles','FG3M_60G_Mavg_7+ Dribbles','FG3M_60G_Mavg_Touch 2-6 Seconds',
       'FG3M_60G_Mavg_Touch 6+ Seconds', 'FG3M_60G_Mavg_Touch < 2 Seconds','FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGM_3_AB_60G_Mavg',
       'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg','OFFmiles_PER_MIN',]])

In [1021]:
df_3s_category['FG3A_Corner_Rate'] = df_3s_category['FG3A_60G_Mavg']/(df_3s_category['FGA_3_RC_60G_Mavg']+df_3s_category['FGA_3_LC_60G_Mavg'])
#df_3s_category['FG3A_Corner_Rate'] = df_3s_category['FG3A_60G_Mavg']/(df_3s_category['FGA_3_RC_60G_Msavg']+df_3s_category['FGA_3_LC_60G_Msavg'])

In [1022]:
DF_3s_cluster = pd.DataFrame(df_full_pt[['PLAYER_ID','PLAYER_NAME','FG3A_60G_Mavg','FG3A_per_MIN_60G']])

In [1023]:
df_3s_category.fillna(0, inplace=True)

In [1024]:
#df_3s_category.isnull().sum() 
df_3s_category.replace([np.inf, -np.inf], np.nan, inplace=True)

In [1025]:
df_3s_category.fillna(0, inplace=True)

In [1026]:
from sklearn.preprocessing import StandardScaler

# Assume df contains your original data (without non-numeric columns like player names or IDs)
scaler = StandardScaler()
df_scaled_player = scaler.fit_transform(df_3s_category)

In [1027]:
from sklearn.cluster import KMeans

# Define the number of clusters (e.g., 5 categories)
kmeans = KMeans(n_clusters=6, random_state=42)
DF_3s_cluster['Cluster'] = kmeans.fit_predict(df_scaled_player)

In [1028]:
DF_3s_cluster.loc[DF_3s_cluster['Cluster']==5]

,PLAYER_ID,PLAYER_NAME,FG3A_60G_Mavg,FG3A_per_MIN_60G,Cluster
7,1642272,Jared McCain,5.782609,0.224741,5
28,1641711,Gradey Dick,6.083333,0.206816,5
30,1641706,Brandon Miller,9.183333,0.270689,5
31,1641705,Victor Wembanyama,8.383333,0.253517,5
77,1630530,Trey Murphy III,8.250000,0.230165,5
86,1630202,Payton Pritchard,7.666667,0.273451,5
87,1630198,Isaiah Joe,5.883333,0.287313,5
99,1630170,Devin Vassell,6.383333,0.199879,5
110,1629675,Naz Reid,6.066667,0.214247,5
112,1629661,Cameron Johnson,6.933333,0.225630,5


In [1029]:
DF_3s_cluster['as_of'] = pd.to_datetime(today)

In [1030]:
DF_3s_cluster['id'] = DF_3s_cluster['as_of'].astype(str)+"_"+DF_3s_cluster['PLAYER_ID'].astype(str)

In [1031]:
DF_3s_cluster

,PLAYER_ID,PLAYER_NAME,FG3A_60G_Mavg,FG3A_per_MIN_60G,Cluster,as_of,id
0,1642419,Jamison Battle,3.465116,0.254431,1,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_1642419
1,1642377,Jaylen Wells,5.233333,0.198379,1,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_1642377
2,1642366,Quinten Post,4.111111,0.254873,1,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_1642366
3,1642354,KJ Simpson,3.217391,0.157419,2,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_1642354
4,1642348,Justin Edwards,4.000000,0.166015,4,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_1642348
...,...,...,...,...,...,...,...
236,201143,Al Horford,4.950000,0.181465,1,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_201143
237,201142,Kevin Durant,6.166667,0.166785,5,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_201142
238,200768,Kyle Lowry,3.150000,0.137134,2,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_200768
239,101108,Chris Paul,4.783333,0.167975,2,2025-03-15 16:24:17.960182,2025-03-15 16:24:17.960182_101108


In [1032]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_3s_cluster_new"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    DF_3s_cluster.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = DF_3s_cluster[~DF_3s_cluster['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'player_3s_cluster_new' already exists. Checking for new records...
Inserted 241 new records into 'player_3s_cluster_new'.


# Rank 3s CS, PU, Corners - Scale compared to average?

### Catch and Shoot, Offscreen Catch and Shoot, PNP, Pullup, ISO Pullup, Corners

In [1033]:
df_full_pt

,SEASON_YEAR,PLAYER_ID,TEAM_ID,GAME_ID,GAME_DATE,GAMES_IN_WINDOW_60G,MIN_60G_Mavg,PTS_60G_Mavg,REB_60G_Mavg,AST_60G_Mavg,FGA_60G_Mavg,FG3M_60G_Mavg,FG3A_60G_Mavg,FGA_60G_Sum,FG3M_60G_Sum,FG3A_60G_Sum,MIN_60G_Sum,PTS_60G_Sum,HEIGHT_INCHES,WEIGHT,SEASON_EXP,PLAYER_NAME,OVERALL_SCORE_Cut,OVERALL_SCORE_Handoff,OVERALL_SCORE_Isolation,OVERALL_SCORE_Misc,OVERALL_SCORE_OffRebound,OVERALL_SCORE_OffScreen,OVERALL_SCORE_PRBallHandler,OVERALL_SCORE_PRRollMan,OVERALL_SCORE_Postup,OVERALL_SCORE_Spotup,OVERALL_SCORE_Transition,OVERALL_SCORE_PERCENTILE_Cut,OVERALL_SCORE_PERCENTILE_Handoff,OVERALL_SCORE_PERCENTILE_Isolation,OVERALL_SCORE_PERCENTILE_Misc,OVERALL_SCORE_PERCENTILE_OffRebound,OVERALL_SCORE_PERCENTILE_OffScreen,OVERALL_SCORE_PERCENTILE_PRBallHandler,OVERALL_SCORE_PERCENTILE_PRRollMan,OVERALL_SCORE_PERCENTILE_Postup,OVERALL_SCORE_PERCENTILE_Spotup,OVERALL_SCORE_PERCENTILE_Transition,SCORING_EFFICIENCY_Cut,SCORING_EFFICIENCY_Handoff,SCORING_EFFICIENCY_Isolation,SCORING_EFFICIENCY_Misc,SCORING_EFFICIENCY_OffRebound,SCORING_EFFICIENCY_OffScreen,SCORING_EFFICIENCY_PRBallHandler,SCORING_EFFICIENCY_PRRollMan,SCORING_EFFICIENCY_Postup,SCORING_EFFICIENCY_Spotup,SCORING_EFFICIENCY_Transition,SCORING_EFFICIENCY_PERCENTILE_Cut,SCORING_EFFICIENCY_PERCENTILE_Handoff,SCORING_EFFICIENCY_PERCENTILE_Isolation,SCORING_EFFICIENCY_PERCENTILE_Misc,SCORING_EFFICIENCY_PERCENTILE_OffRebound,SCORING_EFFICIENCY_PERCENTILE_OffScreen,SCORING_EFFICIENCY_PERCENTILE_PRBallHandler,SCORING_EFFICIENCY_PERCENTILE_PRRollMan,SCORING_EFFICIENCY_PERCENTILE_Postup,SCORING_EFFICIENCY_PERCENTILE_Spotup,SCORING_EFFICIENCY_PERCENTILE_Transition,VOLUME_SCORE_Cut,VOLUME_SCORE_Handoff,VOLUME_SCORE_Isolation,VOLUME_SCORE_Misc,VOLUME_SCORE_OffRebound,VOLUME_SCORE_OffScreen,VOLUME_SCORE_PRBallHandler,VOLUME_SCORE_PRRollMan,VOLUME_SCORE_Postup,VOLUME_SCORE_Spotup,VOLUME_SCORE_Transition,VOLUME_SCORE_PERCENTILE_Cut,VOLUME_SCORE_PERCENTILE_Handoff,VOLUME_SCORE_PERCENTILE_Isolation,VOLUME_SCORE_PERCENTILE_Misc,VOLUME_SCORE_PERCENTILE_OffRebound,VOLUME_SCORE_PERCENTILE_OffScreen,VOLUME_SCORE_PERCENTILE_PRBallHandler,VOLUME_SCORE_PERCENTILE_PRRollMan,VOLUME_SCORE_PERCENTILE_Postup,VOLUME_SCORE_PERCENTILE_Spotup,VOLUME_SCORE_PERCENTILE_Transition,POSS_PCT_Cut,POSS_PCT_Handoff,POSS_PCT_Isolation,POSS_PCT_Misc,POSS_PCT_OffRebound,POSS_PCT_OffScreen,POSS_PCT_PRBallHandler,POSS_PCT_PRRollMan,POSS_PCT_Postup,POSS_PCT_Spotup,POSS_PCT_Transition,AVG_SPEED_OFF_60G_Sum,AVG_SPEED_DEF_60G_Sum,OFFmiles_PER_MIN,DEFmiles_PER_MIN,OREB_PER_CHANCE,CONTEST_OREB_PER_CHANCE,DREB_PER_CHANCE,CONTEST_DREB_PER_CHANCE,OREB_CHANCES_60G_Sum,DREB_CHANCES_60G_Sum,PASSES_MADE_60G_Sum,PASSES_RECEIVED_60G_Sum,AST_60G_Sum,POTENTIAL_AST_60G_Sum,SECONDARY_AST_60G_Sum,PASSES_MIN_60G_Sum,AST_PASS_60G_Sum,AST_PER_MIN,AST_TO_POTENTIAL,SECONDARY_AST_RATE,POINTS_PER_AST,PU_FGA_PCT_60Day,PU_FG2_PCT_60Day,PU_FG3_PCT_60Day,PU_FGA_rate_60Day,PU_FG3A_rate_60Day,CS_FG2_PCT_60Day,CS_FG3_PCT_60Day,CS_FGA_rate_60Day,CS_FG3A_rate_60Day,DRIVES/MIN_60Day,DRIVES_AST_PASS_RATE,DRIVES_PASS_RATE,SHOTS_DRIVE_RATE,PTS_PER_DRIVE,FGA_60G_Mavg_0 Dribbles,FGA_60G_Mavg_1 Dribble,FGA_60G_Mavg_2 Dribbles,FGA_60G_Mavg_3-6 Dribbles,FGA_60G_Mavg_7+ Dribbles,FG2M_60G_Mavg_0 Dribbles,FG2M_60G_Mavg_1 Dribble,FG2M_60G_Mavg_2 Dribbles,FG2M_60G_Mavg_3-6 Dribbles,FG2M_60G_Mavg_7+ Dribbles,FG3M_60G_Mavg_0 Dribbles,FG3M_60G_Mavg_1 Dribble,FG3M_60G_Mavg_2 Dribbles,FG3M_60G_Mavg_3-6 Dribbles,FG3M_60G_Mavg_7+ Dribbles,FGA_60G_Mavg_Touch 2-6 Seconds,FGA_60G_Mavg_Touch 6+ Seconds,FGA_60G_Mavg_Touch < 2 Seconds,FG2M_60G_Mavg_Touch 2-6 Seconds,FG2M_60G_Mavg_Touch 6+ Seconds,FG2M_60G_Mavg_Touch < 2 Seconds,FG3M_60G_Mavg_Touch 2-6 Seconds,FG3M_60G_Mavg_Touch 6+ Seconds,FG3M_60G_Mavg_Touch < 2 Seconds,FGA_3_AB_60G_Mavg,FGA_3_LC_60G_Mavg,FGA_3_RC_60G_Mavg,FGM_3_AB_60G_Mavg,FGM_3_LC_60G_Mavg,FGM_3_RC_60G_Mavg,FG3A_per_MIN_60G
0,2024-25,1642419,1610612761,0022400947,2025-03-12,43.0,13.619070,5.860465,2.116279,0.790698,4.581395,1.441860,3.465116,197.0,62.0,149.0,585.620000,252.

In [1034]:
df_3s_stats = pd.DataFrame(df_full_pt[['PLAYER_NAME','PLAYER_ID','TEAM_ID','MIN_60G_Mavg','FG3A_per_MIN_60G','FG3A_60G_Mavg','HEIGHT_INCHES','OVERALL_SCORE_Handoff','SCORING_EFFICIENCY_Handoff','VOLUME_SCORE_Handoff','OVERALL_SCORE_OffScreen','SCORING_EFFICIENCY_OffScreen','VOLUME_SCORE_OffScreen',
                                       'OVERALL_SCORE_Isolation','SCORING_EFFICIENCY_Isolation','VOLUME_SCORE_Isolation',
                                       'OVERALL_SCORE_PRRollMan','SCORING_EFFICIENCY_PRRollMan','VOLUME_SCORE_PRRollMan','OVERALL_SCORE_PRBallHandler','SCORING_EFFICIENCY_PRBallHandler','VOLUME_SCORE_PRBallHandler',
                                       'OVERALL_SCORE_Spotup','SCORING_EFFICIENCY_Spotup','VOLUME_SCORE_Spotup',
            'OFFmiles_PER_MIN','PU_FG3_PCT_60Day','PU_FG3A_rate_60Day','CS_FG3_PCT_60Day','CS_FG3A_rate_60Day',
           'FG3M_60G_Mavg_0 Dribbles','FG3M_60G_Mavg_2 Dribbles','FG3M_60G_Mavg_3-6 Dribbles','FG3M_60G_Mavg_7+ Dribbles','FG3M_60G_Mavg_Touch 2-6 Seconds',
       'FG3M_60G_Mavg_Touch 6+ Seconds', 'FG3M_60G_Mavg_Touch < 2 Seconds','FGA_3_AB_60G_Mavg', 'FGA_3_LC_60G_Mavg',
       'FGA_3_RC_60G_Mavg', 'FGM_3_AB_60G_Mavg',
       'FGM_3_LC_60G_Mavg', 'FGM_3_RC_60G_Mavg']])

In [1035]:
df_3s_stats

,PLAYER_NAME,PLAYER_ID,TEAM_ID,MIN_60G_Mavg,FG3A_per_MIN_60G,FG3A_60G_Mavg,HEIGHT_INCHES,OVERALL_SCORE_Handoff,SCORING_EFFICIENCY_Handoff,VOLUME_SCORE_Handoff,OVERALL_SCORE_OffScreen,SCORING_EFFICIENCY_OffScreen,VOLUME_SCORE_OffScreen,OVERALL_SCORE_Isolation,SCORING_EFFICIENCY_Isolation,VOLUME_SCORE_Isolation,OVERALL_SCORE_PRRollMan,SCORING_EFFICIENCY_PRRollMan,VOLUME_SCORE_PRRollMan,OVERALL_SCORE_PRBallHandler,SCORING_EFFICIENCY_PRBallHandler,VOLUME_SCORE_PRBallHandler,OVERALL_SCORE_Spotup,SCORING_EFFICIENCY_Spotup,VOLUME_SCORE_Spotup,OFFmiles_PER_MIN,PU_FG3_PCT_60Day,PU_FG3A_rate_60Day,CS_FG3_PCT_60Day,CS_FG3A_rate_60Day,FG3M_60G_Mavg_0 Dribbles,FG3M_60G_Mavg_2 Dribbles,FG3M_60G_Mavg_3-6 Dribbles,FG3M_60G_Mavg_7+ Dribbles,FG3M_60G_Mavg_Touch 2-6 Seconds,FG3M_60G_Mavg_Touch 6+ Seconds,FG3M_60G_Mavg_Touch < 2 Seconds,FGA_3_AB_60G_Mavg,FGA_3_LC_60G_Mavg,FGA_3_RC_60G_Mavg,FGM_3_AB_60G_Mavg,FGM_3_LC_60G_Mavg,FGM_3_RC_60G_Mavg
0,Jamison Battle,1642419,1610612761,13.619070,0.254431,3.465116,79.0,0.0,0.0,0.0,37.7,51.3,19.7,0.0,0.0,0.0,23.7,35.6,14.2,0.0,0.0,0.0,42.1,61.4,16.9,0.042483,0.440,0.04,0.403,0.20,1.3,0.2,0.0,0.0,0.3,0.0,1.4,2.025000,1.000000,0.700000,0.850000,0.500000,0.200000
1,Jaylen Wells,1642377,1610612763,26.380472,0.198379,5.233333,80.0,0.0,0.0,0.0,39.1,56.6,16.9,37.8,51.7,19.5,0.0,0.0,0.0,47.1,69.8,16.6,59.1,49.9,49.7,0.038706,0.410,0.02,0.371,0.16,1.6,0.1,0.1,0.0,0.0,0.0,1.7,3.133333,1.366667,0.666667,1.100000,0.516667,0.300000
2,Quinten Post,1642366,1610612744,16.130062,0.254873,4.111111,84.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39.2,42.6,29.0,0.0,0.0,0.0,44.7,56.6,24.6,0.042121,0.333,0.01,0.456,0.22,1.8,0.0,0.0,0.0,0.0,0.0,1.9,3.192308,0.269231,0.769231,1.307692,0.076923,0.423077
3,KJ Simpson,1642354,1610612766,20.438406,0.157419,3.217391,72.0,43.8,51.1,28.1,0.0,0.0,0.0,42.1,56.7,20.9,0.0,0.0,0.0,30.7,34.2,24.9,28.2,27.9,26.9,0.041886,0.310,0.06,0.171,0.09,0.4,0.1,0.2,0.2,0.2,0.2,0.4,2.695652,0.304348,0.217391,0.608696,0.086957,0.000000
4,Justin Edwards,1642348,1610612755,24.094140,0.166015,4.000000,79.0,0.0,0.0,0.0,0.0,0.0,0.0,23.0,29.8,18.2,0.0,0.0,0.0,53.3,79.2,16.8,50.4,52.4,35.9,0.040300,0.273,0.01,0.326,0.12,1.3,0.1,0.0,0.1,0.0,0.3,1.3,3.033333,0.733333,0.366667,0.966667,0.266667,0.166667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,Al Horford,201143,1610612738,27.278000,0.181465,4.950000,81.0,0.0,0.0,0.0,45.7,72.1,12.6,0.0,0.0,0.0,34.0,37.4,26.6,0.0,0.0,0.0,50.2,51.6,36.3,0.036643,0.333,0.00,0.375,0.17,1.9,0.0,0.0,0.0,0.0,0.0,1.9,3.200000,0.716667,1.016667,1.183333,0.300000,0.383333
237,Kevin Durant,201142,1610612756,36.973667,0.166785,6.166667,83.0,45.3,62.2,20.6,66.9,55.3,55.6,74.7,63.9,58.8,47.1,50.9,32.6,52.0,55.7,35.2,65.3,60.0,49.5,0.035964,0.362,0.04,0.426,0.12,2.0,0.1,0.2,0.1,0.2,0.2,2.1,4.583333,0.683333,0.816667,1.700000,0.350000,0.416667
238,Kyle Lowry,200768,1610612755,22.970306,0.137134,3.150000,72.0,48.6,66.8,21.1,48.3,68.5,19.2,18.4,21.4,19.3,0.0,0.0,0.0,29.8,34.8,23.1,32.9,45.8,17.8,0.038511,0.367,0.04,0.325,0.09,0.8,0.2,0.3,0.1,0.4,0.0,0.0,2.350000,0.533333,0.366667,0.833333,0.116667,0.166667
239,Chris Paul,101108,1610612759,28.476500,0.167975,4.783333,72.0,44.0,64.2,17.2,0.0,0.0,0.0,36.1,45.8,22.2,0.0,0.0,0.0,46.5,51.0,31.9,37.3,50.8,19.6,0.035005,0.371,0.10,0.383,0.06,0.8,0.1,0.4,0.4,0.6,0.5,0.9,4.066667,0.350000,0.350000,1.583333,0.100000,0.133333


In [1036]:
df_3s_stats['FG3A_Corner_Rate'] = df_3s_stats['FG3A_60G_Mavg']/(df_3s_stats['FGA_3_RC_60G_Mavg']+df_3s_stats['FGA_3_LC_60G_Mavg'])
df_3s_stats['Corner_FG_PCT'] = (df_3s_stats['FGM_3_RC_60G_Mavg']+df_3s_stats['FGM_3_LC_60G_Mavg'])/(df_3s_stats['FGA_3_RC_60G_Mavg']+df_3s_stats['FGA_3_LC_60G_Mavg'])
df_3s_stats['Corner_FGA_rate'] = (df_3s_stats['FGA_3_RC_60G_Mavg']+df_3s_stats['FGA_3_LC_60G_Mavg'])/df_3s_stats['MIN_60G_Mavg']

In [1037]:
df_3s_stats.fillna(0, inplace=True)

In [1038]:
league_avg_fg_pct = df_3s_stats['CS_FG3_PCT_60Day'].mean()
league_avg_fga_per_min = df_3s_stats['CS_FG3A_rate_60Day'].mean()
league_avg_screen_fg_pct = df_3s_stats['OVERALL_SCORE_OffScreen'].mean()
league_avg_pts_per_fgm_pnrman = df_3s_stats['OVERALL_SCORE_PRRollMan'].mean()
league_avg_pts_per_fgm_ho = df_3s_stats['OVERALL_SCORE_Handoff'].mean()
league_avg_pts_per_fgm_pnrball = df_3s_stats['OVERALL_SCORE_PRBallHandler'].mean()
#league_avg_poss_pct_pnr = df_3s_stats['POSS_PCT_PRRollMan'].mean()
league_avg_fg_pct_PU = df_3s_stats['PU_FG3_PCT_60Day'].mean()
league_avg_fga_per_min_PU = df_3s_stats['PU_FG3A_rate_60Day'].mean()
league_avg_iso_fg_pct = df_3s_stats['OVERALL_SCORE_Isolation'].mean()
league_avg_spotup_fg_pct = df_3s_stats['OVERALL_SCORE_Spotup'].mean()
league_avg_corner_fg_pct = df_3s_stats['Corner_FG_PCT'].mean()
league_avg_corner_fga_per_min = df_3s_stats['Corner_FGA_rate'].mean()

In [1039]:
#df_3s_stats[['PLAYER_NAME','PTS/MAKE_PRRollMan']].sort_values(by='PTS/MAKE_PRRollMan')

In [1040]:
'''df_3s_stats['scoring_profile_score'] = (
        # Compare player's pts per FGM to league average
        df_3s_stats['PTS/MAKE_PRRollMan'] / league_avg_pts_per_fgm_pnrman
)'''

"df_3s_stats['scoring_profile_score'] = (\n        # Compare player's pts per FGM to league average\n        df_3s_stats['PTS/MAKE_PRRollMan'] / league_avg_pts_per_fgm_pnrman\n)"

In [1041]:
#league_avg_screen_fg_pct

In [1042]:

offscreen_score = (
        (df_3s_stats['OVERALL_SCORE_OffScreen'] / league_avg_screen_fg_pct) # Scale by frequency to reward volume
    )
iso_score = (
        (df_3s_stats['OVERALL_SCORE_Isolation'] / league_avg_iso_fg_pct)   # Scale by frequency to reward volume
    )
spotup_score = (
        (df_3s_stats['OVERALL_SCORE_Spotup'] / league_avg_spotup_fg_pct)   # Scale by frequency to reward volume
    )

pnr_score = (
        (df_3s_stats['OVERALL_SCORE_PRRollMan']/league_avg_pts_per_fgm_pnrman)
    # Scale by frequency to reward volume
    )
pnr_ball_score = (
        (df_3s_stats['OVERALL_SCORE_PRBallHandler']/league_avg_pts_per_fgm_pnrball)
    # Scale by frequency to reward volume
    )
ho_score = (
        (df_3s_stats['OVERALL_SCORE_Handoff']/league_avg_pts_per_fgm_ho)
    # Scale by frequency to reward volume
    )

In [1043]:
#df_3s_stats['SHOT_FREQ_Isolation']

In [1044]:
shooting_component = (
        (df_3s_stats['CS_FG3_PCT_60Day'] / league_avg_fg_pct) * 
        (df_3s_stats['CS_FG3A_rate_60Day'] / league_avg_fga_per_min))
shooting_component_pu = (
        (df_3s_stats['PU_FG3_PCT_60Day'] / league_avg_fg_pct_PU) * 
        (df_3s_stats['PU_FG3A_rate_60Day'] / league_avg_fga_per_min_PU))
shooting_component_corner = (
        (df_3s_stats['Corner_FG_PCT'] / league_avg_corner_fg_pct) * 
        (df_3s_stats['Corner_FGA_rate'] / league_avg_corner_fga_per_min))

In [1045]:
df_3s_stats['offscreen_cse_score'] = (
        shooting_component * 0.5 +  # 60% weight on shooting
        offscreen_score * 0.3 +
        spotup_score * 0.2 # 40% weight on movement
    ) * np.log1p(df_3s_stats['MIN_60G_Mavg']) * 100
average_score = df_3s_stats['offscreen_cse_score'].mean()
df_3s_stats['offscreen_cse_score'] = (df_3s_stats['offscreen_cse_score'] / average_score) * 100

df_3s_stats['pnp_cse_score'] = (
        shooting_component * 0.55 +  # 60% weight on shooting
        pnr_score * 0.30 +  # 60% weight on shooting
        spotup_score * 0.15 # 40% weight on movement
    ) * np.log1p(df_3s_stats['MIN_60G_Mavg']) * 100
average_score = df_3s_stats['pnp_cse_score'].mean()
df_3s_stats['pnp_cse_score'] = (df_3s_stats['pnp_cse_score'] / average_score) * 100


df_3s_stats['iso_pue_score'] = (
        shooting_component_pu * 0.6 +  # 60% weight on shooting
        iso_score * 0.3 +
        pnr_ball_score * 0.1 # 40% weight on movement
    ) * np.log1p(df_3s_stats['MIN_60G_Mavg']) * 100
average_score = df_3s_stats['iso_pue_score'].mean()
df_3s_stats['iso_pue_score'] = (df_3s_stats['iso_pue_score'] / average_score) * 100

In [1046]:
df_3s_stats['cse_score'] = (
        (df_3s_stats['CS_FG3_PCT_60Day'] / league_avg_fg_pct) * 
        (df_3s_stats['CS_FG3A_rate_60Day'] / league_avg_fga_per_min) * 
        np.log1p(df_3s_stats['MIN_60G_Mavg'])  # Log transformation reduces extreme values
    ) * 100
average_score = df_3s_stats['cse_score'].mean()
df_3s_stats['cse_score'] = (df_3s_stats['cse_score'] / average_score) * 100

df_3s_stats['CnS_Rating'] = (df_3s_stats['CS_FG3A_rate_60Day'] * 
                              df_3s_stats['CS_FG3_PCT_60Day'] * 
                              np.sqrt(df_3s_stats['MIN_60G_Mavg'])).round(3)
df_3s_stats['pue_score'] = (
        (df_3s_stats['PU_FG3_PCT_60Day'] / league_avg_fg_pct_PU) * 
        (df_3s_stats['PU_FG3A_rate_60Day'] / league_avg_fga_per_min_PU) * 
        np.log1p(df_3s_stats['MIN_60G_Mavg'])  # Log transformation reduces extreme values
    ) * 100
average_score = df_3s_stats['pue_score'].mean()
df_3s_stats['pue_score'] = (df_3s_stats['pue_score'] / average_score) * 100

df_3s_stats['PU_Rating'] = (df_3s_stats['PU_FG3A_rate_60Day'] * 
                              df_3s_stats['PU_FG3_PCT_60Day'] * 
                              np.sqrt(df_3s_stats['MIN_60G_Mavg'])).round(3)

df_3s_stats['corner_score'] = (
        (df_3s_stats['Corner_FG_PCT'] / league_avg_corner_fg_pct) * 
        (df_3s_stats['Corner_FGA_rate'] / league_avg_corner_fga_per_min) * 
        np.log1p(df_3s_stats['MIN_60G_Mavg'])  # Log transformation reduces extreme values
    ) * 100
average_score = df_3s_stats['corner_score'].mean()
df_3s_stats['corner_score'] = (df_3s_stats['corner_score'] / average_score) * 100


In [1047]:
#league_avg_corner_fg_pct

In [1048]:
#df_3s_stats['iso_pue_percentile'] = df_3s_stats['iso_pue_percentile'].rank(pct=True, ascending=False) * 100

In [1049]:
df_3s_scores = pd.DataFrame(df_3s_stats[['PLAYER_ID','PLAYER_NAME','MIN_60G_Mavg','cse_score','offscreen_cse_score','pnp_cse_score','pue_score','iso_pue_score','corner_score']])

In [1050]:
#df_3s_scores.sort_values(by='OVERALL_SCORE_OffScreen')

In [1051]:
df_3s_scores['as_of'] = pd.to_datetime(today)

In [1052]:
df_3s_scores['id'] = df_3s_scores['as_of'].astype(str)+"_"+df_3s_scores['PLAYER_ID'].astype(str)

In [1053]:
# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_3s_scores"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    df_3s_scores.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = df_3s_scores[~df_3s_scores['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'player_3s_scores' already exists. Checking for new records...
Inserted 241 new records into 'player_3s_scores'.
